# TD — Priorização de Melhorias Produto Físico

Notebook unificado para extrair, processar, validar e publicar o Farol de
Priorização de Melhorias de Produto Físico/T&D.

Este notebook consolida:
- SQL de extração BigQuery (`pipeline_reversas_priorizacao_produtos.sql`);
- funções analíticas antes mantidas em `td_analysis_functions.py`;
- pipeline principal antes distribuído em `TD_Priorizacao_Melhorias_v20260611.ipynb`;
- etapas de QA;
- geração dos outputs finais;
- atualização/exportação dos resultados.

## Princípios

- Não alterar regra de negócio sem revisão.
- Não alterar thresholds sem alinhamento.
- Não depender de caminhos absolutos locais.
- Preparado para execução em Deepnote.
- Manter o pipeline legível e auditável.

## Estrutura

| Bloco | Descrição |
|---|---|
| 1 | Configuração do ambiente |
| 2 | Parâmetros do pipeline |
| 3 | SQL de extração dos dados |
| 4 | Execução da extração no BigQuery |
| 5 | Funções utilitárias e regras de negócio |
| 6 | Preparação e normalização dos DataFrames |
| 7 | Classificação e score de priorização |
| 8 | Diagnósticos e tabelas analíticas |
| 9 | QA e validações |
| 10 | Construção dos outputs finais |
| 11 | Atualização/exportação dos resultados |
| 12 | Log final de execução |
| 13 | Apêndice de debug (opcional) |


## 1. Configuração do ambiente

Concentra todos os imports, caminhos relativos e configurações globais.

**Entrada:** nenhuma
**Saída:** variáveis de ambiente e caminhos disponíveis para todos os blocos
**Quando mexer:** ao adicionar nova biblioteca ou ajustar configurações de display.

> Todos os caminhos usam `Path.cwd()` — nenhum caminho absoluto do Mac (`/Users/...`).
> Compatível com Deepnote e outros ambientes de execução.


In [78]:
# ── Imports ──────────────────────────────────────────────────────────────────
from __future__ import annotations

import json
import os
import warnings
from dataclasses import dataclass
from datetime import datetime
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd
from IPython.display import display

# ── Caminhos relativos ────────────────────────────────────────────────────────
# Todos os caminhos são relativos — sem /Users/... ou caminhos absolutos locais.
from pathlib import Path

NOTEBOOK_DIR = Path.cwd()                               # analyses/relatorio_td/
PROJECT_ROOT = NOTEBOOK_DIR.parent.parent               # LA_Coding_Projects/
OUTPUT_DIR   = PROJECT_ROOT / "outputs" / "relatorio_td"
DATA_DIR     = NOTEBOOK_DIR / "data"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

# SQL_DIR: aponta para o arquivo fonte externo (compatibilidade/debug).
# No pipeline produtivo, o SQL está embutido no Bloco 3 — não é necessário.
SQL_DIR = NOTEBOOK_DIR

# ── Configurações de display ──────────────────────────────────────────────────
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 60)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
warnings.filterwarnings("ignore", category=FutureWarning)

DATE_TAG = datetime.now().strftime("%Y%m%d")

print(f"✅ Ambiente configurado")
print(f"   NOTEBOOK_DIR : {NOTEBOOK_DIR}")
print(f"   OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"   DATE_TAG     : {DATE_TAG}")


✅ Ambiente configurado
   NOTEBOOK_DIR : /Users/insider/LA_Coding_Projects/analyses/relatorio_td
   OUTPUT_DIR   : /Users/insider/LA_Coding_Projects/outputs/relatorio_td
   DATE_TAG     : 20260623


## 2. Parâmetros do pipeline

**Todos** os parâmetros editáveis do pipeline estão aqui — não espalhar flags pelos demais blocos.

**Entrada:** variáveis de ambiente opcionais
**Saída:** variáveis de configuração usadas pelo pipeline inteiro
**Quando mexer:** ao alterar janela de análise, flags de execução ou ID da planilha.

| Parâmetro | Descrição | Default |
|---|---|---|
| `PROJECT_ID` | Projeto BigQuery | `insider-data-lake` |
| `LOAD_ANALITICA` | Executa Camada 1 (query pesada, ~60s) | `True` |
| `EXPORT_DEBUG_FILES` | Exporta Excel de debug para `OUTPUT_DIR` | `False` |
| `UPDATE_GOOGLE_SHEETS` | Atualiza planilha existente | `True` |
| `RUN_DEBUG_APPENDIX` | Executa bloco de debug (Bloco 13) | `False` |
| `RUN_VALIDATION` | Executa queries de validação pós-migração (Pontos 1/2/3) | `False` |
| `SPREADSHEET_ID` | ID da planilha a atualizar | hardcoded |


In [79]:
# ── Projeto BigQuery ──────────────────────────────────────────────────────────
PROJECT_ID = os.getenv("BQ_PROJECT_ID", "insider-data-lake")

# ── Data de execução ──────────────────────────────────────────────────────────
RUN_DATE = pd.Timestamp.now(tz="America/Sao_Paulo")

# ── Flags de execução ─────────────────────────────────────────────────────────
# Se True, executa Camada 1 (analítica) no BigQuery — query pesada (~60s).
LOAD_ANALITICA = True

# Se True, exporta arquivo Excel de debug para OUTPUT_DIR.
EXPORT_DEBUG_FILES = False

# Se True, atualiza a planilha Google Sheets existente.
# Requer GOOGLE_SERVICE_ACCOUNT_JSON ou autenticação gcloud local.
UPDATE_GOOGLE_SHEETS = True

# Se True, executa o Bloco 13 de debug.
RUN_DEBUG_APPENDIX = False

# Se True, executa as queries de validação pós-migração v2 (Pontos 1/2/3 do Bloco 4).
# Custo: ~3 queries adicionais no BQ. Manter False em execuções de produção.
RUN_VALIDATION = False

# ── Google Sheets ─────────────────────────────────────────────────────────────
# ID da planilha EXISTENTE — nunca criar uma nova planilha.
#
# 🟢 PLANILHA OFICIAL (ativa):
#    https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E
#
# 🧪 CÓPIA DE TESTE (usar para validação):
#    https://docs.google.com/spreadsheets/d/1kUxUw-MbuAyjktkxqOQ5b9KCnprJioRJv0NTDE45aso
#
# Para alterar sem editar o notebook, defina a variável de ambiente:
#   TD_PRIORIZACAO_SPREADSHEET_ID=<ID>
SPREADSHEET_ID = os.getenv(
    "TD_PRIORIZACAO_SPREADSHEET_ID",
    "1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E",  # planilha oficial
    # "1kUxUw-MbuAyjktkxqOQ5b9KCnprJioRJv0NTDE45aso",  # cópia de teste
)

print(f"PROJECT_ID            : {PROJECT_ID}")
print(f"RUN_DATE              : {RUN_DATE}")
print(f"LOAD_ANALITICA        : {LOAD_ANALITICA}")
print(f"EXPORT_DEBUG_FILES    : {EXPORT_DEBUG_FILES}")
print(f"UPDATE_GOOGLE_SHEETS  : {UPDATE_GOOGLE_SHEETS}")
print(f"RUN_VALIDATION        : {RUN_VALIDATION}")
print(f"SPREADSHEET_ID        : {SPREADSHEET_ID[:24]}...")


PROJECT_ID            : insider-data-lake
RUN_DATE              : 2026-06-23 18:07:42.380520-03:00
LOAD_ANALITICA        : True
EXPORT_DEBUG_FILES    : False
UPDATE_GOOGLE_SHEETS  : True
RUN_VALIDATION        : False
SPREADSHEET_ID        : 1ikFhRdMUe1uK4bVn8lh8_7Z...


## 3. SQL de extração dos dados

SQL incorporado diretamente para eliminar dependência de arquivo externo.

**Origem:** `pipeline_reversas_priorizacao_produtos.sql` (preservado como referência histórica).
**Quando mexer:** ao alterar filtros, janela ou regras de scoring — revisar metodologia antes.

> ⚠️ **Não alterar thresholds, pesos ou CASE statements sem alinhamento com a equipe.**
>
> A query abaixo foi incorporada ao notebook para reduzir dependência de arquivos locais.
> Caso seja versionada externamente no futuro, manter esta célula sincronizada com o SQL fonte.

### Fontes de Dados (v2 — 2026-06-18)

| Tabela | Projeto | Uso |
|---|---|---|
| `business.insider_orders` | `insider-data-lake` | Pedidos válidos: `order_status = 'paid'`, `is_cancelled = FALSE`, exclusão de cupons internos, lojas `shopify_insider-world` e `shopify_insider-store-loja` |
| `business.insider_order_items` | `insider-data-lake` | Itens de pedido: `sku`, `quantity`, `product_title`, `variant_color`, `variant_size` |
| `fpa.analytical_dre` | `insider-data-lake` | Receita e volume: `revenue_after_discounts`, `non_refunded_quantity` (agregado por `product_name`) |
| `integrated.skus` | `insider-data-lake` | Dimensão SKU: `product_name`, `category`, `gender`, `color`, `size`, `sku_state` (fallback) |
| `prepared_br.prepared__troquecommerce_order_details_br` | `insider-lake-sensitive` | Reversas ativas (`status ≠ Cancelado`), dedup por `(order_name, id_reversa, sku)` |
| `sop_silver.return_reason_tags` | `insider-data-lake` | Tags qualitativas de motivo, dedup `(order_name, sku)` + UNNEST |
| `sop_silver.portfolio_skp_clustering` | `insider-data-lake` | Cluster estratégico do produto (grão `product_name`) |
| `sop_bronze.eval_produto_portfolio` | `insider-data-lake` | Scorecard de portfólio — 3 pilares (query avulsa) |

| Variável | Camada | Grain | Uso |
|---|---|---|---|
| `EXECUTIVE_QUERY` | Camada 2 | `product_name × category × gender` | Pipeline principal |
| `ANALYTICAL_QUERY` | Camada 1 | `order × sku × tag` | Auditoria e drill-down (`LOAD_ANALITICA=True`) |
| `SCORECARD_QUERY` | Avulso | `product_name` | Pilares do scorecard de portfólio |

In [80]:
# Camada 2 — Tabela executiva por produto
# Objetivo: gerar farol decisório de priorização de melhoria física.
# Grain esperado: 1 linha por product_name × category × gender
# ⚠️ Não alterar thresholds, pesos ou CASE statements sem revisar a metodologia.
EXECUTIVE_QUERY = """
-- ==============================================================================
-- PIPELINE DE ANÁLISE DE REVERSAS E PRIORIZAÇÃO DE PRODUTOS
-- ==============================================================================
--
-- Camada 2 — Tabela executiva agregada por produto   (output principal)
-- Camada 1 — Tabela analítica item/reversa/tag       (auditoria e drill-down)
--
-- Fontes de dados (v2 — 2026-06-18):
--   Vendas:   business.insider_orders + business.insider_order_items
--   Reversas: prepared_br.prepared__troquecommerce_order_details_br (dedup por id_reversa)
--   Receita:  fpa.analytical_dre (agregado por product_name)
--   Tags:     sop_silver.return_reason_tags (grão order_name × sku — sem id_reversa)
--   SKU dim:  integrated.skus (fallback color/size e product_name canônico)
--   Cluster:  sop_silver.portfolio_skp_clustering (grão product_name)
--
-- Histórico de correções:
--   B1–B7 — ver versões anteriores.
--   B8 — Migração de integrated.orders/order_items → business.insider_orders/order_items.
--         Reversas dedup mudou de (order_name, sku) para (order_name, id_reversa, sku).
--         Receita e volume passaram a vir de fpa.analytical_dre (grão product_name).
-- ==============================================================================


-- ==============================================================================
-- CAMADA 2: TABELA EXECUTIVA AGREGADA POR PRODUTO
-- Objetivo: gerar farol decisório de priorização de melhoria física.
-- Granularidade: product_name × category × gender
-- ==============================================================================

WITH params AS (
  SELECT
    DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 12 MONTH) AS start_date,
    CURRENT_DATE("America/Sao_Paulo") AS end_date,
    30 AS min_items_vendidos,
    5 AS min_items_returned
),

-- === BASE DE VENDAS ========================================================

orders AS (
  SELECT DISTINCT
    o.order_id,
    o.order_name,
    o.processed_at,
    DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") AS data_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), MONTH) AS mes_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), WEEK(MONDAY)) AS semana_compra
  FROM `insider-data-lake.business.insider_orders` o
  CROSS JOIN params p
  WHERE o.order_status = 'paid'
    AND o.is_cancelled = FALSE
    AND (
      o.coupon_code IS NULL OR (
        NOT STARTS_WITH(o.coupon_code, 'TF-')
        AND NOT STARTS_WITH(o.coupon_code, 'TFIN')
        AND NOT STARTS_WITH(o.coupon_code, 'IR')
        AND NOT (o.coupon_code LIKE '%Item errado%')
      )
    )
    AND o.order_name IS NOT NULL
    AND o.processed_at IS NOT NULL
    AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
    AND DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
),

order_items AS (
  SELECT
    i.order_id,
    i.sku,
    i.product_title,
    i.variant_color,
    i.variant_size,
    i.quantity
  FROM `insider-data-lake.business.insider_order_items` i
  WHERE i.sku IS NOT NULL
),

-- Consolida itens ao nível (order_id, sku) para evitar duplicação de reversa
-- quando o mesmo SKU aparece em múltiplos registros de um mesmo pedido.
order_items_grouped AS (
  SELECT
    order_id,
    sku,
    ANY_VALUE(product_title) AS product_title,
    ANY_VALUE(variant_color) AS variant_color,
    ANY_VALUE(variant_size) AS variant_size,
    SUM(quantity) AS qt_items
  FROM order_items
  GROUP BY order_id, sku
),

sku_dim AS (
  SELECT
    sku,
    ANY_VALUE(product_name) AS product_name,
    ANY_VALUE(category) AS category,
    ANY_VALUE(gender) AS gender,
    ANY_VALUE(color) AS color,
    ANY_VALUE(size) AS size,
    ANY_VALUE(sku_state) AS sku_state
  FROM `insider-data-lake.integrated.skus`
  WHERE sku IS NOT NULL
  GROUP BY sku
),

portfolio_clustering AS (
  -- Tabela em granularidade product_name (não tem coluna sku).
  SELECT
    product_name,
    ANY_VALUE(cluster) AS portfolio_cluster,
    ANY_VALUE(TO_JSON_STRING(pc)) AS portfolio_cluster_payload
  FROM `insider-data-lake.sop_silver.portfolio_skp_clustering` pc
  GROUP BY product_name
),

-- Grain resultante: (order_id, sku) — sem GROUP BY, pois order_items_grouped
-- já consolidou os itens e todos os joins são 1:1 por SKU.
sales_item_base AS (
  SELECT
    o.order_id,
    o.order_name,
    o.data_compra,
    o.mes_compra,
    o.semana_compra,

    oi.sku,
    COALESCE(s.product_name, oi.product_title) AS product_name,
    s.category,
    s.gender,
    COALESCE(s.color, oi.variant_color) AS color,
    COALESCE(s.size, oi.variant_size) AS size,
    s.sku_state,

    oi.product_title,
    oi.variant_color,
    oi.variant_size,

    oi.qt_items
  FROM orders o
  JOIN order_items_grouped oi
    ON o.order_id = oi.order_id
  LEFT JOIN sku_dim s
    ON s.sku = oi.sku
  -- Apenas produtos perenes e lançamentos (exclui desativados e cápsulas)
  WHERE s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
),

-- === REVERSAS =============================================================

reversas_unicas AS (
  SELECT
    r.order_name,
    r.id_reversa,
    r.status,
    r.reverse_type,
    r.sku,
    r.return_reason,
    r.updated_at,
    r.created_at,
    DATE(r.created_at, "America/Sao_Paulo") AS data_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), MONTH) AS mes_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), WEEK(MONDAY)) AS semana_reversa,
    r.client_comment,
    SAFE_CAST(r.reverse_shipping_cost AS FLOAT64) AS reverse_shipping_cost,
    SAFE_CAST(r.retained_bonus AS FLOAT64) AS retained_bonus,
    SAFE_CAST(r.exchange_value AS FLOAT64) AS exchange_value,
    SAFE_CAST(r.refund_value AS FLOAT64) AS refund_value,

    -- COALESCE aplicado na fonte para não inflar contagens; NULL de return_quantity
    -- indica quantidade não informada no sistema (tratada como 0).
    COALESCE(SAFE_CAST(r.return_quantity AS FLOAT64), 0) AS qt_items_returned,

    CASE
      WHEN LOWER(r.return_reason) LIKE '%ficou grande%' THEN 'Tamanho'
      WHEN r.return_reason = 'Peça íntima' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Produto com defeito' THEN 'Defeito'
      WHEN r.return_reason = 'Arrependimento' THEN 'Desistência'
      WHEN r.return_reason = 'Recebi um produto errado' THEN 'Produto errado'
      WHEN LOWER(r.return_reason) LIKE '%ficou pequeno%' THEN 'Tamanho'
      WHEN r.return_reason = 'Cor diferente do esperado' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Recebi novo pedido de reposição' THEN 'Produto errado'
      WHEN r.return_reason = 'Recebi pedido duplicado' THEN 'Produto errado'
      WHEN r.return_reason = 'Arrependiemento' THEN 'Desistência'
      WHEN r.return_reason = 'Insatisfação (tamanho e cor inclusos)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Arrependimento (tamanho e cor inclusos)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason LIKE '%Tamanho%' THEN 'Tamanho'
      WHEN r.return_reason = 'Insatisfação com o produto' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Falha na Entrega' THEN 'Problema na entrega'
      WHEN r.return_reason = 'Demora na entrega' THEN 'Problema na entrega'
      WHEN r.return_reason = 'Ficou Grande' THEN 'Tamanho'
      WHEN r.return_reason = 'Ficou Pequeno' THEN 'Tamanho'
      WHEN r.return_reason = 'Não gostei do produto' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Defeito' THEN 'Defeito'
      WHEN r.return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Insatisfação' THEN 'Insatisfação com o produto'
      WHEN r.return_reason = 'Defeitos' THEN 'Defeito'
      WHEN r.return_reason = 'Desbotamento' THEN 'Desbotamento'
      WHEN r.return_reason = 'Desistência' THEN 'Desistência'
      WHEN r.return_reason = 'Não gostei da qualidade' THEN 'Insatisfação com o produto'
      WHEN r.return_reason IS NULL THEN NULL
      ELSE 'Outros'
    END AS motivo_classificado,

    -- Classifica o problema: Físico = atributo do produto;
    -- Logístico = falha operacional/entrega; Desistência = mudança de intenção.
    CASE
      WHEN LOWER(r.return_reason) LIKE '%ficou grande%' THEN 'Físico'
      WHEN r.return_reason = 'Peça íntima' THEN 'Físico'
      WHEN r.return_reason = 'Produto com defeito' THEN 'Físico'
      WHEN r.return_reason = 'Arrependimento' THEN 'Desistência'
      WHEN r.return_reason = 'Recebi um produto errado' THEN 'Logístico'
      WHEN LOWER(r.return_reason) LIKE '%ficou pequeno%' THEN 'Físico'
      WHEN r.return_reason = 'Cor diferente do esperado' THEN 'Físico'
      WHEN r.return_reason = 'Recebi novo pedido de reposição' THEN 'Logístico'
      WHEN r.return_reason = 'Recebi pedido duplicado' THEN 'Logístico'
      WHEN r.return_reason = 'Arrependiemento' THEN 'Desistência'
      WHEN r.return_reason = 'Insatisfação (tamanho e cor inclusos)' THEN 'Físico'
      WHEN r.return_reason = 'Arrependimento (tamanho e cor inclusos)' THEN 'Desistência'
      WHEN r.return_reason LIKE '%Tamanho%' THEN 'Físico'
      WHEN r.return_reason = 'Insatisfação com o produto' THEN 'Físico'
      WHEN r.return_reason = 'Falha na Entrega' THEN 'Logístico'
      WHEN r.return_reason = 'Demora na entrega' THEN 'Logístico'
      WHEN r.return_reason = 'Ficou Grande' THEN 'Físico'
      WHEN r.return_reason = 'Ficou Pequeno' THEN 'Físico'
      WHEN r.return_reason = 'Não gostei do produto' THEN 'Físico'
      WHEN r.return_reason = 'Defeito' THEN 'Físico'
      WHEN r.return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Físico'
      WHEN r.return_reason = 'Insatisfação' THEN 'Físico'
      WHEN r.return_reason = 'Defeitos' THEN 'Físico'
      WHEN r.return_reason = 'Desbotamento' THEN 'Físico'
      WHEN r.return_reason = 'Desistência' THEN 'Desistência'
      WHEN r.return_reason = 'Não gostei da qualidade' THEN 'Físico'
      WHEN r.return_reason IS NULL THEN NULL
      ELSE 'Outros'
    END AS tipo_problema,

    CASE
      WHEN r.reverse_type IS NULL THEN 'Sem reversa'
      WHEN LOWER(r.reverse_type) LIKE '%troca%' THEN 'Troca'
      WHEN LOWER(r.reverse_type) LIKE '%devol%' THEN 'Devolução'
      ELSE r.reverse_type
    END AS reverse_type_classificado

  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br` r
  CROSS JOIN params p
  WHERE r.status <> 'Cancelado'
    AND r.sku IS NOT NULL
    AND r.return_reason IS NOT NULL
    AND r.id_reversa IS NOT NULL
    AND DATE(r.created_at, "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.order_name, r.id_reversa, r.sku
    ORDER BY r.updated_at DESC
  ) = 1
),

reversas_tags_latest AS (
  SELECT
    order_name,
    sku,
    tags,
    created_at,
    DATE_TRUNC(DATE(created_at), MONTH) AS ingestion_date
  FROM `insider-data-lake.sop_silver.return_reason_tags`
  WHERE order_name IS NOT NULL
    AND sku IS NOT NULL
    AND tags IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_name, sku
    ORDER BY created_at DESC
  ) = 1
),

reversas_tag AS (
  SELECT
    order_name,
    sku,
    tag,
    ingestion_date
  FROM reversas_tags_latest,
  UNNEST(tags) AS tag
),

-- === MÉTRICAS POR PRODUTO =================================================

-- Receita e volume de vendas a partir do DRE (FP&A)
-- Grain: product_name (agregado de order × sku × atribuição)
-- JOIN com orders para herdar filtro de período; JOIN com sku_dim para product_name
dre_product_metrics AS (
  SELECT
    s.product_name,
    SUM(dre.non_refunded_quantity) AS qt_items_vendidos,
    SUM(dre.revenue_after_discounts) AS receita_liquida
  FROM `insider-data-lake.fpa.analytical_dre` dre
  JOIN orders o
    ON dre.order_id = o.order_id
  JOIN sku_dim s
    ON dre.sku = s.sku
  WHERE s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
  GROUP BY s.product_name
),

sales_product_metrics AS (
  SELECT
    sib.product_name,
    sib.category,
    sib.gender,
    ANY_VALUE(pc.portfolio_cluster) AS portfolio_cluster,
    ANY_VALUE(pc.portfolio_cluster_payload) AS portfolio_cluster_payload,

    COUNT(DISTINCT sib.order_id) AS qt_pedidos,
    COUNT(DISTINCT sib.sku) AS qt_skus,
    -- Volume e receita: DRE como fonte de verdade; fallback para order_items se DRE vazio
    COALESCE(ANY_VALUE(dpm.qt_items_vendidos), SUM(sib.qt_items)) AS qt_items_vendidos,
    COALESCE(ANY_VALUE(dpm.receita_liquida), 0) AS receita_liquida
  FROM sales_item_base sib
  LEFT JOIN portfolio_clustering pc
    ON pc.product_name = sib.product_name
  LEFT JOIN dre_product_metrics dpm
    ON dpm.product_name = sib.product_name
  GROUP BY sib.product_name, sib.category, sib.gender
),

-- Join de reversas às vendas na grain (order_name, sku).
-- Tags NÃO entram aqui para não inflar contagens de T&D.
-- Nota: com dedup por id_reversa, pode haver múltiplas linhas por (order_name, sku)
-- quando existem múltiplas reversas para o mesmo item. COUNT DISTINCT protege as contagens.
td_joined AS (
  SELECT
    ru.order_name,
    ru.sku,
    sib.product_name,
    sib.category,
    sib.gender,
    sib.color,
    sib.size,

    ru.data_reversa,
    ru.mes_reversa,
    ru.reverse_type_classificado,
    ru.return_reason,
    ru.motivo_classificado,
    ru.tipo_problema,
    ru.client_comment,

    ru.qt_items_returned,
    ru.exchange_value,
    ru.refund_value
  FROM reversas_unicas ru
  JOIN sales_item_base sib
    ON ru.order_name = sib.order_name
   AND ru.sku = sib.sku
),

td_product_metrics AS (
  SELECT
    product_name,
    category,
    gender,

    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas,
    SUM(qt_items_returned) AS qt_items_returned,

    COUNT(DISTINCT IF(reverse_type_classificado = 'Troca', CONCAT(order_name, '|', sku), NULL)) AS qt_trocas,
    COUNT(DISTINCT IF(reverse_type_classificado = 'Devolução', CONCAT(order_name, '|', sku), NULL)) AS qt_devolucoes,

    COUNT(DISTINCT IF(tipo_problema = 'Físico', CONCAT(order_name, '|', sku), NULL)) AS qt_reversas_fisico,
    COUNT(DISTINCT IF(tipo_problema = 'Logístico', CONCAT(order_name, '|', sku), NULL)) AS qt_reversas_logistico,

    SUM(exchange_value) AS valor_troca,
    SUM(refund_value) AS valor_devolucao
  FROM td_joined
  GROUP BY product_name, category, gender
),

category_metrics AS (
  SELECT
    spm.category,
    SUM(spm.qt_items_vendidos) AS qt_items_vendidos_categoria,
    SUM(COALESCE(tpm.qt_items_returned, 0)) AS qt_items_returned_categoria,
    SAFE_DIVIDE(
      SUM(COALESCE(tpm.qt_items_returned, 0)),
      SUM(spm.qt_items_vendidos)
    ) AS td_rate_categoria
  FROM sales_product_metrics spm
  LEFT JOIN td_product_metrics tpm
    ON spm.product_name = tpm.product_name
   AND spm.category = tpm.category
   AND spm.gender = tpm.gender
  GROUP BY spm.category
),

-- qt_reversas_tag: COUNT DISTINCT (order_name|sku) por tag — não infla T&D.
-- Uma reversa com N tags contribui com 1 para o contador de cada tag.
tag_counts AS (
  SELECT
    tj.product_name,
    tj.category,
    tj.gender,
    rt.tag AS problema_tag,
    COUNT(DISTINCT CONCAT(tj.order_name, '|', tj.sku)) AS qt_reversas_tag
  FROM td_joined tj
  JOIN reversas_tag rt
    ON rt.order_name = tj.order_name
   AND rt.sku = tj.sku
  WHERE rt.tag IS NOT NULL
  GROUP BY tj.product_name, tj.category, tj.gender, rt.tag
),

-- Denominador correto para pct de tags: total de reversas distintas do produto,
-- independente de terem ou não tags. Evita que pct_top_3_total ultrapasse 100%.
td_reversas_total AS (
  SELECT
    product_name,
    category,
    gender,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_total
  FROM td_joined
  GROUP BY product_name, category, gender
),

tag_ranked AS (
  SELECT
    tc.*,
    rt.qt_reversas_total,
    ROW_NUMBER() OVER (
      PARTITION BY tc.product_name, tc.category, tc.gender
      ORDER BY tc.qt_reversas_tag DESC, tc.problema_tag
    ) AS tag_rank
  FROM tag_counts tc
  JOIN td_reversas_total rt
    ON tc.product_name = rt.product_name
   AND tc.category = rt.category
   AND tc.gender = rt.gender
  WHERE tc.problema_tag NOT IN (
    'caimento_bom',
    'conforto_positivo',
    'feedback_positivo_geral',
    'modelagem_boa',
    'tamanho_ideal',
    'tecido_qualidade_boa'
  )
),

top_tags AS (
  SELECT
    product_name,
    category,
    gender,

    MAX(IF(tag_rank = 1, problema_tag, NULL)) AS top_1_problema,
    MAX(IF(tag_rank = 1, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_1_pct,

    MAX(IF(tag_rank = 2, problema_tag, NULL)) AS top_2_problema,
    MAX(IF(tag_rank = 2, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_2_pct,

    MAX(IF(tag_rank = 3, problema_tag, NULL)) AS top_3_problema,
    MAX(IF(tag_rank = 3, SAFE_DIVIDE(qt_reversas_tag, qt_reversas_total), NULL)) AS top_3_pct,

    SAFE_DIVIDE(
      SUM(IF(tag_rank <= 3, qt_reversas_tag, 0)),
      MAX(qt_reversas_total)
    ) AS pct_top_3_total
  FROM tag_ranked
  WHERE tag_rank <= 3
  GROUP BY product_name, category, gender
),

color_concentration AS (
  SELECT
    product_name,
    category,
    gender,
    color,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_color,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) / SUM(COUNT(DISTINCT CONCAT(order_name, '|', sku))) OVER (
      PARTITION BY product_name, category, gender
    ) AS pct_reversas_color
  FROM td_joined
  WHERE color IS NOT NULL
  GROUP BY product_name, category, gender, color
),

main_color AS (
  SELECT
    product_name,
    category,
    gender,
    color AS principal_cor_afetada,
    pct_reversas_color AS principal_cor_pct
  FROM color_concentration
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_name, category, gender
    ORDER BY pct_reversas_color DESC, color
  ) = 1
),

size_concentration AS (
  SELECT
    product_name,
    category,
    gender,
    size,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) AS qt_reversas_size,
    COUNT(DISTINCT CONCAT(order_name, '|', sku)) / SUM(COUNT(DISTINCT CONCAT(order_name, '|', sku))) OVER (
      PARTITION BY product_name, category, gender
    ) AS pct_reversas_size
  FROM td_joined
  WHERE size IS NOT NULL
  GROUP BY product_name, category, gender, size
),

main_size AS (
  SELECT
    product_name,
    category,
    gender,
    size AS principal_tamanho_afetado,
    pct_reversas_size AS principal_tamanho_pct
  FROM size_concentration
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY product_name, category, gender
    ORDER BY pct_reversas_size DESC, size
  ) = 1
),

comments_sample AS (
  SELECT
    product_name,
    category,
    gender,
    STRING_AGG(client_comment, ' || ' LIMIT 10) AS comentarios_amostra
  FROM (
    SELECT DISTINCT
      product_name,
      category,
      gender,
      client_comment
    FROM td_joined
    WHERE client_comment IS NOT NULL
      AND LENGTH(TRIM(client_comment)) > 0
  )
  GROUP BY product_name, category, gender
),

trend_base AS (
  SELECT
    product_name,
    category,
    gender,

    COUNT(DISTINCT IF(
      data_reversa >= DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 3 MONTH),
      CONCAT(order_name, '|', sku),
      NULL
    )) AS reversas_ultimos_3m,

    COUNT(DISTINCT IF(
      data_reversa >= DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 6 MONTH)
      AND data_reversa < DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 3 MONTH),
      CONCAT(order_name, '|', sku),
      NULL
    )) AS reversas_3m_anteriores
  FROM td_joined
  GROUP BY product_name, category, gender
),

trend_classified AS (
  SELECT
    *,
    CASE
      WHEN reversas_ultimos_3m + reversas_3m_anteriores < 5 THEN 'Sem volume para tendência'
      WHEN reversas_3m_anteriores = 0 AND reversas_ultimos_3m > 0 THEN 'Apareceu nos últimos 3 meses'
      WHEN SAFE_DIVIDE(reversas_ultimos_3m - reversas_3m_anteriores, reversas_3m_anteriores) >= 0.25 THEN 'Aumentou nos últimos 3 meses'
      WHEN SAFE_DIVIDE(reversas_ultimos_3m - reversas_3m_anteriores, reversas_3m_anteriores) <= -0.25 THEN 'Caiu nos últimos 3 meses'
      ELSE 'Estável'
    END AS tendencia_reversas
  FROM trend_base
),

-- ─── Sell-through: mapeamento SKU → product_name ─────────────────────────────────────

sku_map_st AS (
  SELECT DISTINCT
    product_name,
    sku
  FROM `insider-data-lake.integrated.skus`
  WHERE product_name IS NOT NULL
    AND sku IS NOT NULL
),

-- ─── Sell-through: primeira data de venda por produto ────────────────────────────────

first_sale AS (
  SELECT
    product_name,
    ANY_VALUE(first_sale_date) AS first_sale_date
  FROM `insider-data-lake.sop_silver.portfolio_skp_clustering`
  WHERE first_sale_date IS NOT NULL
  GROUP BY product_name
),

-- ─── Sell-through: unidades vendidas em janelas de 30 / 60 / 90 dias ─────────────────

sales_windows AS (
  SELECT
    sm.product_name,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 30 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_30d,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 60 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_60d,
    SUM(CASE
      WHEN DATE(o.processed_at, 'America/Sao_Paulo')
             BETWEEN fs.first_sale_date
             AND DATE_ADD(fs.first_sale_date, INTERVAL 90 DAY)
      THEN oi.quantity ELSE 0
    END) AS sold_90d
  FROM sku_map_st sm
  JOIN first_sale fs ON sm.product_name = fs.product_name
  JOIN `insider-data-lake.business.insider_order_items` oi ON sm.sku = oi.sku
  JOIN `insider-data-lake.business.insider_orders` o ON oi.order_id = o.order_id
  WHERE o.is_cancelled = FALSE
    AND o.order_status = 'paid'
  GROUP BY 1
),

-- ─── Sell-through: estoque físico no dia de primeira venda ───────────────────────────

launch_stock AS (
  SELECT
    sm.product_name,
    SUM(st.physical_stock) AS initial_stock
  FROM sku_map_st sm
  JOIN first_sale fs ON sm.product_name = fs.product_name
  JOIN `insider-data-lake.integrated.stock` st
    ON sm.sku = st.sku
   AND st.stock_date = fs.first_sale_date
  GROUP BY 1
),

-- ─── Sell-through: razão vendas / estoque inicial ────────────────────────────────────

sell_through AS (
  SELECT
    sw.product_name,
    SAFE_DIVIDE(sw.sold_30d, ls.initial_stock) AS sell_through_30d,
    SAFE_DIVIDE(sw.sold_60d, ls.initial_stock) AS sell_through_60d,
    SAFE_DIVIDE(sw.sold_90d, ls.initial_stock) AS sell_through_90d
  FROM sales_windows sw
  LEFT JOIN launch_stock ls ON sw.product_name = ls.product_name
),

-- ─── MC3: campos diretos de eval_produto_portfolio ───────────────────────────────────

mc3_base AS (
  SELECT
    product_name,
    ANY_VALUE(mc3_ratio) AS mc3_ratio,
    ANY_VALUE(mc3_ratio_cat4) AS mc3_ratio_cat4,
    ANY_VALUE(mc3_ratio_portfolio) AS mc3_ratio_portfolio,
    ANY_VALUE(net_profit_after_marketing_costs) AS net_profit_after_marketing_costs
  FROM `insider-data-lake.sop_bronze.eval_produto_portfolio`
  WHERE product_name IS NOT NULL
  GROUP BY product_name
),

-- === SCORING ==============================================================

percentiles AS (
  SELECT
    *,

    CUME_DIST() OVER (ORDER BY receita_liquida) AS percentil_receita,
    CUME_DIST() OVER (ORDER BY qt_items_vendidos) AS percentil_unidades,
    CUME_DIST() OVER (ORDER BY td_rate) AS percentil_td_rate,
    CUME_DIST() OVER (ORDER BY qt_items_returned) AS percentil_volume_td,
    CUME_DIST() OVER (ORDER BY delta_vs_categoria) AS percentil_delta_vs_categoria,

    -- Percentis de sell-through (NULL se sem cobertura — fallback no scored CTE)
    CUME_DIST() OVER (ORDER BY sell_through_30d) AS percentil_sell_through_30d,
    CUME_DIST() OVER (ORDER BY sell_through_60d) AS percentil_sell_through_60d,
    CUME_DIST() OVER (ORDER BY sell_through_90d) AS percentil_sell_through_90d,

    -- Receita média mensal vs categoria: percentil particionado por category
    CUME_DIST() OVER (
      PARTITION BY category
      ORDER BY receita_media_mensal_vs_categoria
    ) AS percentil_receita_vs_categoria,

    -- Sub-scores MC3 (escala discreta 0.25 / 0.50 / 0.75 / 1.00)
    CASE
      WHEN mc3_ratio IS NULL OR mc3_ratio_cat4 IS NULL THEN NULL
      WHEN mc3_ratio >= mc3_ratio_cat4             THEN 1.00
      WHEN mc3_ratio >= 0.90 * mc3_ratio_cat4      THEN 0.75
      WHEN mc3_ratio >= 0.75 * mc3_ratio_cat4      THEN 0.50
      ELSE 0.25
    END AS mc3_vs_categoria_score,

    CASE
      WHEN mc3_ratio IS NULL OR mc3_ratio_portfolio IS NULL THEN NULL
      WHEN mc3_ratio >= mc3_ratio_portfolio          THEN 1.00
      WHEN mc3_ratio >= 0.90 * mc3_ratio_portfolio   THEN 0.75
      WHEN mc3_ratio >= 0.75 * mc3_ratio_portfolio   THEN 0.50
      ELSE 0.25
    END AS mc3_vs_portfolio_score,

    CUME_DIST() OVER (ORDER BY net_profit_after_marketing_costs) AS representatividade_mc3_score

  FROM (
    SELECT
      spm.product_name,
      spm.category,
      spm.gender,
      spm.portfolio_cluster,
      spm.portfolio_cluster_payload,

      spm.qt_pedidos,
      spm.qt_skus,
      spm.qt_items_vendidos,
      spm.receita_liquida,

      COALESCE(tpm.qt_reversas, 0) AS qt_reversas,
      COALESCE(tpm.qt_items_returned, 0) AS qt_items_returned,
      COALESCE(tpm.qt_trocas, 0) AS qt_trocas,
      COALESCE(tpm.qt_devolucoes, 0) AS qt_devolucoes,
      COALESCE(tpm.qt_reversas_fisico, 0) AS qt_reversas_fisico,
      COALESCE(tpm.qt_reversas_logistico, 0) AS qt_reversas_logistico,
      COALESCE(tpm.valor_troca, 0) AS valor_troca,
      COALESCE(tpm.valor_devolucao, 0) AS valor_devolucao,

      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos) AS td_rate,

      cm.td_rate_categoria,
      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos) - cm.td_rate_categoria AS delta_vs_categoria,
      SAFE_DIVIDE(
        SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), spm.qt_items_vendidos),
        cm.td_rate_categoria
      ) AS ratio_vs_categoria,

      SAFE_DIVIDE(spm.receita_liquida, SUM(spm.receita_liquida) OVER ()) AS share_receita_portfolio,
      SAFE_DIVIDE(spm.qt_items_vendidos, SUM(spm.qt_items_vendidos) OVER ()) AS share_unidades_portfolio,
      SAFE_DIVIDE(COALESCE(tpm.qt_items_returned, 0), SUM(COALESCE(tpm.qt_items_returned, 0)) OVER ()) AS share_td_portfolio,

      tt.top_1_problema,
      tt.top_1_pct,
      tt.top_2_problema,
      tt.top_2_pct,
      tt.top_3_problema,
      tt.top_3_pct,
      tt.pct_top_3_total,

      mc.principal_cor_afetada,
      mc.principal_cor_pct,
      ms.principal_tamanho_afetado,
      ms.principal_tamanho_pct,

      tb.reversas_ultimos_3m,
      tb.reversas_3m_anteriores,
      tb.tendencia_reversas,

      cs.comentarios_amostra,

      -- Sell-through (NULL quando sem cobertura de SKU/estoque)
      st.sell_through_30d,
      st.sell_through_60d,
      st.sell_through_90d,

      -- Receita média mensal vs média da categoria (ratio; usado para CUME_DIST)
      SAFE_DIVIDE(
        spm.receita_liquida,
        NULLIF(AVG(spm.receita_liquida) OVER (PARTITION BY spm.category), 0)
      ) AS receita_media_mensal_vs_categoria,

      -- MC3 fields (NULL quando produto não está em eval_produto_portfolio)
      mb.mc3_ratio,
      mb.mc3_ratio_cat4,
      mb.mc3_ratio_portfolio,
      mb.net_profit_after_marketing_costs

    FROM sales_product_metrics spm
    LEFT JOIN td_product_metrics tpm
      ON spm.product_name = tpm.product_name
     AND spm.category = tpm.category
     AND spm.gender = tpm.gender
    LEFT JOIN category_metrics cm
      ON spm.category = cm.category
    LEFT JOIN top_tags tt
      ON spm.product_name = tt.product_name
     AND spm.category = tt.category
     AND spm.gender = tt.gender
    LEFT JOIN main_color mc
      ON spm.product_name = mc.product_name
     AND spm.category = mc.category
     AND spm.gender = mc.gender
    LEFT JOIN main_size ms
      ON spm.product_name = ms.product_name
     AND spm.category = ms.category
     AND spm.gender = ms.gender
    LEFT JOIN trend_classified tb
      ON spm.product_name = tb.product_name
     AND spm.category = tb.category
     AND spm.gender = tb.gender
    LEFT JOIN comments_sample cs
      ON spm.product_name = cs.product_name
     AND spm.category = cs.category
     AND spm.gender = cs.gender
    LEFT JOIN sell_through st
      ON spm.product_name = st.product_name
    LEFT JOIN mc3_base mb
      ON spm.product_name = mb.product_name
  )
),

scored AS (
  SELECT
    *,

    0.5 * percentil_td_rate
      + 0.3 * percentil_volume_td
      + 0.2 * percentil_delta_vs_categoria AS td_score,

    -- commercial_score_v2 = 0.70 × tracao_vendas_score + 0.30 × mc3_score.
    -- Fallback (a): sell-through indisponível → usa percentil_receita/percentil_unidades.
    -- Fallback (b): mc3 indisponível → usa apenas tracao_vendas_score com peso total.
    CASE
      WHEN mc3_vs_categoria_score IS NOT NULL
      THEN
        0.70 * COALESCE(
            0.30 * percentil_sell_through_30d
              + 0.30 * percentil_sell_through_60d
              + 0.25 * percentil_sell_through_90d
              + 0.15 * percentil_receita_vs_categoria,
            0.60 * percentil_receita + 0.40 * percentil_unidades
          )
          + 0.30 * (
              0.50 * mc3_vs_categoria_score
                + 0.30 * mc3_vs_portfolio_score
                + 0.20 * representatividade_mc3_score
            )
      ELSE
        COALESCE(
          0.30 * percentil_sell_through_30d
            + 0.30 * percentil_sell_through_60d
            + 0.25 * percentil_sell_through_90d
            + 0.15 * percentil_receita_vs_categoria,
          0.60 * percentil_receita + 0.40 * percentil_unidades
        )
    END AS commercial_score_v2,

    -- priority_score expande td_score e commercial_score_v2 inline (BigQuery não
    -- permite referência a alias da mesma cláusula SELECT).
    0.50 * (
      0.5 * percentil_td_rate
        + 0.3 * percentil_volume_td
        + 0.2 * percentil_delta_vs_categoria
    )
    + 0.50 * (
      CASE
        WHEN mc3_vs_categoria_score IS NOT NULL
        THEN
          0.70 * COALESCE(
              0.30 * percentil_sell_through_30d
                + 0.30 * percentil_sell_through_60d
                + 0.25 * percentil_sell_through_90d
                + 0.15 * percentil_receita_vs_categoria,
              0.60 * percentil_receita + 0.40 * percentil_unidades
            )
            + 0.30 * (
                0.50 * mc3_vs_categoria_score
                  + 0.30 * mc3_vs_portfolio_score
                  + 0.20 * representatividade_mc3_score
              )
        ELSE
          COALESCE(
            0.30 * percentil_sell_through_30d
              + 0.30 * percentil_sell_through_60d
              + 0.25 * percentil_sell_through_90d
              + 0.15 * percentil_receita_vs_categoria,
            0.60 * percentil_receita + 0.40 * percentil_unidades
          )
      END
    ) AS priority_score
  FROM percentiles
),

final AS (
  SELECT
    *,

    CASE
      -- Volume mínimo estatístico verificado primeiro — guarda contra ruído.
      WHEN qt_items_vendidos < (SELECT min_items_vendidos FROM params)
        OR qt_items_returned < (SELECT min_items_returned FROM params)
        THEN 'Sem evidência suficiente'

      -- Alta dor + relevância comercial → ação imediata.
      WHEN td_score >= 0.70
        AND commercial_score_v2 >= 0.60
        THEN 'Priorizar melhoria'

      -- Alta dor, baixa tração comercial → vigilância.
      WHEN td_score >= 0.70
        AND commercial_score_v2 < 0.60
        THEN 'Monitorar'

      -- Produto de alta relevância com dor moderada → alerta preventivo.
      WHEN commercial_score_v2 >= 0.70
        AND td_score >= 0.50
        AND td_score < 0.70
        THEN 'Alerta em produto relevante'

      -- Dor baixa ou produto irrelevante → não priorizar.
      WHEN td_score < 0.50
        OR commercial_score_v2 < 0.40
        THEN 'Não priorizar agora'

      -- Zona cinza: td_score ∈ [0.50, 0.70) e commercial_score_v2 ∈ [0.40, 0.70).
      -- Dor e tração medianas — acompanhar sem ação imediata.
      ELSE 'Monitorar'
    END AS sinal_priorizacao,

    CONCAT(
      'Top problemas: ',
      COALESCE(top_1_problema, 'sem tag'),
      IF(top_2_problema IS NOT NULL, CONCAT(', ', top_2_problema), ''),
      IF(top_3_problema IS NOT NULL, CONCAT(', ', top_3_problema), ''),
      '. ',
      'Top 3 concentram ',
      CAST(ROUND(100 * COALESCE(pct_top_3_total, 0), 1) AS STRING),
      '% das reversas com tag. ',
      IF(
        principal_cor_afetada IS NOT NULL AND principal_cor_pct >= 0.50,
        CONCAT('Há concentração relevante na cor ', principal_cor_afetada, '. '),
        ''
      ),
      IF(
        principal_tamanho_afetado IS NOT NULL AND principal_tamanho_pct >= 0.50,
        CONCAT('Há concentração relevante no tamanho ', principal_tamanho_afetado, '. '),
        ''
      ),
      IF(
        tendencia_reversas IS NOT NULL,
        CONCAT('Tendência: ', tendencia_reversas, '.'),
        ''
      )
    ) AS resumo_pre_llm

  FROM scored
)

SELECT
  product_name,
  category,
  gender,
  portfolio_cluster,

  sinal_priorizacao,
  priority_score,
  td_score,
  commercial_score_v2,

  qt_pedidos,
  qt_skus,
  qt_items_vendidos,
  receita_liquida,

  qt_reversas,
  qt_items_returned,
  qt_trocas,
  qt_devolucoes,
  qt_reversas_fisico,
  qt_reversas_logistico,
  valor_troca,
  valor_devolucao,

  td_rate,
  td_rate_categoria,
  delta_vs_categoria,
  ratio_vs_categoria,

  share_receita_portfolio,
  share_unidades_portfolio,
  share_td_portfolio,

  top_1_problema,
  top_1_pct,
  top_2_problema,
  top_2_pct,
  top_3_problema,
  top_3_pct,
  pct_top_3_total,

  principal_cor_afetada,
  principal_cor_pct,
  principal_tamanho_afetado,
  principal_tamanho_pct,

  reversas_ultimos_3m,
  reversas_3m_anteriores,
  tendencia_reversas,

  comentarios_amostra,
  resumo_pre_llm,
  portfolio_cluster_payload

FROM final
ORDER BY
  CASE sinal_priorizacao
    WHEN 'Priorizar melhoria' THEN 1
    WHEN 'Alerta em produto relevante' THEN 2
    WHEN 'Monitorar' THEN 3
    WHEN 'Sem evidência suficiente' THEN 4
    WHEN 'Não priorizar agora' THEN 5
    ELSE 6
  END,
  priority_score DESC,
  qt_items_returned DESC;
"""

In [81]:
# CTEs compartilhados entre EXECUTIVE_QUERY e ANALYTICAL_QUERY.
# Alterar aqui propaga para ambas as queries.
# ⚠️ Não alterar lógica de filtragem sem revisão da metodologia.
_SHARED_CTES = """
WITH params AS (
  SELECT
    DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 12 MONTH) AS start_date,
    CURRENT_DATE("America/Sao_Paulo") AS end_date,
    30 AS min_items_vendidos,
    5  AS min_items_returned
),

orders AS (
  SELECT DISTINCT
    o.order_id,
    o.order_name,
    o.processed_at,
    DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") AS data_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), MONTH)        AS mes_compra,
    DATE_TRUNC(DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo"), WEEK(MONDAY)) AS semana_compra
  FROM `insider-data-lake.business.insider_orders` o
  CROSS JOIN params p
  WHERE o.order_status = 'paid'
    AND o.is_cancelled = FALSE
    AND (
      o.coupon_code IS NULL OR (
        NOT STARTS_WITH(o.coupon_code, 'TF-')
        AND NOT STARTS_WITH(o.coupon_code, 'TFIN')
        AND NOT STARTS_WITH(o.coupon_code, 'IR')
        AND NOT (o.coupon_code LIKE '%Item errado%')
      )
    )
    AND o.order_name IS NOT NULL
    AND o.processed_at IS NOT NULL
    AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
    AND DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
),

order_items AS (
  SELECT
    i.order_id,
    i.sku,
    i.product_title,
    i.variant_color,
    i.variant_size,
    i.quantity
  FROM `insider-data-lake.business.insider_order_items` i
  WHERE i.sku IS NOT NULL
),

order_items_grouped AS (
  SELECT
    order_id,
    sku,
    ANY_VALUE(product_title)  AS product_title,
    ANY_VALUE(variant_color)  AS variant_color,
    ANY_VALUE(variant_size)   AS variant_size,
    SUM(quantity)             AS qt_items
  FROM order_items
  GROUP BY order_id, sku
),

sku_dim AS (
  SELECT
    sku,
    ANY_VALUE(product_name) AS product_name,
    ANY_VALUE(category)     AS category,
    ANY_VALUE(gender)       AS gender,
    ANY_VALUE(color)        AS color,
    ANY_VALUE(size)         AS size,
    ANY_VALUE(sku_state)    AS sku_state
  FROM `insider-data-lake.integrated.skus`
  WHERE sku IS NOT NULL
  GROUP BY sku
),

portfolio_clustering AS (
  SELECT
    product_name,
    ANY_VALUE(cluster) AS portfolio_cluster
  FROM `insider-data-lake.sop_silver.portfolio_skp_clustering`
  GROUP BY product_name
),

reversas_unicas AS (
  SELECT
    r.order_name,
    r.id_reversa,
    r.status,
    r.reverse_type,
    r.sku,
    r.return_reason,
    r.updated_at,
    r.created_at,
    DATE(r.created_at, "America/Sao_Paulo")                                          AS data_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), MONTH)                       AS mes_reversa,
    DATE_TRUNC(DATE(r.created_at, "America/Sao_Paulo"), WEEK(MONDAY))                AS semana_reversa,
    r.client_comment,
    SAFE_CAST(r.reverse_shipping_cost AS FLOAT64) AS reverse_shipping_cost,
    SAFE_CAST(r.retained_bonus        AS FLOAT64) AS retained_bonus,
    SAFE_CAST(r.exchange_value        AS FLOAT64) AS exchange_value,
    SAFE_CAST(r.refund_value          AS FLOAT64) AS refund_value,
    COALESCE(SAFE_CAST(r.return_quantity AS FLOAT64), 0) AS qt_items_returned
  FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br` r
  CROSS JOIN params p
  WHERE r.status <> 'Cancelado'
    AND r.sku IS NOT NULL
    AND r.return_reason IS NOT NULL
    AND r.id_reversa IS NOT NULL
    AND DATE(r.created_at, "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY r.order_name, r.id_reversa, r.sku
    ORDER BY r.updated_at DESC
  ) = 1
),

reversas_tags_latest AS (
  SELECT
    order_name,
    sku,
    tags,
    created_at,
    DATE_TRUNC(DATE(created_at), MONTH) AS ingestion_date
  FROM `insider-data-lake.sop_silver.return_reason_tags`
  WHERE order_name IS NOT NULL
    AND sku IS NOT NULL
    AND tags IS NOT NULL
  QUALIFY ROW_NUMBER() OVER (
    PARTITION BY order_name, sku
    ORDER BY created_at DESC
  ) = 1
),

reversas_tag AS (
  SELECT
    order_name,
    sku,
    tag,
    ingestion_date
  FROM reversas_tags_latest,
  UNNEST(tags) AS tag
  WHERE tag NOT IN (
    'caimento_bom', 'conforto_positivo', 'feedback_positivo_geral',
    'modelagem_boa', 'tamanho_ideal', 'tecido_qualidade_boa'
  )
)
"""


In [82]:
# Camada 1 — Tabela analítica item/reversa/tag  (otimizada)
# Objetivo: auditoria, drill-down e leitura qualitativa por SKU/produto.
# Grain: order_id × order_name × sku × problema_tag (múltiplas linhas por reversa com N tags)
# Escopo: APENAS itens que tiveram reversa ativa (INNER JOIN com reversas_unicas).
# Sem ORDER BY no BQ: ordenação delegada ao pandas (ver Bloco 4).
# Uso: requer LOAD_ANALITICA=True
ANALYTICAL_QUERY = _SHARED_CTES + """
-- ==============================================================================
-- CAMADA 1: TABELA ANALÍTICA ITEM / REVERSA / TAG  (otimizada)
-- Grain: order_id × order_name × sku × problema_tag
-- Escopo: APENAS itens que tiveram reversa ativa (INNER JOIN com reversas_unicas)
-- Sem ORDER BY: ordenação delegada ao pandas após load.
-- ==============================================================================

, return_reason_map AS (
  -- Mapeamento de return_reason → motivo_classificado e tipo_problema.
  -- CTE de lookup: evita duplicação dos CASE statements.
  -- ⚠️ Não alterar sem revisão — espelha a lógica da EXECUTIVE_QUERY.
  SELECT DISTINCT
    return_reason,
    CASE
      WHEN LOWER(return_reason) LIKE '%ficou grande%'                     THEN 'Tamanho'
      WHEN return_reason = 'Peça íntima'                                  THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Produto com defeito'                          THEN 'Defeito'
      WHEN return_reason = 'Arrependimento'                               THEN 'Desistência'
      WHEN return_reason = 'Recebi um produto errado'                     THEN 'Produto errado'
      WHEN LOWER(return_reason) LIKE '%ficou pequeno%'                    THEN 'Tamanho'
      WHEN return_reason = 'Cor diferente do esperado'                    THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Recebi novo pedido de reposição'              THEN 'Produto errado'
      WHEN return_reason = 'Recebi pedido duplicado'                      THEN 'Produto errado'
      WHEN return_reason = 'Arrependiemento'                              THEN 'Desistência'
      WHEN return_reason = 'Insatisfação (tamanho e cor inclusos)'        THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Arrependimento (tamanho e cor inclusos)'      THEN 'Insatisfação com o produto'
      WHEN return_reason LIKE '%Tamanho%'                                 THEN 'Tamanho'
      WHEN return_reason = 'Insatisfação com o produto'                   THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Falha na Entrega'                             THEN 'Problema na entrega'
      WHEN return_reason = 'Demora na entrega'                            THEN 'Problema na entrega'
      WHEN return_reason = 'Ficou Grande'                                 THEN 'Tamanho'
      WHEN return_reason = 'Ficou Pequeno'                                THEN 'Tamanho'
      WHEN return_reason = 'Não gostei do produto'                        THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Defeito'                                      THEN 'Defeito'
      WHEN return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Insatisfação'                                 THEN 'Insatisfação com o produto'
      WHEN return_reason = 'Defeitos'                                     THEN 'Defeito'
      WHEN return_reason = 'Desbotamento'                                 THEN 'Desbotamento'
      WHEN return_reason = 'Desistência'                                  THEN 'Desistência'
      WHEN return_reason = 'Não gostei da qualidade'                      THEN 'Insatisfação com o produto'
      WHEN return_reason IS NULL                                          THEN NULL
      ELSE 'Outros'
    END AS motivo_classificado,
    CASE
      WHEN LOWER(return_reason) LIKE '%ficou grande%'                     THEN 'Físico'
      WHEN return_reason = 'Peça íntima'                                  THEN 'Físico'
      WHEN return_reason = 'Produto com defeito'                          THEN 'Físico'
      WHEN return_reason = 'Arrependimento'                               THEN 'Desistência'
      WHEN return_reason = 'Recebi um produto errado'                     THEN 'Logístico'
      WHEN LOWER(return_reason) LIKE '%ficou pequeno%'                    THEN 'Físico'
      WHEN return_reason = 'Cor diferente do esperado'                    THEN 'Físico'
      WHEN return_reason = 'Recebi novo pedido de reposição'              THEN 'Logístico'
      WHEN return_reason = 'Recebi pedido duplicado'                      THEN 'Logístico'
      WHEN return_reason = 'Arrependiemento'                              THEN 'Desistência'
      WHEN return_reason = 'Insatisfação (tamanho e cor inclusos)'        THEN 'Físico'
      WHEN return_reason = 'Arrependimento (tamanho e cor inclusos)'      THEN 'Desistência'
      WHEN return_reason LIKE '%Tamanho%'                                 THEN 'Físico'
      WHEN return_reason = 'Insatisfação com o produto'                   THEN 'Físico'
      WHEN return_reason = 'Falha na Entrega'                             THEN 'Logístico'
      WHEN return_reason = 'Demora na entrega'                            THEN 'Logístico'
      WHEN return_reason = 'Ficou Grande'                                 THEN 'Físico'
      WHEN return_reason = 'Ficou Pequeno'                                THEN 'Físico'
      WHEN return_reason = 'Não gostei do produto'                        THEN 'Físico'
      WHEN return_reason = 'Defeito'                                      THEN 'Físico'
      WHEN return_reason = 'Peça íntima (calcinha, sutiã, meia ou cueca)' THEN 'Físico'
      WHEN return_reason = 'Insatisfação'                                 THEN 'Físico'
      WHEN return_reason = 'Defeitos'                                     THEN 'Físico'
      WHEN return_reason = 'Desbotamento'                                 THEN 'Físico'
      WHEN return_reason = 'Desistência'                                  THEN 'Desistência'
      WHEN return_reason = 'Não gostei da qualidade'                      THEN 'Físico'
      WHEN return_reason IS NULL                                          THEN NULL
      ELSE 'Outros'
    END AS tipo_problema
  FROM reversas_unicas
),

base_analitica AS (
  SELECT
    -- Datas
    o.data_compra,
    o.mes_compra,
    o.semana_compra,
    ru.data_reversa,
    ru.mes_reversa,
    ru.semana_reversa,

    -- Identificadores
    o.order_id,
    o.order_name,
    oi.sku,
    ru.id_reversa,

    -- Produto
    COALESCE(s.product_name, oi.product_title)    AS product_name,
    s.category,
    s.gender,
    COALESCE(s.color, oi.variant_color)            AS color,
    COALESCE(s.size,  oi.variant_size)             AS size,
    s.sku_state,
    oi.product_title,
    oi.variant_color,
    oi.variant_size,
    CONCAT(
      COALESCE(s.product_name, oi.product_title),
      ' ',
      COALESCE(s.color, oi.variant_color)
    )                                              AS variant_color_name,

    -- Cluster
    pc.portfolio_cluster,

    -- Reversa
    ru.status           AS reverse_status,
    ru.reverse_type,
    ru.return_reason,
    rm.motivo_classificado,
    rm.tipo_problema,
    ru.client_comment,
    CASE
      WHEN ru.reverse_type IS NULL                    THEN 'Sem reversa'
      WHEN LOWER(ru.reverse_type) LIKE '%troca%'      THEN 'Troca'
      WHEN LOWER(ru.reverse_type) LIKE '%devol%'      THEN 'Devolução'
      ELSE ru.reverse_type
    END                                            AS reverse_type_classificado,

    -- Tags
    rt.tag              AS problema_tag,
    rt.ingestion_date   AS tag_ingestion_month,

    -- Métricas
    oi.qt_items,
    ru.qt_items_returned,
    ru.reverse_shipping_cost,
    ru.retained_bonus,
    ru.exchange_value,
    ru.refund_value

  FROM reversas_unicas ru                       -- âncora: apenas itens com reversa
  JOIN orders o
    ON o.order_name = ru.order_name
  JOIN order_items_grouped oi
    ON oi.order_id  = o.order_id
   AND oi.sku       = ru.sku
  LEFT JOIN sku_dim s
    ON s.sku = oi.sku
  LEFT JOIN portfolio_clustering pc
    ON pc.product_name = COALESCE(s.product_name, oi.product_title)
  LEFT JOIN reversas_tag rt
    ON rt.order_name = ru.order_name
   AND rt.sku        = ru.sku
  LEFT JOIN return_reason_map rm
    ON rm.return_reason = ru.return_reason
  WHERE s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
)

SELECT *
FROM base_analitica
-- Sem ORDER BY: ordenação delegada ao pandas (ver Bloco 4).
"""


In [83]:
# Scorecard — Pilares do portfólio por produto
# Objetivo: buscar score_tracao_comercial, score_unit_economics, score_satisfacao_marca
# Fonte: sop_bronze.eval_produto_portfolio (escala 0–1 × 100)
# Grain: product_name
# Nota: eval_produto_portfolio NÃO tem coluna cluster — cluster vem de portfolio_skp_clustering.
SCORECARD_QUERY = """
  SELECT
    product_name,
    ROUND(ANY_VALUE(score_vendas_geral) * 100, 1)           AS score_tracao_comercial,
    ROUND(ANY_VALUE(score_viabilidade_financeira) * 100, 1)  AS score_unit_economics,
    ROUND(ANY_VALUE(score_satisf_cliente) * 100, 1)          AS score_satisfacao_marca
  FROM `insider-data-lake.sop_bronze.eval_produto_portfolio`
  GROUP BY product_name
"""


## 4. Execução da extração no BigQuery

Executa as queries SQL e gera os DataFrames principais.

**Entrada:** `EXECUTIVE_QUERY`, `ANALYTICAL_QUERY`, `SCORECARD_QUERY`
**Saída:** `executive_df`, `analytical_df` (ou `None` se `LOAD_ANALITICA=False`), `df_scores_raw`
**Quando mexer:** ao alterar projeto BigQuery ou método de autenticação.

> **Deepnote:** este bloco pode ser substituído por SQL blocks nativos conectados à integração BigQuery.
> O output esperado é um DataFrame chamado `executive_df` com grain `product_name × category × gender`.


In [84]:
from google.cloud import bigquery

# Autenticação via Application Default Credentials (ADC).
# Localmente: `gcloud auth application-default login` se necessário.
# No Deepnote: configure a integração BigQuery nas variáveis de ambiente.
client = bigquery.Client(project=PROJECT_ID)
print(f"✅ BigQuery client criado | projeto: {PROJECT_ID}")


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/auth/_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
Python(5549) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


✅ BigQuery client criado | projeto: insider-data-lake


In [85]:
if RUN_VALIDATION:
    # ============================================================
    # VALIDAÇÕES PÓS-MIGRAÇÃO v2 (2026-06-18)
    # Ponto 1 — order_id no DRE: JOIN com orders funciona?
    # Ponto 2 — non_refunded_quantity vs qt_items: magnitude da diferença
    # Ponto 3 — Múltiplas reversas por (order_name, sku) com dedup id_reversa
    # ============================================================

    # --- PONTO 1: order_id no DRE × business.insider_orders ---
    q_dre_join = """
    WITH dre_sample AS (
      SELECT DISTINCT dre.order_id
      FROM `insider-data-lake.fpa.analytical_dre` dre
      WHERE dre.order_id IS NOT NULL
      LIMIT 500000
    )
    SELECT
      COUNT(*) AS total_order_ids_dre,
      COUNTIF(o.order_id IS NOT NULL) AS matched_em_insider_orders,
      ROUND(COUNTIF(o.order_id IS NOT NULL) / COUNT(*) * 100, 2) AS pct_matched
    FROM dre_sample d
    LEFT JOIN `insider-data-lake.business.insider_orders` o
      ON d.order_id = o.order_id
     AND o.order_status = 'paid'
     AND o.is_cancelled = FALSE
     AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
    """
    df_p1 = client.query(q_dre_join).to_dataframe()
    print("=== Ponto 1 — DRE × insider_orders JOIN ===")
    print(df_p1.to_string(index=False))
    pct_match = df_p1['pct_matched'].iloc[0]
    print(f"\n→ {'✅ JOIN funciona bem' if pct_match > 50 else '⚠️ Baixo match — investigar'} ({pct_match:.1f}% dos order_ids do DRE batem com orders filtrados)")
    print()

    # --- PONTO 2: non_refunded_quantity vs qt_items bruto ---
    q_vol_compare = """
    WITH periodo AS (
      SELECT
        DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 12 MONTH) AS start_date,
        CURRENT_DATE("America/Sao_Paulo") AS end_date
    ),
    vendas_items AS (
      SELECT
        s.product_name,
        SUM(i.quantity) AS qt_items_bruto
      FROM `insider-data-lake.business.insider_orders` o
      JOIN `insider-data-lake.business.insider_order_items` i ON o.order_id = i.order_id
      JOIN `insider-data-lake.integrated.skus` s ON s.sku = i.sku
      CROSS JOIN periodo p
      WHERE o.order_status = 'paid'
        AND o.is_cancelled = FALSE
        AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
        AND DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
        AND i.sku IS NOT NULL
        AND s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
      GROUP BY s.product_name
    ),
    vendas_dre AS (
      SELECT
        s.product_name,
        SUM(dre.non_refunded_quantity) AS qt_non_refunded,
        SUM(dre.revenue_after_discounts) AS receita_dre
      FROM `insider-data-lake.fpa.analytical_dre` dre
      JOIN `insider-data-lake.business.insider_orders` o ON dre.order_id = o.order_id
      JOIN `insider-data-lake.integrated.skus` s ON dre.sku = s.sku
      CROSS JOIN periodo p
      WHERE o.order_status = 'paid'
        AND o.is_cancelled = FALSE
        AND o.store IN ('shopify_insider-world', 'shopify_insider-store-loja')
        AND DATE(TIMESTAMP(o.processed_at), "America/Sao_Paulo") BETWEEN p.start_date AND p.end_date
        AND s.sku_state IN ('ativo_perene', 'ativo_em_lancamento')
      GROUP BY s.product_name
    )
    SELECT
      COUNT(*) AS total_produtos,
      SUM(vi.qt_items_bruto) AS total_qt_items_bruto,
      SUM(vd.qt_non_refunded) AS total_non_refunded,
      ROUND(SAFE_DIVIDE(SUM(vd.qt_non_refunded), SUM(vi.qt_items_bruto)), 4) AS ratio_dre_vs_items,
      ROUND(SUM(vd.receita_dre)) AS total_receita_dre
    FROM vendas_items vi
    JOIN vendas_dre vd ON vi.product_name = vd.product_name
    """
    df_p2 = client.query(q_vol_compare).to_dataframe()
    print("=== Ponto 2 — non_refunded_quantity vs qt_items bruto ===")
    print(df_p2.to_string(index=False))
    ratio = df_p2['ratio_dre_vs_items'].iloc[0]
    print(f"\n→ ratio = {ratio:.4f}  ({'✅ DRE < items (exclui devolvidos — esperado)' if 0.7 < ratio < 1.05 else '⚠️ Ratio fora do esperado — investigar'})")
    print()

    # --- PONTO 3: Múltiplas reversas por (order_name, sku) com id_reversa ---
    q_multi_rev = """
    WITH deduped AS (
      SELECT
        order_name,
        sku,
        COUNT(DISTINCT id_reversa) AS n_reversas_distintas
      FROM `insider-lake-sensitive.prepared_br.prepared__troquecommerce_order_details_br`
      WHERE status <> 'Cancelado'
        AND sku IS NOT NULL
        AND id_reversa IS NOT NULL
        AND DATE(created_at, "America/Sao_Paulo")
            >= DATE_SUB(CURRENT_DATE("America/Sao_Paulo"), INTERVAL 12 MONTH)
      GROUP BY order_name, sku
    )
    SELECT
      COUNT(*) AS pares_order_sku,
      COUNTIF(n_reversas_distintas = 1) AS pares_com_1_reversa,
      COUNTIF(n_reversas_distintas > 1) AS pares_com_multiplas_reversas,
      ROUND(SAFE_DIVIDE(COUNTIF(n_reversas_distintas > 1), COUNT(*)) * 100, 2) AS pct_multi_reversa,
      ROUND(AVG(n_reversas_distintas), 3) AS media_reversas_por_par,
      MAX(n_reversas_distintas) AS max_reversas_por_par
    FROM deduped
    """
    df_p3 = client.query(q_multi_rev).to_dataframe()
    print("=== Ponto 3 — Múltiplas reversas por (order_name, sku) ===")
    print(df_p3.to_string(index=False))
    pct = df_p3['pct_multi_reversa'].iloc[0]
    media = df_p3['media_reversas_por_par'].iloc[0]
    if pct > 5:
        print(f"\n⚠️ {pct:.2f}% dos pares têm múltiplas reversas — COUNT DISTINCT protege, mas verificar se SUM(qt_items_returned) está correto")
    else:
        print(f"\n✅ {pct:.2f}% dos pares têm múltiplas reversas (média {media:.3f}/par) — impacto baixo, COUNT DISTINCT suficiente")

else:
    print("⏭️  RUN_VALIDATION=False — validações pós-migração puladas.")


⏭️  RUN_VALIDATION=False — validações pós-migração puladas.


In [86]:
# ── Camada 2: Tabela executiva por produto ────────────────────────────────────
print("Executando EXECUTIVE_QUERY...")
executive_df = client.query(EXECUTIVE_QUERY).to_dataframe()

print(f"\n✅ executive_df: {executive_df.shape[0]:,} linhas × {executive_df.shape[1]:,} colunas")
print(f"   Colunas: {list(executive_df.columns)}")
display(executive_df.head(3))


Executando EXECUTIVE_QUERY...


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



✅ executive_df: 117 linhas × 44 colunas
   Colunas: ['product_name', 'category', 'gender', 'portfolio_cluster', 'sinal_priorizacao', 'priority_score', 'td_score', 'commercial_score_v2', 'qt_pedidos', 'qt_skus', 'qt_items_vendidos', 'receita_liquida', 'qt_reversas', 'qt_items_returned', 'qt_trocas', 'qt_devolucoes', 'qt_reversas_fisico', 'qt_reversas_logistico', 'valor_troca', 'valor_devolucao', 'td_rate', 'td_rate_categoria', 'delta_vs_categoria', 'ratio_vs_categoria', 'share_receita_portfolio', 'share_unidades_portfolio', 'share_td_portfolio', 'top_1_problema', 'top_1_pct', 'top_2_problema', 'top_2_pct', 'top_3_problema', 'top_3_pct', 'pct_top_3_total', 'principal_cor_afetada', 'principal_cor_pct', 'principal_tamanho_afetado', 'principal_tamanho_pct', 'reversas_ultimos_3m', 'reversas_3m_anteriores', 'tendencia_reversas', 'comentarios_amostra', 'resumo_pre_llm', 'portfolio_cluster_payload']


,product_name,category,gender,portfolio_cluster,sinal_priorizacao,priority_score,td_score,commercial_score_v2,qt_pedidos,qt_skus,qt_items_vendidos,receita_liquida,qt_reversas,qt_items_returned,qt_trocas,qt_devolucoes,qt_reversas_fisico,qt_reversas_logistico,valor_troca,valor_devolucao,td_rate,td_rate_categoria,delta_vs_categoria,ratio_vs_categoria,share_receita_portfolio,share_unidades_portfolio,share_td_portfolio,top_1_problema,top_1_pct,top_2_problema,top_2_pct,top_3_problema,top_3_pct,pct_top_3_total,principal_cor_afetada,principal_cor_pct,principal_tamanho_afetado,principal_tamanho_pct,reversas_ultimos_3m,reversas_3m_anteriores,tendencia_reversas,comentarios_amostra,resumo_pre_llm,portfolio_cluster_payload
0,Calça FutureForm Feminino,Calça,female,Hero,Priorizar melhoria,0.9326,0.9299,0.9352,21969,15,"22,055.0905","7,029,505.0617",3213,"3,218.0000",2701,489,2941,17,"1,168,584.6600","302,608.3100",0.1459,0.1055,0.0404,1.3834,0.0217,0.0083,0.0247,caimento_ruim,0.1148,modelagem_ruim,0.0918,tamanho_pequeno,0.0514,0.2580,Preto,0.5353,M,0.3617,747,937,Estável,a cor é bem diferente do site e não recebi a c...,"Top problemas: caimento_ruim, modelagem_ruim, ...","{""cluster"":""Hero"",""product_name"":""Calça Future..."
1,Camisa FutureForm Masculino,Camisa Social,male,Core,Priorizar melhoria,0.9143,0.8991,0.9294,11485,20,"12,916.0351","4,250,771.3057",1973,"1,999.0000",1746,216,1906,7,"895,081.5800","200,701.4600",0.1548,0.1277,0.0271,1.2119,0.0131,0.0048,0.0153,tamanho_grande,0.1221,caimento_ruim,0.0299,modelagem_ruim,0.0264,0.1784,Ocean Blue,0.3137,G,0.3284,734,442,Aumentou nos últimos 3 meses,Acreditava que o material das roupas fossem me...,"Top problemas: tamanho_grande, caimento_ruim, ...","{""cluster"":""Core"",""product_name"":""Camisa Futur..."
2,Shorts Kyoto Feminino,Shorts,female,Core,Priorizar melhoria,0.8920,0.8538,0.9302,12267,15,"12,679.5089","2,566,026.1668",2113,"2,118.0000",1761,330,1999,7,"608,980.5500","182,775.3900",0.1670,0.1670,0.0000,1.0000,0.0079,0.0047,0.0163,caimento_ruim,0.0842,tamanho_grande,0.0800,tamanho_pequeno,0.0743,0.2385,Preto,0.3928,M,0.3743,620,612,Estável,Não veste bem || comprei 3 shorts iguais taman...,"Top problemas: caimento_ruim, tamanho_grande, ...","{""cluster"":""Core"",""product_name"":""Shorts Kyoto..."


In [87]:
# ── Camada 1: Tabela analítica (opcional) ────────────────────────────────────
# Query pesada (~60s). Necessária para QA de duplicação de tags e aba amostra_analitica.
if LOAD_ANALITICA:
    print("Executando ANALYTICAL_QUERY (LOAD_ANALITICA=True)...")
    analytical_df = (
        client.query(ANALYTICAL_QUERY)
        .to_dataframe()
        .sort_values(
            ["data_compra", "order_name", "sku", "problema_tag"],
            ascending=[False, True, True, True],
            na_position="last",
        )
        .reset_index(drop=True)
    )
    print(f"\n✅ analytical_df: {analytical_df.shape[0]:,} linhas × {analytical_df.shape[1]:,} colunas")
    print(f"   Colunas: {list(analytical_df.columns)}")
else:
    analytical_df = None
    print("⏭️  LOAD_ANALITICA=False — analytical_df não carregado (None).")


Executando ANALYTICAL_QUERY (LOAD_ANALITICA=True)...


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(



✅ analytical_df: 130,803 linhas × 36 colunas
   Colunas: ['data_compra', 'mes_compra', 'semana_compra', 'data_reversa', 'mes_reversa', 'semana_reversa', 'order_id', 'order_name', 'sku', 'id_reversa', 'product_name', 'category', 'gender', 'color', 'size', 'sku_state', 'product_title', 'variant_color', 'variant_size', 'variant_color_name', 'portfolio_cluster', 'reverse_status', 'reverse_type', 'return_reason', 'motivo_classificado', 'tipo_problema', 'client_comment', 'reverse_type_classificado', 'problema_tag', 'tag_ingestion_month', 'qt_items', 'qt_items_returned', 'reverse_shipping_cost', 'retained_bonus', 'exchange_value', 'refund_value']


In [88]:
# ── Scorecard de portfólio ────────────────────────────────────────────────────
print("Executando SCORECARD_QUERY...")
df_scores_raw = client.query(SCORECARD_QUERY).to_dataframe()

print(f"\n✅ df_scores_raw: {df_scores_raw.shape[0]:,} produto(s) com scorecard")
print(f"   Colunas: {list(df_scores_raw.columns)}")


Executando SCORECARD_QUERY...

✅ df_scores_raw: 126 produto(s) com scorecard
   Colunas: ['product_name', 'score_tracao_comercial', 'score_unit_economics', 'score_satisfacao_marca']


/Users/insider/LA_Coding_Projects/.venv-1/lib/python3.13/site-packages/google/cloud/bigquery/table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


## 5. Funções utilitárias e regras de negócio

Funções incorporadas de `td_analysis_functions.py` para eliminar dependência do arquivo externo.

**Entrada:** nenhuma (definições)
**Saída:** constantes, dataclasses e funções disponíveis para os blocos seguintes
**Quando mexer:** ao corrigir bugs — manter `td_analysis_functions.py` em sincronia.

> ⚠️ **Não alterar thresholds, labels, mapeamentos de tags ou lógica de classificação sem revisão.**
>
> As funções abaixo foram incorporadas de `td_analysis_functions.py` para facilitar execução
> e manutenção em Deepnote. O arquivo original permanece como referência histórica.

### Função omitida intencionalmente
- `create_product_summary_prompt`: gera prompt para LLM mas não é chamada pelo pipeline.
  Preservada em `td_analysis_functions.py` caso seja necessária no futuro.

### Estrutura
- **5.1** Constantes, dataclasses e mapeamentos de negócio
- **5.2** Funções de validação, coerção e preparação
- **5.3** Funções analíticas (classificação, ranking, benchmark, tags)
- **5.4** Funções de síntese heurística (tweets, scorecard view, mensagem)
- **5.5** Formatação do workbook e exportação Excel (debug)


### 5.1 — Constantes, dataclasses e mapeamentos de negócio

`PrioritizationThresholds`, `TAG_TO_ACTION`, `POSITIVE_TAGS`, `LOGISTICA_TAGS`,
`EXECUTIVE_REQUIRED_COLUMNS`, `ANALYTICAL_REQUIRED_COLUMNS`, `SINAL_DESCRICAO`.


In [89]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Mapping, Sequence

import numpy as np
import pandas as pd


EXECUTIVE_REQUIRED_COLUMNS = {
    "product_name",
    "category",
    "sinal_priorizacao",
    "priority_score",
    "td_score",
    "commercial_score_v2",
    "qt_items_vendidos",
    "receita_liquida",
    "qt_items_returned",
    "td_rate",
    "td_rate_categoria",
    "delta_vs_categoria",
}

# Name of the commercial-score column produced by the SQL pipeline.
# v2 = 0.70 × tracao_vendas_score + 0.30 × mc3_score (with fallbacks).
COMMERCIAL_SCORE_COL: str = "commercial_score_v2"

ANALYTICAL_REQUIRED_COLUMNS = {
    "order_name",
    "sku",
    "product_name",
    "category",
    "color",
    "size",
    "tipo_problema",
    "problema_tag",
    "client_comment",
    "qt_items_returned",
}


@dataclass(frozen=True)
class PrioritizationThresholds:
    """Thresholds used to classify products in Python.

    Keep these aligned with the SQL params and CASE statement unless you are
    deliberately running a sensitivity analysis.
    """

    min_items_vendidos: int = 30
    min_items_returned: int = 5
    td_score_prioritize: float = 0.70
    commercial_score_prioritize: float = 0.60
    commercial_score_alert: float = 0.70
    td_score_alert_min: float = 0.50
    low_td_score: float = 0.50
    low_commercial_score: float = 0.40


# ---------------------------------------------------------------------------
# Tag → action type mapping
# ---------------------------------------------------------------------------

TAG_TO_ACTION: dict[str, str] = {
    # Modelagem
    "caimento_ruim": "Modelagem",
    "modelagem_ruim": "Modelagem",
    "sustentacao_ruim": "Modelagem",
    # Grade
    "comprimento_curto": "Grade",
    "comprimento_longo": "Grade",
    "tamanho_grande": "Grade",
    "tamanho_pequeno": "Grade",
    "tamanho_pepequeno": "Grade",
    # Tecido
    "tecido_fino": "Tecido",
    "tecido_grosso": "Tecido",
    "tecido_transparente": "Tecido",
    "tecido_qualidade_ruim": "Tecido",
    "tecido_marca_corpo": "Tecido",
    "tecido_quente": "Tecido",
    "tecido_amassa": "Tecido",
    "pilling_bolinhas": "Tecido",
    "encolhimento": "Tecido",
    # Defeito → Investigação adicional
    "defeito_costura": "Investigação adicional",
    "defeito_aviamento": "Investigação adicional",
    "defeito_fio_puxado": "Investigação adicional",
    "defeito_furo_rasgo": "Investigação adicional",
    "defeito_gola": "Investigação adicional",
    "defeito_mancha": "Investigação adicional",
    # Ambíguo → Investigação adicional
    "conforto_negativo": "Investigação adicional",
    # Comunicação
    "cor_diferente_site": "PDP/Comunicação",
}

POSITIVE_TAGS: frozenset[str] = frozenset({
    "caimento_bom",
    "conforto_positivo",
    "feedback_positivo_geral",
    "modelagem_boa",
    "tamanho_ideal",
    "tecido_qualidade_boa",
})

LOGISTICA_TAGS: frozenset[str] = frozenset({
    "atendimento_ineficiente",
    "logistica_adiantamento",
    "logistica_atraso",
    "logistica_embalagem",
    "logistica_item_errado",
    "logistica_item_faltando",
    "provador_virtual_impreciso",
})

SINAL_DESCRICAO: dict[str, str] = {
    "Priorizar melhoria": "alta T&D e alta tração comercial",
    "Alerta em produto relevante": "produto relevante com T&D em escalada",
    "Monitorar": "alta T&D, baixa tração comercial",
    "Não priorizar agora": "sem urgência no momento",
    "Sem evidência suficiente": "volume insuficiente para diagnóstico",
}

# ---------------------------------------------------------------------------
# Workbook formatting constants (openpyxl)
# ---------------------------------------------------------------------------

_TAB_COLORS: dict[str, str] = {
    "farol_executivo": "4472C4",
    "resumo_farol": "70AD47",
    "benchmark_categoria": "9DC3E6",
    "resumo_tags": "9DC3E6",
    "priorizar_melhoria": "C00000",
    "tweets_analiticos": "C00000",
    "relatorio_pf": "ED7D31",
    "amostra_analitica": "A5A5A5",
}

_SINAL_ROW_COLORS: dict[str, str] = {
    "Priorizar melhoria": "FFD7D7",
    "Alerta em produto relevante": "FFF9D7",
    "Monitorar": "FFE8CC",
    "Não priorizar agora": "F2F2F2",
    "Sem evidência suficiente": "FFFFFF",
}


def validate_columns(
    df: pd.DataFrame,
    required_columns: Iterable[str],
    dataframe_name: str = "dataframe",
) -> None:
    """Raise a clear error if required columns are missing."""

    missing = sorted(set(required_columns) - set(df.columns))
    if missing:
        raise ValueError(
            f"{dataframe_name} is missing required columns: {', '.join(missing)}"
        )


def coerce_numeric(
    df: pd.DataFrame,
    columns: Sequence[str],
) -> pd.DataFrame:
    """Return a copy with selected columns converted to numeric values."""

    out = df.copy()
    for column in columns:
        if column in out.columns:
            out[column] = pd.to_numeric(out[column], errors="coerce")
    return out


def prepare_executive_df(df: pd.DataFrame) -> pd.DataFrame:
    """Validate and normalize the executive product-level dataframe."""

    validate_columns(df, EXECUTIVE_REQUIRED_COLUMNS, "executive_df")

    numeric_columns = [
        "priority_score",
        "td_score",
        "commercial_score_v2",
        "qt_pedidos",
        "qt_skus",
        "qt_items_vendidos",
        "receita_liquida",
        "qt_reversas",
        "qt_items_returned",
        "qt_trocas",
        "qt_devolucoes",
        "valor_troca",
        "valor_devolucao",
        "td_rate",
        "td_rate_categoria",
        "delta_vs_categoria",
        "ratio_vs_categoria",
        "share_receita_portfolio",
        "share_unidades_portfolio",
        "share_td_portfolio",
        "top_1_pct",
        "top_2_pct",
        "top_3_pct",
        "pct_top_3_total",
        "principal_cor_pct",
        "principal_tamanho_pct",
        "reversas_ultimos_3m",
        "reversas_3m_anteriores",
        "qt_reversas_fisico",
        "qt_reversas_logistico",
    ]
    out = coerce_numeric(df, numeric_columns)

    sort_cols = ["priority_score", "qt_items_returned", "receita_liquida"]
    available_sort_cols = [c for c in sort_cols if c in out.columns]
    return out.sort_values(available_sort_cols, ascending=False).reset_index(drop=True)


def classify_priority(
    df: pd.DataFrame,
    thresholds: PrioritizationThresholds = PrioritizationThresholds(),
) -> pd.DataFrame:
    """Recompute prioritization labels in Python for QA or sensitivity tests."""

    out = prepare_executive_df(df)

    conditions = [
        (out["qt_items_vendidos"] < thresholds.min_items_vendidos)
        | (out["qt_items_returned"] < thresholds.min_items_returned),
        (out["td_score"] >= thresholds.td_score_prioritize)
        & (out[COMMERCIAL_SCORE_COL] >= thresholds.commercial_score_prioritize),
        (out["td_score"] >= thresholds.td_score_prioritize)
        & (out[COMMERCIAL_SCORE_COL] < thresholds.commercial_score_prioritize),
        (out[COMMERCIAL_SCORE_COL] >= thresholds.commercial_score_alert)
        & (out["td_score"] >= thresholds.td_score_alert_min)
        & (out["td_score"] < thresholds.td_score_prioritize),
        (out["td_score"] < thresholds.low_td_score)
        | (out[COMMERCIAL_SCORE_COL] < thresholds.low_commercial_score),
    ]
    labels = [
        "Sem evidência suficiente",
        "Priorizar melhoria",
        "Monitorar",
        "Alerta em produto relevante",
        "Não priorizar agora",
    ]

    out["sinal_priorizacao_python"] = np.select(conditions, labels, default="Monitorar")
    out["sinal_confere_sql"] = out["sinal_priorizacao_python"].eq(
        out["sinal_priorizacao"]
    )
    return out


def priority_summary(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize portfolio impact by prioritization bucket."""

    out = prepare_executive_df(df)
    summary = (
        out.groupby("sinal_priorizacao", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            unidades_vendidas=("qt_items_vendidos", "sum"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
            td_rate_medio=("td_rate", "mean"),
            priority_score_medio=("priority_score", "mean"),
        )
        .reset_index()
    )
    total_revenue = summary["receita_liquida"].sum()
    total_returned = summary["itens_retornados"].sum()
    summary["share_receita"] = np.where(
        total_revenue > 0, summary["receita_liquida"] / total_revenue, 0
    )
    summary["share_itens_retornados"] = np.where(
        total_returned > 0, summary["itens_retornados"] / total_returned, 0
    )
    return summary.sort_values("itens_retornados", ascending=False).reset_index(drop=True)


def top_offenders(
    df: pd.DataFrame,
    bucket: str | None = "Priorizar melhoria",
    n: int = 20,
    min_items_returned: int = 5,
) -> pd.DataFrame:
    """Return top offender products ranked by priority score and returned items."""

    out = prepare_executive_df(df)
    if bucket is not None:
        out = out[out["sinal_priorizacao"].eq(bucket)]
    out = out[out["qt_items_returned"].fillna(0) >= min_items_returned]
    return out.sort_values(
        ["priority_score", "qt_items_returned", "td_rate"],
        ascending=[False, False, False],
    ).head(n)


def category_benchmark(df: pd.DataFrame) -> pd.DataFrame:
    """Aggregate product-level results by category for benchmarking."""

    out = prepare_executive_df(df)
    category = (
        out.groupby("category", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            unidades_vendidas=("qt_items_vendidos", "sum"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
            produtos_priorizar=(
                "sinal_priorizacao",
                lambda s: int((s == "Priorizar melhoria").sum()),
            ),
            td_rate_medio_produto=("td_rate", "mean"),
            td_rate_mediano_produto=("td_rate", "median"),
        )
        .reset_index()
    )
    category["td_rate_categoria_recalculado"] = np.where(
        category["unidades_vendidas"] > 0,
        category["itens_retornados"] / category["unidades_vendidas"],
        0,
    )
    return category.sort_values(
        ["produtos_priorizar", "itens_retornados"], ascending=False
    ).reset_index(drop=True)


def problema_tipo_split(df: pd.DataFrame) -> pd.DataFrame:
    """Breakdown T&D reversas by problem type (Físico / Logístico / Desistência).

    Uses the pre-aggregated qt_reversas_fisico and qt_reversas_logistico columns
    from the executive table.  The remainder (Outros + Desistência) is inferred
    as total reversas minus the two explicit buckets.
    """

    out = prepare_executive_df(df)
    total_reversas = float(out["qt_reversas"].fillna(0).sum())
    qt_fisico = (
        float(out["qt_reversas_fisico"].fillna(0).sum())
        if "qt_reversas_fisico" in out.columns
        else 0.0
    )
    qt_logistico = (
        float(out["qt_reversas_logistico"].fillna(0).sum())
        if "qt_reversas_logistico" in out.columns
        else 0.0
    )
    qt_outros = max(0.0, total_reversas - qt_fisico - qt_logistico)

    rows = [
        {"tipo_problema": "Físico", "qt_reversas": qt_fisico},
        {"tipo_problema": "Logístico", "qt_reversas": qt_logistico},
        {"tipo_problema": "Outros / Desistência", "qt_reversas": qt_outros},
    ]
    result = pd.DataFrame(rows)
    result["pct_total"] = result["qt_reversas"].div(
        total_reversas if total_reversas > 0 else 1.0
    )
    return result.sort_values("qt_reversas", ascending=False).reset_index(drop=True)


def tag_summary_from_executive(df: pd.DataFrame) -> pd.DataFrame:
    """Summarize top problem tags already pivoted in the executive table."""

    out = prepare_executive_df(df)
    frames = []
    for rank in (1, 2, 3):
        problem_col = f"top_{rank}_problema"
        pct_col = f"top_{rank}_pct"
        if problem_col in out.columns:
            tmp = out[["product_name", "sinal_priorizacao", problem_col, pct_col]].copy()
            tmp = tmp.rename(columns={problem_col: "problema_tag", pct_col: "pct_no_produto"})
            tmp["rank"] = rank
            frames.append(tmp)
    if not frames:
        return pd.DataFrame(columns=["problema_tag", "produtos", "pct_medio_no_produto"])

    long_df = pd.concat(frames, ignore_index=True)
    long_df = long_df.dropna(subset=["problema_tag"])
    return (
        long_df.groupby("problema_tag", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            produtos_priorizar=(
                "sinal_priorizacao",
                lambda s: int((s == "Priorizar melhoria").sum()),
            ),
            pct_medio_no_produto=("pct_no_produto", "mean"),
        )
        .reset_index()
        .sort_values(["produtos_priorizar", "produtos"], ascending=False)
        .reset_index(drop=True)
    )


def validate_no_tag_duplication(
    analytical_df: pd.DataFrame,
    executive_df: pd.DataFrame,
    tolerance: float = 0.01,
) -> Mapping[str, float | bool]:
    """Check if analytical tag-level data would inflate returned item counts.

    This compares the executive returned-items total against a deduplicated
    analytical total by order_name + sku. It is a QA guardrail: the executive
    T&D numerator should not be built by summing tag rows.
    """

    validate_columns(analytical_df, ANALYTICAL_REQUIRED_COLUMNS, "analytical_df")
    executive = prepare_executive_df(executive_df)
    analytical = coerce_numeric(analytical_df, ["qt_items_returned"])

    dedup = analytical.drop_duplicates(["order_name", "sku"])
    analytical_total = float(dedup["qt_items_returned"].fillna(0).sum())
    executive_total = float(executive["qt_items_returned"].fillna(0).sum())
    diff = executive_total - analytical_total
    pct_diff = diff / executive_total if executive_total else 0.0

    return {
        "executive_total_returned": executive_total,
        "dedup_analytical_total_returned": analytical_total,
        "absolute_diff": diff,
        "pct_diff_vs_executive": pct_diff,
        "passes": abs(pct_diff) <= tolerance,
    }


def build_llm_context_rows(
    df: pd.DataFrame,
    only_buckets: Sequence[str] = (
        "Priorizar melhoria",
        "Alerta em produto relevante",
        "Monitorar",
    ),
    max_rows: int = 50,
) -> list[dict]:
    """Build compact product dictionaries to feed into an LLM summarizer."""

    out = prepare_executive_df(df)
    out = out[out["sinal_priorizacao"].isin(only_buckets)]
    out = out.sort_values("priority_score", ascending=False).head(max_rows)

    fields = [
        "product_name",
        "category",
        "gender",
        "portfolio_cluster",
        "sinal_priorizacao",
        "td_rate",
        "td_rate_categoria",
        "delta_vs_categoria",
        "qt_items_vendidos",
        "receita_liquida",
        "qt_items_returned",
        "qt_trocas",
        "qt_devolucoes",
        "qt_reversas_fisico",
        "qt_reversas_logistico",
        "commercial_score_v2",
        "td_score",
        "priority_score",
        "top_1_problema",
        "top_1_pct",
        "top_2_problema",
        "top_2_pct",
        "top_3_problema",
        "top_3_pct",
        "pct_top_3_total",
        "principal_cor_afetada",
        "principal_cor_pct",
        "principal_tamanho_afetado",
        "principal_tamanho_pct",
        "tendencia_reversas",
        "comentarios_amostra",
        "resumo_pre_llm",
    ]
    available = [field for field in fields if field in out.columns]
    rows = out[available].replace({np.nan: None}).to_dict(orient="records")
    return rows


def create_product_summary_prompt(product_row: Mapping[str, object]) -> str:
    """Create a strict prompt for one product's qualitative summary."""

    return f"""
Você é uma analista de Produto/Físico da Insider Store.
Escreva um resumo curto, em um único parágrafo, sobre os problemas de T&D do produto abaixo.

Regras:
- Não invente causa.
- Só mencione cor ou tamanho se houver concentração informada.
- Só mencione tendência se houver evidência informada.
- Separe problema físico de problema logístico.
- Seja executivo e acionável.
- Máximo de 80 palavras.

Dados do produto:
{product_row}
""".strip()


def get_tipo_de_acao(top_1_tag: str | None) -> str:
    """Map the top-1 return tag to a recommended action type for the PF team.

    Returns one of: Modelagem, Grade, Tecido, PDP/Comunicação,
    Investigar tags negativas, Logístico — fora do escopo PF,
    or Investigação adicional.
    """
    if top_1_tag is None:
        return "Investigação adicional"
    if top_1_tag in POSITIVE_TAGS:
        return "Investigar tags negativas"
    if top_1_tag in LOGISTICA_TAGS:
        return "Logístico — fora do escopo PF"
    return TAG_TO_ACTION.get(top_1_tag, "Investigação adicional")


def build_tweet_heuristic(row: Mapping[str, object]) -> str:
    """Generate a rule-based ≤80-word analyst tweet for one product.

    Input is a product dict as produced by build_llm_context_rows().
    Covers: top-3 problems, color/size concentration (if ≥50%), T&D rate
    vs category, top-3 leverage, trend (if not stable), and suggested action.
    When Top 1 tag is positive, emits a disclaimer to investigate negative tags.
    """

    def _f(v: object) -> float | None:
        try:
            return float(v) if v is not None else None
        except (TypeError, ValueError):
            return None

    top_1_tag = row.get("top_1_problema")
    tipo_de_acao = get_tipo_de_acao(top_1_tag if isinstance(top_1_tag, str) else None)
    is_positive = isinstance(top_1_tag, str) and top_1_tag in POSITIVE_TAGS

    parts: list[str] = []

    # Positive tags disclaimer — replaces the offenders section
    if is_positive:
        parts.append(
            "⚠️ Tags predominantes são positivas. "
            "Recomenda-se investigar as tags negativas para diagnóstico completo."
        )
    else:
        # Top 3 problems
        tags: list[str] = []
        for rank in (1, 2, 3):
            tag = row.get(f"top_{rank}_problema")
            pct = _f(row.get(f"top_{rank}_pct"))
            if tag:
                suffix = f" ({pct:.0%})" if pct is not None else ""
                tags.append(f"{tag}{suffix}")
        if tags:
            parts.append(f"Ofensores: {', '.join(tags)}.")

    # Color concentration (≥50%)
    cor = row.get("principal_cor_afetada")
    cor_pct = _f(row.get("principal_cor_pct"))
    if cor and cor_pct is not None and cor_pct >= 0.50:
        parts.append(f"Cor crítica: {cor} ({cor_pct:.0%} das reversas).")

    # Size concentration (≥50%)
    tam = row.get("principal_tamanho_afetado")
    tam_pct = _f(row.get("principal_tamanho_pct"))
    if tam and tam_pct is not None and tam_pct >= 0.50:
        parts.append(f"Tamanho crítico: {tam} ({tam_pct:.0%} das reversas).")

    # T&D rate vs category
    td_rate = _f(row.get("td_rate"))
    td_cat = _f(row.get("td_rate_categoria"))
    if td_rate is not None and td_cat is not None:
        delta = td_rate - td_cat
        direction = "acima" if delta >= 0 else "abaixo"
        parts.append(
            f"T&D: {td_rate:.1%} produto vs {td_cat:.1%} categoria "
            f"({abs(delta):.1%}pp {direction})."
        )

    # Top-3 leverage
    pct_top3 = _f(row.get("pct_top_3_total"))
    if pct_top3 is not None:
        parts.append(f"Top 3 concentra {pct_top3:.0%} das reversas tagueadas.")

    # Trend (skip stable / insufficient)
    trend = row.get("tendencia_reversas")
    if trend and trend not in ("Estável", "Sem volume para tendência"):
        parts.append(f"Tendência recente: {trend}.")

    # Suggested action type
    parts.append(f"Ação sugerida: {tipo_de_acao}.")

    result = " ".join(parts)
    words = result.split()
    if len(words) > 80:
        result = " ".join(words[:80]) + "…"
    return result or "Dados insuficientes para síntese."


def build_scorecard_product_view(
    executive_df: pd.DataFrame,
    scorecard_df: pd.DataFrame,
    n: int = 50,
    only_buckets: Sequence[str] | None = None,
) -> pd.DataFrame:
    """Merge the executive T&D table with the portfolio scorecard pillars.

    Parameters
    ----------
    executive_df:
        Output of prepare_executive_df(). Must contain product_name, category,
        gender, td_rate, td_rate_categoria, top_N_problema/pct columns.
    scorecard_df:
        Must contain: product_name, cluster, score_tracao_comercial,
        score_unit_economics, score_satisfacao_marca,
        td_produto_pct, td_categoria_pct.
    n:
        Max rows returned, sorted by priority_score DESC.
    only_buckets:
        If provided, filter to these sinal_priorizacao values before merging.
    """

    exec_out = prepare_executive_df(executive_df)
    if only_buckets is not None:
        exec_out = exec_out[exec_out["sinal_priorizacao"].isin(only_buckets)]

    merged = exec_out.merge(
        scorecard_df[
            [
                "product_name",
                "cluster",
                "score_tracao_comercial",
                "score_unit_economics",
                "score_satisfacao_marca",
                "td_produto_pct",
                "td_categoria_pct",
            ]
        ],
        on="product_name",
        how="left",
    )

    # Build "Top 3 motivos" string from pivoted columns already in executive_df
    def _top3_str(row: pd.Series) -> str:
        parts = []
        for rank in (1, 2, 3):
            tag = row.get(f"top_{rank}_problema")
            pct = row.get(f"top_{rank}_pct")
            if pd.isna(tag) or tag is None:
                continue
            pct_str = f"{pct:.0%}" if pd.notna(pct) else ""
            parts.append(f"{rank}. {tag} ({pct_str})")
        return "<br>".join(parts) if parts else "—"

    merged["top_3_motivos_td"] = merged.apply(_top3_str, axis=1)
    merged["tweet_analitico"] = merged.apply(build_tweet_heuristic, axis=1)

    cols = [
        "product_name",
        "category",
        "gender",
        "cluster",
        "sinal_priorizacao",
        "td_score",
        "score_tracao_comercial",
        "score_unit_economics",
        "score_satisfacao_marca",
        "td_categoria_pct",
        "td_produto_pct",
        "td_rate",
        "td_rate_categoria",
        "tendencia_reversas",
        "top_3_motivos_td",
        "tweet_analitico",
        "priority_score",
        "qt_items_returned",
        "receita_liquida",
    ]
    available = [c for c in cols if c in merged.columns]
    return (
        merged[available]
        .sort_values("priority_score", ascending=False)
        .head(n)
        .reset_index(drop=True)
    )


def build_mensagem_consolidada(row: Mapping[str, object]) -> str:
    """Build the consolidated 3-line message for the tweets_analiticos tab.

    Format:
        {product} - Sinal: {bucket} ({descrição})
        Top 3 problemas: {tag1}, {tag2} e {tag3}
        Tweet: {resumo analítico}
    """
    product = str(row.get("product_name") or "—")
    sinal = str(row.get("sinal_priorizacao") or "—")
    sinal_desc = SINAL_DESCRICAO.get(sinal, sinal.lower())

    tags: list[str] = []
    for rank in (1, 2, 3):
        tag = row.get(f"top_{rank}_problema")
        if tag and str(tag) not in ("nan", "None", ""):
            tags.append(str(tag))

    if len(tags) >= 3:
        tags_str = f"{tags[0]}, {tags[1]} e {tags[2]}"
    elif len(tags) == 2:
        tags_str = f"{tags[0]} e {tags[1]}"
    elif len(tags) == 1:
        tags_str = tags[0]
    else:
        tags_str = "sem tags disponíveis"

    tweet = str(row.get("tweet_analitico") or row.get("tweet_analista") or "—")

    return (
        f"{product} - Sinal: {sinal} ({sinal_desc})\n"
        f"Top 3 problemas: {tags_str}\n"
        f"Tweet: {tweet}"
    )


def _apply_workbook_formatting(wb: object) -> None:
    """Apply professional formatting to all sheets: header colors, tab colors,
    row coloring by sinal_priorizacao, freeze pane, and column widths."""
    from openpyxl.styles import Font, PatternFill, Alignment
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill("solid", fgColor="1F497D")
    header_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
    body_font = Font(name="Arial", size=10)

    for ws in wb.worksheets:  # type: ignore[attr-defined]
        name = ws.title

        # Tab color
        color = _TAB_COLORS.get(name)
        if color:
            ws.sheet_properties.tabColor = color

        # Freeze top row
        ws.freeze_panes = "A2"
        ws.row_dimensions[1].height = 28

        # Identify key columns
        sinal_col_idx: int | None = None
        mensagem_col_idx: int | None = None
        for cell in ws[1]:
            if cell.value == "sinal_priorizacao":
                sinal_col_idx = cell.column
            if cell.value == "mensagem_consolidada":
                mensagem_col_idx = cell.column

        # Header row
        for cell in ws[1]:
            cell.font = header_font
            cell.fill = header_fill
            cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)

        # Body: font + row coloring + alignment
        for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
            sinal_val = row[sinal_col_idx - 1].value if sinal_col_idx else None
            row_fill = None
            if sinal_val and str(sinal_val) in _SINAL_ROW_COLORS:
                row_fill = PatternFill("solid", fgColor=_SINAL_ROW_COLORS[str(sinal_val)])
            for cell in row:
                cell.font = body_font
                if row_fill:
                    cell.fill = row_fill
                if mensagem_col_idx and cell.column == mensagem_col_idx:
                    cell.alignment = Alignment(wrap_text=True, vertical="top")
                else:
                    cell.alignment = Alignment(vertical="center")
            if name == "tweets_analiticos":
                ws.row_dimensions[row_idx].height = 90

        # Column widths
        wide_cols = {"mensagem_consolidada", "tweet_analitico", "top_3_motivos_td", "comentarios_amostra"}
        medium_cols = {"product_name", "resumo_pre_llm"}
        for col in ws.columns:
            hdr = str(col[0].value or "")
            letter = get_column_letter(col[0].column)
            if hdr == "mensagem_consolidada":
                ws.column_dimensions[letter].width = 90
            elif hdr in wide_cols:
                ws.column_dimensions[letter].width = 60
            elif hdr in medium_cols:
                ws.column_dimensions[letter].width = 35
            else:
                max_len = max((len(str(c.value or "")) for c in col), default=8)
                ws.column_dimensions[letter].width = min(max(max_len + 2, 10), 32)


def build_tweets_tab(
    executive_df: pd.DataFrame,
    buckets: Sequence[str] = ("Priorizar melhoria", "Alerta em produto relevante"),
) -> pd.DataFrame:
    """Build the tweets_analiticos tab — one actionable tweet per product.

    Filtered to action buckets only (default: Priorizar melhoria + Alerta em
    produto relevante). Sorted by bucket priority then priority_score DESC.

    Primary column is ``mensagem_consolidada``: a 3-line block with product
    name + sinal description, Top 3 problems, and the analytical tweet.
    """
    out = prepare_executive_df(executive_df)
    out = out[out["sinal_priorizacao"].isin(buckets)].copy()

    out["tipo_de_acao"] = out["top_1_problema"].apply(
        lambda t: get_tipo_de_acao(t if isinstance(t, str) else None)
    )
    out["tweet_analitico"] = out.apply(build_tweet_heuristic, axis=1)
    out["mensagem_consolidada"] = out.apply(build_mensagem_consolidada, axis=1)

    if "portfolio_cluster" in out.columns:
        out = out.rename(columns={"portfolio_cluster": "cluster"})

    bucket_order = {b: i for i, b in enumerate(buckets)}
    out["_bucket_order"] = out["sinal_priorizacao"].map(bucket_order).fillna(99)
    out = (
        out.sort_values(["_bucket_order", "priority_score"], ascending=[True, False])
        .drop(columns=["_bucket_order"])
        .reset_index(drop=True)
    )

    cols = [
        "product_name",
        "sinal_priorizacao",
        "mensagem_consolidada",
        "tipo_de_acao",
        "category",
        "gender",
        "cluster",
        "td_rate",
        "td_rate_categoria",
        "tendencia_reversas",
        "priority_score",
    ]
    available = [c for c in cols if c in out.columns]
    return out[available]


def export_priority_workbook(
    executive_df: pd.DataFrame,
    output_path: str,
    analytical_df: pd.DataFrame | None = None,
    scorecard_view_df: pd.DataFrame | None = None,
    tweets_df: pd.DataFrame | None = None,
) -> None:
    """Export core analysis tabs to an Excel workbook.

    Tab order: farol_executivo, resumo_farol, benchmark_categoria, resumo_tags,
    priorizar_melhoria, tweets_analiticos (if provided), relatorio_pf (if provided),
    amostra_analitica (if provided).
    """

    executive = prepare_executive_df(executive_df)
    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        executive.to_excel(writer, sheet_name="farol_executivo", index=False)
        priority_summary(executive).to_excel(writer, sheet_name="resumo_farol", index=False)
        category_benchmark(executive).to_excel(writer, sheet_name="benchmark_categoria", index=False)
        tag_summary_from_executive(executive).to_excel(writer, sheet_name="resumo_tags", index=False)
        top_offenders(executive, bucket="Priorizar melhoria", n=50).to_excel(
            writer, sheet_name="priorizar_melhoria", index=False
        )
        if tweets_df is not None:
            tweets_df.to_excel(writer, sheet_name="tweets_analiticos", index=False)
        if scorecard_view_df is not None:
            scorecard_view_df.to_excel(writer, sheet_name="relatorio_pf", index=False)
        if analytical_df is not None:
            analytical_df.head(100000).to_excel(writer, sheet_name="amostra_analitica", index=False)
        _apply_workbook_formatting(writer.book)

## 6. Preparação e normalização dos DataFrames

Aplica validações, coerção de tipos e ordenação padrão sobre os dados brutos extraídos.

**Entrada:** `executive_df`, `df_scores_raw`
**Saída:** `executive_prepared_df`, `scorecard_df`
**Quando mexer:** ao adicionar novo campo de normalização ou alterar join do scorecard.


In [90]:
# ── Preparação do DataFrame executivo ────────────────────────────────────────
# Valida colunas obrigatórias, coerce numéricos e ordena por priority_score DESC.
executive_prepared_df = prepare_executive_df(executive_df)

print(f"✅ executive_prepared_df: {executive_prepared_df.shape}")
display(executive_prepared_df.head(3))


✅ executive_prepared_df: (117, 44)


,product_name,category,gender,portfolio_cluster,sinal_priorizacao,priority_score,td_score,commercial_score_v2,qt_pedidos,qt_skus,qt_items_vendidos,receita_liquida,qt_reversas,qt_items_returned,qt_trocas,qt_devolucoes,qt_reversas_fisico,qt_reversas_logistico,valor_troca,valor_devolucao,td_rate,td_rate_categoria,delta_vs_categoria,ratio_vs_categoria,share_receita_portfolio,share_unidades_portfolio,share_td_portfolio,top_1_problema,top_1_pct,top_2_problema,top_2_pct,top_3_problema,top_3_pct,pct_top_3_total,principal_cor_afetada,principal_cor_pct,principal_tamanho_afetado,principal_tamanho_pct,reversas_ultimos_3m,reversas_3m_anteriores,tendencia_reversas,comentarios_amostra,resumo_pre_llm,portfolio_cluster_payload
0,Calça FutureForm Feminino,Calça,female,Hero,Priorizar melhoria,0.9326,0.9299,0.9352,21969,15,"22,055.0905","7,029,505.0617",3213,"3,218.0000",2701,489,2941,17,"1,168,584.6600","302,608.3100",0.1459,0.1055,0.0404,1.3834,0.0217,0.0083,0.0247,caimento_ruim,0.1148,modelagem_ruim,0.0918,tamanho_pequeno,0.0514,0.2580,Preto,0.5353,M,0.3617,747,937,Estável,a cor é bem diferente do site e não recebi a c...,"Top problemas: caimento_ruim, modelagem_ruim, ...","{""cluster"":""Hero"",""product_name"":""Calça Future..."
1,Camisa FutureForm Masculino,Camisa Social,male,Core,Priorizar melhoria,0.9143,0.8991,0.9294,11485,20,"12,916.0351","4,250,771.3057",1973,"1,999.0000",1746,216,1906,7,"895,081.5800","200,701.4600",0.1548,0.1277,0.0271,1.2119,0.0131,0.0048,0.0153,tamanho_grande,0.1221,caimento_ruim,0.0299,modelagem_ruim,0.0264,0.1784,Ocean Blue,0.3137,G,0.3284,734,442,Aumentou nos últimos 3 meses,Acreditava que o material das roupas fossem me...,"Top problemas: tamanho_grande, caimento_ruim, ...","{""cluster"":""Core"",""product_name"":""Camisa Futur..."
2,Shorts Kyoto Feminino,Shorts,female,Core,Priorizar melhoria,0.8920,0.8538,0.9302,12267,15,"12,679.5089","2,566,026.1668",2113,"2,118.0000",1761,330,1999,7,"608,980.5500","182,775.3900",0.1670,0.1670,0.0000,1.0000,0.0079,0.0047,0.0163,caimento_ruim,0.0842,tamanho_grande,0.0800,tamanho_pequeno,0.0743,0.2385,Preto,0.3928,M,0.3743,620,612,Estável,Não veste bem || comprei 3 shorts iguais taman...,"Top problemas: caimento_ruim, tamanho_grande, ...","{""cluster"":""Core"",""product_name"":""Shorts Kyoto..."


In [91]:
# ── Construção do scorecard_df ────────────────────────────────────────────────
# Adiciona td_produto_pct, td_categoria_pct (de executive_prepared_df) e
# cluster (de portfolio_cluster) ao scorecard de pilares.
# Nota: eval_produto_portfolio não tem cluster — ele vem da CTE portfolio_clustering via executive_df.

_td_ref = (
    executive_prepared_df[["product_name", "td_rate", "td_rate_categoria"]]
    .drop_duplicates("product_name")
    .rename(columns={"td_rate": "td_produto_pct", "td_rate_categoria": "td_categoria_pct"})
)

_cluster_ref = (
    executive_prepared_df[["product_name", "portfolio_cluster"]]
    .drop_duplicates("product_name")
    .rename(columns={"portfolio_cluster": "cluster"})
)

scorecard_df = (
    df_scores_raw
    .merge(_td_ref, on="product_name", how="left")
    .merge(_cluster_ref, on="product_name", how="left")
)

_coverage = scorecard_df["score_tracao_comercial"].notna().sum()
print(f"✅ scorecard_df: {scorecard_df.shape}")
print(f"   Scores populados: {_coverage}/{len(scorecard_df)} produto(s)")
print(f"   Produtos sem scorecard (ficarão com —): {len(scorecard_df) - _coverage}")


✅ scorecard_df: (126, 7)
   Scores populados: 126/126 produto(s)
   Produtos sem scorecard (ficarão com —): 0


## 7. Classificação e score de priorização

Recomputa `sinal_priorizacao` em Python (espelho do SQL) para QA e adiciona `sinal_kill_keep`.

**Entrada:** `executive_prepared_df`
**Saída:** `classified_df` (com `sinal_priorizacao_python` e `sinal_confere_sql`),
  `executive_prepared_df` com coluna `sinal_kill_keep` adicionada
**Quando mexer:** ao ajustar thresholds (atenção: deve estar em sincronia com o SQL).


In [92]:
# ── QA: Python vs SQL ────────────────────────────────────────────────────────
# classify_priority reimplementa a lógica SQL em Python para validação cruzada.
classified_df = classify_priority(executive_prepared_df)

match_rate = classified_df["sinal_confere_sql"].mean()
n_divergencias = (~classified_df["sinal_confere_sql"]).sum()

print(f"Match Python vs SQL: {match_rate:.2%}  ({n_divergencias} divergências)")

if n_divergencias > 0:
    divergencias = classified_df.loc[
        ~classified_df["sinal_confere_sql"],
        [
            "product_name",
            "sinal_priorizacao",
            "sinal_priorizacao_python",
            "priority_score",
            "td_score",
            "commercial_score_v2",
            "qt_items_vendidos",
            "qt_items_returned",
        ],
    ]
    print("\n⚠️  Divergências encontradas:")
    display(divergencias.head(50))


Match Python vs SQL: 100.00%  (0 divergências)


In [93]:
# ── sinal_kill_keep ──────────────────────────────────────────────────────────
# Regra QA: Alta T&D + Alta Tração → Priorizar | Alta T&D + Baixa Tração → Avaliar saída.
# Fonte: lógica original de TD_Priorizacao_Melhorias_v20260611.ipynb (célula b402d21a).
# ⚠️ Não alterar thresholds (0.70 / 0.60 / 30 / 5) sem revisão metodológica.
executive_prepared_df["sinal_kill_keep"] = np.select(
    [
        (executive_prepared_df["qt_items_vendidos"] < 30)
        | (executive_prepared_df["qt_items_returned"] < 5),
        (executive_prepared_df["td_score"] >= 0.70)
        & (executive_prepared_df["commercial_score_v2"] >= 0.60),
        (executive_prepared_df["td_score"] >= 0.70)
        & (executive_prepared_df["commercial_score_v2"] < 0.60),
    ],
    [
        "Sem evidência suficiente",
        "Priorizar melhoria",
        "Não Priorizar (Avaliar Descontinuação/Reformulação)",
    ],
    default="Não priorizar agora",
)

print("✅ sinal_kill_keep adicionado ao executive_prepared_df")
print(executive_prepared_df["sinal_kill_keep"].value_counts().to_string())


✅ sinal_kill_keep adicionado ao executive_prepared_df
sinal_kill_keep
Não priorizar agora                                    91
Priorizar melhoria                                     14
Não Priorizar (Avaliar Descontinuação/Reformulação)     9
Sem evidência suficiente                                3


## 8. Diagnósticos e tabelas analíticas

Gera as tabelas intermediárias que alimentam os outputs finais.

**Entrada:** `executive_prepared_df`, `scorecard_df`
**Saída:** `summary_df`, `priorizar_melhoria_raw_df`, `alerta_raw_df`,
  `benchmark_categoria_raw_df`, `top_problemas_raw_df`, `df_tipo`, `df_trend`,
  `scorecard_view_df`, `tweets_df`
**Quando mexer:** ao adicionar nova tabela de diagnóstico ou ajustar parâmetros de ranking.


In [94]:
# ── §1. Resumo do portfólio por farol ────────────────────────────────────────
summary_df = priority_summary(executive_prepared_df)
print("summary_df:", summary_df.shape)
display(summary_df)


summary_df: (5, 9)


,sinal_priorizacao,produtos,unidades_vendidas,receita_liquida,itens_retornados,td_rate_medio,priority_score_medio,share_receita,share_itens_retornados
0,Monitorar,34,"1,153,396.2113","150,328,470.1876","68,927.0000",0.0917,0.6025,0.4638,0.5292
1,Não priorizar agora,56,"1,145,385.6872","87,719,715.0973","25,805.0000",0.0358,0.4413,0.2706,0.1981
2,Priorizar melhoria,14,"220,990.8250","47,556,535.7603","25,433.0000",0.1463,0.7992,0.1467,0.1953
3,Alerta em produto relevante,10,"152,400.8681","38,421,884.5796","10,081.0000",0.0840,0.7104,0.1185,0.0774
4,Sem evidência suficiente,3,354.6287,"101,262.4690",8.0000,0.0161,0.3604,0.0003,0.0001


In [95]:
# ── §2. Top produtos — Priorizar melhoria ────────────────────────────────────
priorizar_melhoria_raw_df = top_offenders(
    executive_prepared_df,
    bucket="Priorizar melhoria",
    n=50,
)
print("priorizar_melhoria_raw_df:", priorizar_melhoria_raw_df.shape)
display(priorizar_melhoria_raw_df[[
    "product_name", "category", "sinal_priorizacao",
    "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "qt_items_returned",
]].head(10))


priorizar_melhoria_raw_df: (14, 45)


,product_name,category,sinal_priorizacao,priority_score,td_score,commercial_score_v2,td_rate,td_rate_categoria,qt_items_returned
0,Calça FutureForm Feminino,Calça,Priorizar melhoria,0.9326,0.9299,0.9352,0.1459,0.1055,"3,218.0000"
1,Camisa FutureForm Masculino,Camisa Social,Priorizar melhoria,0.9143,0.8991,0.9294,0.1548,0.1277,"1,999.0000"
2,Shorts Kyoto Feminino,Shorts,Priorizar melhoria,0.8920,0.8538,0.9302,0.1670,0.1670,"2,118.0000"
3,Camisa FutureForm Feminino,Camisa Social,Priorizar melhoria,0.8601,0.7444,0.9758,0.1151,0.1277,"2,381.0000"
4,Tube Dress Feminino,Vestido,Priorizar melhoria,0.8518,0.8650,0.8386,0.1414,0.1341,"2,474.0000"
5,Bermuda Kyoto Feminino,Bermuda,Priorizar melhoria,0.8393,0.8744,0.8042,0.1508,0.1129,"1,212.0000"
6,Saia Mini Kyoto Feminino,Saia,Priorizar melhoria,0.8150,0.9308,0.6993,0.1739,0.0973,"1,608.0000"
7,Shorts Esportivo Serotonin Feminino,Shorts Esportivo,Priorizar melhoria,0.7932,0.8692,0.7171,0.1731,0.0873,658.0000
10,Saia Midi Kyoto Feminino,Saia,Priorizar melhoria,0.7424,0.8359,0.6489,0.1315,0.0973,"1,166.0000"
11,Core T-shirt Masculino,T-shirt,Priorizar melhoria,0.7407,0.7769,0.7046,0.0761,0.0497,"7,268.0000"


In [96]:
# ── §3. Alerta em produto relevante ─────────────────────────────────────────
alerta_raw_df = top_offenders(
    executive_prepared_df,
    bucket="Alerta em produto relevante",
    n=20,
)
print("alerta_raw_df:", alerta_raw_df.shape)
display(alerta_raw_df[[
    "product_name", "category", "td_score", "commercial_score_v2",
    "td_rate", "receita_liquida",
]].head(10))


alerta_raw_df: (10, 45)


,product_name,category,td_score,commercial_score_v2,td_rate,receita_liquida
8,Camiseta Polo Core Masculino,T-shirt,0.5932,0.9346,0.0522,"7,768,858.4249"
9,Calça FutureForm Masculino,Calça,0.5983,0.9065,0.0744,"10,746,508.6334"
15,NEXTECH T-shirt Masculino,T-shirt,0.6265,0.8239,0.0677,"1,717,213.0950"
16,Maxi Saia NYIN Feminino,Saia,0.5564,0.8874,0.0674,"10,161,522.1399"
17,Calça Director Masculino,Calça,0.6103,0.8138,0.1090,"466,968.4075"
18,Macacão Fitness InSculpt Feminino,Macacão,0.5538,0.8620,0.1055,"382,202.3547"
19,Camiseta Henley Core Masculino,T-shirt,0.6128,0.7967,0.0585,"4,697,779.6150"
23,NEXTECH T-shirt Feminino,T-shirt,0.6538,0.7048,0.0855,"461,415.8547"
24,Vestido Chemise Sem Mangas FutureForm Feminino,Vestido,0.5675,0.7904,0.1056,"1,078,712.0119"
31,Vestido Curto Gola Canoa FutureForm Feminino,Vestido,0.6034,0.7122,0.1137,"940,704.0426"


In [97]:
# ── §4. Benchmark por categoria ──────────────────────────────────────────────
benchmark_categoria_raw_df = category_benchmark(executive_prepared_df)
print("benchmark_categoria_raw_df:", benchmark_categoria_raw_df.shape)
display(benchmark_categoria_raw_df)


benchmark_categoria_raw_df: (25, 9)


,category,produtos,unidades_vendidas,receita_liquida,itens_retornados,produtos_priorizar,td_rate_medio_produto,td_rate_mediano_produto,td_rate_categoria_recalculado
0,T-shirt,31,"1,109,486.8924","137,293,347.4121","55,139.0000",2,0.0576,0.0554,0.0497
1,Saia,5,"63,936.3166","18,218,702.5550","6,223.0000",2,0.1079,0.0951,0.0973
2,Vestido,7,"36,727.5530","10,351,389.7817","4,924.0000",2,0.1286,0.1161,0.1341
3,Camisa Social,3,"35,337.1357","11,900,552.4953","4,513.0000",2,0.1154,0.1151,0.1277
4,Calça,5,"76,678.2276","24,413,563.9013","8,087.0000",1,0.1064,0.1090,0.1055
5,Bermuda,2,"22,113.3651","5,281,557.5785","2,496.0000",1,0.1210,0.1210,0.1129
6,Shorts,1,"12,679.5089","2,566,026.1668","2,118.0000",1,0.1670,0.1670,0.1670
7,Casaco,3,"44,345.9371","16,172,003.1433","1,862.0000",1,0.0977,0.0555,0.0420
8,Shorts Esportivo,3,"12,813.2408","2,456,632.0336","1,119.0000",1,0.0924,0.0533,0.0873
9,Vestido Esportivo,1,"2,932.2993","452,415.3098",545.0000,1,0.1859,0.1859,0.1859


In [98]:
# ── §5. Top problemas por tag ────────────────────────────────────────────────
top_problemas_raw_df = tag_summary_from_executive(executive_prepared_df)
print("top_problemas_raw_df:", top_problemas_raw_df.shape)
display(top_problemas_raw_df.head(20))


top_problemas_raw_df: (22, 4)


,problema_tag,produtos,produtos_priorizar,pct_medio_no_produto
0,caimento_ruim,83,14,0.0778
1,modelagem_ruim,64,9,0.0665
2,tamanho_pequeno,59,9,0.0925
3,tamanho_grande,44,7,0.0846
4,cor_diferente_site,10,1,0.0556
5,provador_virtual_impreciso,2,1,0.1061
6,tecido_grosso,2,1,0.0274
7,defeito_costura,12,0,0.0653
8,tecido_qualidade_ruim,12,0,0.0458
9,logistica_item_errado,11,0,0.0463


In [99]:
# ── §6. Split por tipo de problema ──────────────────────────────────────────
df_tipo = problema_tipo_split(executive_prepared_df)
print("df_tipo:", df_tipo.shape)
display(df_tipo)


df_tipo: (3, 3)


,tipo_problema,qt_reversas,pct_total
0,Físico,"111,111.0000",0.9362
1,Outros / Desistência,"6,886.0000",0.0580
2,Logístico,690.0000,0.0058


In [100]:
# ── §7. Tendência de reversas ────────────────────────────────────────────────
if "tendencia_reversas" in executive_prepared_df.columns:
    df_trend = (
        executive_prepared_df
        .groupby("tendencia_reversas", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            reversas_ultimos_3m=("reversas_ultimos_3m", "sum"),
            reversas_3m_anteriores=("reversas_3m_anteriores", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
        )
        .reset_index()
        .sort_values("itens_retornados", ascending=False)
        .reset_index(drop=True)
    )
    display(df_trend)
else:
    df_trend = None
    print("⚠️  Coluna tendencia_reversas não encontrada.")


,tendencia_reversas,produtos,reversas_ultimos_3m,reversas_3m_anteriores,itens_retornados
0,Caiu nos últimos 3 meses,39,9751,15802,"63,345.0000"
1,Estável,34,10851,11604,"47,468.0000"
2,Aumentou nos últimos 3 meses,28,7927,3706,"19,145.0000"
3,Apareceu nos últimos 3 meses,8,241,0,250.0000
4,Sem volume para tendência,7,10,5,46.0000
5,NaN,1,0,0,0.0000


In [101]:
# ── §8. Concentração de cor e tamanho ────────────────────────────────────────
_top_buckets = ["Priorizar melhoria", "Alerta em produto relevante"]
df_prep_top = executive_prepared_df[
    executive_prepared_df["sinal_priorizacao"].isin(_top_buckets)
].copy()

_has_cor = (
    df_prep_top["principal_cor_pct"].fillna(0).ge(0.50)
    if "principal_cor_pct" in df_prep_top.columns
    else pd.Series(False, index=df_prep_top.index)
)
_has_tam = (
    df_prep_top["principal_tamanho_pct"].fillna(0).ge(0.50)
    if "principal_tamanho_pct" in df_prep_top.columns
    else pd.Series(False, index=df_prep_top.index)
)
df_conc = df_prep_top[_has_cor | _has_tam].copy()
print(f"Produtos com concentração ≥50% em cor ou tamanho: {len(df_conc)}")
if not df_conc.empty:
    cols_conc = [c for c in [
        "product_name", "category", "principal_cor_afetada", "principal_cor_pct",
        "principal_tamanho_afetado", "principal_tamanho_pct",
    ] if c in df_conc.columns]
    display(df_conc[cols_conc].head(20))


Produtos com concentração ≥50% em cor ou tamanho: 13


,product_name,category,principal_cor_afetada,principal_cor_pct,principal_tamanho_afetado,principal_tamanho_pct
0,Calça FutureForm Feminino,Calça,Preto,0.5353,M,0.3617
4,Tube Dress Feminino,Vestido,Preto,1.0000,M,0.3873
6,Saia Mini Kyoto Feminino,Saia,Preto,0.5439,M,0.3907
7,Shorts Esportivo Serotonin Feminino,Shorts Esportivo,Preto,0.5966,M,0.3712
12,Vestido Esportivo Rush Feminino,Vestido Esportivo,Preto,0.7849,M,0.3879
13,The Perfect Top T-shirt Feminino,T-shirt,Preto,0.6960,M,0.4095
16,Maxi Saia NYIN Feminino,Saia,Preto,0.5091,M,0.2954
17,Calça Director Masculino,Calça,Preto,0.5118,G,0.3386
18,Macacão Fitness InSculpt Feminino,Macacão,Preto,0.5604,M,0.3187
22,Casaco de Tricot Future Knit Masculino,Casaco,Preto,0.6818,G,0.5455


In [102]:
# ── §9. Visão produto com scorecard (relatorio_pf) ───────────────────────────
scorecard_view_df = build_scorecard_product_view(
    executive_df=executive_prepared_df,
    scorecard_df=scorecard_df,
    n=50,
    only_buckets=["Priorizar melhoria", "Alerta em produto relevante", "Monitorar"],
)
print(f"scorecard_view_df: {scorecard_view_df.shape}")
display(scorecard_view_df.head(10))


scorecard_view_df: (50, 19)


,product_name,category,gender,cluster,sinal_priorizacao,td_score,score_tracao_comercial,score_unit_economics,score_satisfacao_marca,td_categoria_pct,td_produto_pct,td_rate,td_rate_categoria,tendencia_reversas,top_3_motivos_td,tweet_analitico,priority_score,qt_items_returned,receita_liquida
0,Calça FutureForm Feminino,Calça,female,Hero,Priorizar melhoria,0.9299,78.6000,19.5000,11.1000,0.1055,0.1459,0.1459,0.1055,Estável,1. caimento_ruim (11%)<br>2. modelagem_ruim (9...,"Ofensores: caimento_ruim (11%), modelagem_ruim...",0.9326,"3,218.0000","7,029,505.0617"
1,Camisa FutureForm Masculino,Camisa Social,male,Core,Priorizar melhoria,0.8991,67.1000,13.9000,45.8000,0.1277,0.1548,0.1548,0.1277,Aumentou nos últimos 3 meses,1. tamanho_grande (12%)<br>2. caimento_ruim (3...,"Ofensores: tamanho_grande (12%), caimento_ruim...",0.9143,"1,999.0000","4,250,771.3057"
2,Shorts Kyoto Feminino,Shorts,female,Core,Priorizar melhoria,0.8538,67.3000,57.2000,27.1000,0.1670,0.1670,0.1670,0.1670,Estável,1. caimento_ruim (8%)<br>2. tamanho_grande (8%...,"Ofensores: caimento_ruim (8%), tamanho_grande ...",0.8920,"2,118.0000","2,566,026.1668"
3,Camisa FutureForm Feminino,Camisa Social,female,Core,Priorizar melhoria,0.7444,72.2000,23.4000,44.4000,0.1277,0.1151,0.1151,0.1277,Aumentou nos últimos 3 meses,1. tamanho_grande (11%)<br>2. cor_diferente_si...,"Ofensores: tamanho_grande (11%), cor_diferente...",0.8601,"2,381.0000","7,187,829.3117"
4,Tube Dress Feminino,Vestido,female,Long tail,Priorizar melhoria,0.8650,79.5000,11.6000,21.4000,0.1341,0.1414,0.1414,0.1341,Caiu nos últimos 3 meses,1. tamanho_pequeno (16%)<br>2. caimento_ruim (...,"Ofensores: tamanho_pequeno (16%), caimento_rui...",0.8518,"2,474.0000","4,478,311.6663"
5,Bermuda Kyoto Feminino,Bermuda,female,Core,Priorizar melhoria,0.8744,60.0000,63.0000,35.2000,0.1129,0.1508,0.1508,0.1129,Estável,1. tamanho_grande (11%)<br>2. caimento_ruim (7...,"Ofensores: tamanho_grande (11%), caimento_ruim...",0.8393,"1,212.0000","1,984,823.4516"
6,Saia Mini Kyoto Feminino,Saia,female,Core,Priorizar melhoria,0.9308,56.4000,31.1000,10.2000,0.0973,0.1739,0.1739,0.0973,Aumentou nos últimos 3 meses,1. caimento_ruim (10%)<br>2. modelagem_ruim (7...,"Ofensores: caimento_ruim (10%), modelagem_ruim...",0.8150,"1,608.0000","1,984,304.5856"
7,Shorts Esportivo Serotonin Feminino,Shorts Esportivo,female,Long tail,Priorizar melhoria,0.8692,45.5000,35.1000,19.7000,0.0873,0.1731,0.1731,0.0873,Estável,1. tamanho_pequeno (11%)<br>2. caimento_ruim (...,"Ofensores: tamanho_pequeno (11%), caimento_rui...",0.7932,658.0000,"675,951.5858"
8,Camiseta Polo Core Masculino,T-shirt,male,Hero,Alerta em produto relevante,0.5932,89.0000,69.2000,61.0000,0.0497,0.0522,0.0522,0.0497,Caiu nos últimos 3 meses,1. tamanho_grande (4%)<br>2. tamanho_pequeno (...,"Ofensores: tamanho_grande (4%), tamanho_pequen...",0.7639,"2,094.0000","7,768,858.4249"
9,Calça FutureForm Masculino,Calça,male,Hero,Alerta em produto relevante,0.5983,86.2000,56.1000,50.8000,0.1055,0.0744,0.0744,0.1055,Caiu nos últimos 3 meses,1. tamanho_pequeno (5%)<br>2. caimento_ruim (4...,"Ofensores: tamanho_pequeno (5%), caimento_ruim...",0.7524,"2,366.0000","10,746,508.6334"


In [103]:
# ── §10. Tweets analíticos (heurístico) ──────────────────────────────────────
# Diagnóstico textual determinístico — sem LLM.
tweets_df = build_tweets_tab(executive_df)
print(
    f"tweets_df: {len(tweets_df)} produto(s) | "
    f"{(tweets_df['sinal_priorizacao'] == 'Priorizar melhoria').sum()} Priorizar + "
    f"{(tweets_df['sinal_priorizacao'] == 'Alerta em produto relevante').sum()} Alerta"
)
display(tweets_df.head(5))


tweets_df: 24 produto(s) | 14 Priorizar + 10 Alerta


,product_name,sinal_priorizacao,mensagem_consolidada,tipo_de_acao,category,gender,cluster,td_rate,td_rate_categoria,tendencia_reversas,priority_score
0,Calça FutureForm Feminino,Priorizar melhoria,Calça FutureForm Feminino - Sinal: Priorizar m...,Modelagem,Calça,female,Hero,0.1459,0.1055,Estável,0.9326
1,Camisa FutureForm Masculino,Priorizar melhoria,Camisa FutureForm Masculino - Sinal: Priorizar...,Grade,Camisa Social,male,Core,0.1548,0.1277,Aumentou nos últimos 3 meses,0.9143
2,Shorts Kyoto Feminino,Priorizar melhoria,Shorts Kyoto Feminino - Sinal: Priorizar melho...,Modelagem,Shorts,female,Core,0.1670,0.1670,Estável,0.8920
3,Camisa FutureForm Feminino,Priorizar melhoria,Camisa FutureForm Feminino - Sinal: Priorizar ...,Grade,Camisa Social,female,Core,0.1151,0.1277,Aumentou nos últimos 3 meses,0.8601
4,Tube Dress Feminino,Priorizar melhoria,Tube Dress Feminino - Sinal: Priorizar melhori...,Grade,Vestido,female,Long tail,0.1414,0.1341,Caiu nos últimos 3 meses,0.8518


## 9. QA e validações

Garante que os dados estão confiáveis **antes** de exportar ou atualizar a planilha.

**Entrada:** `executive_prepared_df`, `classified_df`, `analytical_df` (opcional)
**Saída:** confirmação de QA ou exceção detalhada
**Quando mexer:** ao adicionar novo check ou ajustar tolerância de divergência.

> ⚠️ Este bloco deve ser executado antes do Bloco 11. Se falhar, **não prosseguir**.


In [104]:
# ── Helpers de QA ────────────────────────────────────────────────────────────
def assert_not_empty(df: pd.DataFrame, df_name: str) -> None:
    "Levanta ValueError se o DataFrame estiver vazio."
    if df is not None and df.empty:
        raise ValueError(f"❌ QA FALHOU: {df_name} está vazio")

# ── QA 1: DataFrames não vazios ───────────────────────────────────────────────
assert_not_empty(executive_prepared_df, "executive_prepared_df")
assert_not_empty(df_scores_raw, "df_scores_raw")
assert_not_empty(summary_df, "summary_df")
print("✅ QA 1: DataFrames não vazios — OK")

# ── QA 2: Colunas obrigatórias ────────────────────────────────────────────────
validate_columns(executive_prepared_df, EXECUTIVE_REQUIRED_COLUMNS, "executive_prepared_df")
if analytical_df is not None:
    validate_columns(analytical_df, ANALYTICAL_REQUIRED_COLUMNS, "analytical_df")
print("✅ QA 2: Colunas obrigatórias — OK")

# ── QA 3: Sinal Python vs SQL ─────────────────────────────────────────────────
if "sinal_confere_sql" in classified_df.columns:
    match_rate = classified_df["sinal_confere_sql"].mean()
    if match_rate < 1.0:
        n_div = (~classified_df["sinal_confere_sql"]).sum()
        raise ValueError(
            f"❌ QA FALHOU: Divergência Python vs SQL. "
            f"Match rate: {match_rate:.2%} ({n_div} produtos divergentes). "
            f"Verifique se os thresholds do SQL e do PrioritizationThresholds estão sincronizados."
        )
print(f"✅ QA 3: Sinal Python vs SQL — {match_rate:.2%} match")

# ── QA 4: Grain (sem duplicados por produto) ──────────────────────────────────
dup_products = executive_prepared_df["product_name"].duplicated().sum()
if dup_products > 0:
    raise ValueError(
        f"❌ QA FALHOU: {dup_products} produtos duplicados em executive_prepared_df. "
        f"Grain esperado: 1 linha por product_name."
    )
print(f"✅ QA 4: Grain sem duplicados — OK ({len(executive_prepared_df)} produtos únicos)")

# ── QA 5: Ranges de scores [0, 1] ────────────────────────────────────────────
for col in ["priority_score", "td_score", "commercial_score_v2", "td_rate"]:
    if col in executive_prepared_df.columns:
        vals = executive_prepared_df[col].dropna()
        invalid = vals.lt(0).sum() + vals.gt(1).sum()
        if invalid > 0:
            raise ValueError(
                f"❌ QA FALHOU: Coluna '{col}' possui {invalid} valores fora de [0, 1]."
            )
print("✅ QA 5: Ranges de scores — OK")

# ── QA 6: Duplicação de tags (opcional, requer LOAD_ANALITICA) ────────────────
if LOAD_ANALITICA and analytical_df is not None:
    tag_dup_result = validate_no_tag_duplication(
        analytical_df=analytical_df,
        executive_df=executive_prepared_df,
    )
    print(f"\n  tag_duplication_check: {tag_dup_result}")
    if not tag_dup_result["passes"]:
        raise ValueError(
            f"❌ QA FALHOU: Duplicação de tags detectada. "
            f"Diferença: {tag_dup_result['pct_diff_vs_executive']:.2%}"
        )
    print("✅ QA 6: Sem duplicação de tags — OK")
else:
    print("⏭️  QA 6: Pulado (LOAD_ANALITICA=False).")

print("\n✅ QA concluído com sucesso — pipeline pronto para exportação.")


✅ QA 1: DataFrames não vazios — OK
✅ QA 2: Colunas obrigatórias — OK
✅ QA 3: Sinal Python vs SQL — 100.00% match
✅ QA 4: Grain sem duplicados — OK (117 produtos únicos)
✅ QA 5: Ranges de scores — OK

  tag_duplication_check: {'executive_total_returned': 130254.0, 'dedup_analytical_total_returned': 130258.0, 'absolute_diff': -4.0, 'pct_diff_vs_executive': -3.070922965897401e-05, 'passes': True}
✅ QA 6: Sem duplicação de tags — OK

✅ QA concluído com sucesso — pipeline pronto para exportação.


## 10. Construção dos outputs finais

Gera os DataFrames finais no formato de consumo/exportação, com nomes de coluna legíveis.

**Entrada:** `executive_prepared_df`, `priorizar_melhoria_raw_df`, `scorecard_view_df`,
  `benchmark_categoria_raw_df`, `top_problemas_raw_df`, `summary_df`, `tweets_df`
**Saída:** `farol_completo_df`, `priorizar_melhoria_df`, `relatorio_pf_df`,
  `benchmark_categoria_df`, `top_problemas_df`, `resumo_df`
**Quando mexer:** ao adicionar/renomear colunas nos outputs ou ajustar formatação de valores.

> Nomes bonitos de coluna **só entram nesta camada**. Não renomear colunas técnicas antes.


In [105]:
# ── Headers das abas (contrato com a planilha existente) ─────────────────────
# Alterar aqui propaga para QA e escrita segura no Bloco 11.

FAROL_COMPLETO_HEADERS = [
    "Produto", "Categoria", "Gênero", "Cluster", "Sinal Farol",
    "Priority Score", "TD Score", "Comm. Score v2",
    "TD Rate", "TD Categ.", "Δ vs Cat.",
    "Top 1 Problema", "Top 2", "Top 3",
    "Tendência", "Itens Retorn.", "Receita Líq.", "Share T&D",
]

PRIORIZAR_MELHORIA_HEADERS = [
    "Rank", "Produto", "Categoria", "Gênero", "Cluster",
    "Priority Score", "TD Score", "Comm. Score v2",
    "TD Rate", "TD Categ.", "Δ vs Cat.",
    "Top Problema 1", "Top 2", "Top 3",
    "Tendência", "Itens Retorn.", "Receita Líq.",
]

RELATORIO_PF_HEADERS = [
    "Produto", "Categoria", "Gênero", "Cluster", "Sinal Kill/Keep",
    "TD Score", "Score Tração", "Score Unit Econ.", "Score Satisf.",
    "TD Categoria", "TD Produto", "Δ vs Cat.",
    "Tendência", "Top 3 Motivos de T&D", "Diagnóstico (Tweet)",
    "Priority Score", "Itens Retorn.", "Receita Líq.",
]

BENCHMARK_CATEGORIA_HEADERS = [
    "Categoria", "Qtd Produtos", "Unid. Vendidas", "Receita Líq.",
    "Itens Retornados", "Produtos Priorizar",
    "TD Rate Médio", "TD Rate Mediano", "TD Rate Categoria", "Urgência",
]

TOP_PROBLEMAS_HEADERS = [
    "Tag do Problema", "Produtos Afetados", "Produtos 'Priorizar'",
    "% Médio no Produto", "Rank Severidade", "Classificação",
]

# ── Legado (mantidos para compatibilidade com debug/apêndice) ─────────────────
FAROL_EXECUTIVO_COLS = [
    "product_name", "category", "gender", "portfolio_cluster",
    "sinal_priorizacao", "sinal_kill_keep", "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "delta_vs_categoria",
    "qt_items_vendidos", "receita_liquida", "qt_items_returned",
    "top_1_problema", "top_1_pct", "top_2_problema", "top_2_pct",
    "top_3_problema", "top_3_pct", "pct_top_3_total",
    "principal_cor_afetada", "principal_cor_pct",
    "principal_tamanho_afetado", "principal_tamanho_pct",
    "tendencia_reversas", "reversas_ultimos_3m", "reversas_3m_anteriores",
    "share_receita_portfolio", "share_td_portfolio",
]

PRIORIZAR_MELHORIA_COLS = [
    "product_name", "category", "gender", "portfolio_cluster",
    "priority_score", "td_score", "commercial_score_v2",
    "td_rate", "td_rate_categoria", "delta_vs_categoria",
    "top_1_problema", "top_1_pct", "top_2_problema", "top_2_pct",
    "top_3_problema", "top_3_pct",
    "tendencia_reversas", "qt_items_returned", "receita_liquida",
]

RELATORIO_PF_COLS = [
    "product_name", "category", "gender", "cluster",
    "sinal_priorizacao",
    "td_score", "score_tracao_comercial", "score_unit_economics", "score_satisfacao_marca",
    "td_categoria_pct", "td_produto_pct", "td_rate", "td_rate_categoria",
    "tendencia_reversas", "top_3_motivos_td", "tweet_analitico",
    "priority_score", "qt_items_returned", "receita_liquida",
]

BENCHMARK_CATEGORIA_COLS = [
    "category", "produtos", "unidades_vendidas", "receita_liquida",
    "itens_retornados", "produtos_priorizar",
    "td_rate_medio_produto", "td_rate_mediano_produto", "td_rate_categoria_recalculado",
]

TOP_PROBLEMAS_COLS = [
    "problema_tag", "produtos", "produtos_priorizar",
    "pct_medio_no_produto",
]


# ── Funções de validação de contrato ─────────────────────────────────────────
def assert_required_columns(df: pd.DataFrame, required_columns: list, df_name: str) -> None:
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{df_name} sem colunas obrigatórias: {missing}")


def assert_output_headers(df: pd.DataFrame, expected_headers: list, payload_name: str) -> None:
    actual = list(df.columns)
    if actual != expected_headers:
        raise ValueError(
            f"{payload_name} com headers incorretos.\n"
            f"Esperado: {expected_headers}\n"
            f"Atual   : {actual}"
        )


print("✅ Headers de contrato e funções de assert definidos")


✅ Headers de contrato e funções de assert definidos


In [106]:
# ── Funções de construção de payload ─────────────────────────────────────────
# Cada função produz um DataFrame com as colunas finais da planilha existente.

def _select_available(df: pd.DataFrame, cols: list) -> pd.DataFrame:
    "Seleciona colunas disponíveis no df. Mantido para debug/apêndice."
    available = [c for c in cols if c in df.columns]
    return df[available].copy()


# ── Mapeamentos técnico → header da planilha ─────────────────────────────────
FAROL_COMPLETO_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "portfolio_cluster": "Cluster",
    "sinal_priorizacao": "Sinal Farol",
    "priority_score": "Priority Score",
    "td_score": "TD Score",
    "commercial_score_v2": "Comm. Score v2",
    "td_rate": "TD Rate",
    "td_rate_categoria": "TD Categ.",
    "delta_vs_categoria": "Δ vs Cat.",
    "top_1_problema": "Top 1 Problema",
    "top_2_problema": "Top 2",
    "top_3_problema": "Top 3",
    "tendencia_reversas": "Tendência",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
    "share_td_portfolio": "Share T&D",
}

PRIORIZAR_MELHORIA_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "portfolio_cluster": "Cluster",
    "priority_score": "Priority Score",
    "td_score": "TD Score",
    "commercial_score_v2": "Comm. Score v2",
    "td_rate": "TD Rate",
    "td_rate_categoria": "TD Categ.",
    "delta_vs_categoria": "Δ vs Cat.",
    "top_1_problema": "Top Problema 1",
    "top_2_problema": "Top 2",
    "top_3_problema": "Top 3",
    "tendencia_reversas": "Tendência",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
}

RELATORIO_PF_COLUMNS = {
    "product_name": "Produto",
    "category": "Categoria",
    "gender": "Gênero",
    "cluster": "Cluster",
    "sinal_pf_final": "Sinal Kill/Keep",
    "td_score": "TD Score",
    "score_tracao_comercial": "Score Tração",
    "score_unit_economics": "Score Unit Econ.",
    "score_satisfacao_marca": "Score Satisf.",
    "td_categoria_pct": "TD Categoria",
    "td_produto_pct": "TD Produto",
    "delta_vs_categoria": "Δ vs Cat.",
    "tendencia_reversas": "Tendência",
    "top_3_motivos_td": "Top 3 Motivos de T&D",
    "tweet_analitico": "Diagnóstico (Tweet)",
    "priority_score": "Priority Score",
    "qt_items_returned": "Itens Retorn.",
    "receita_liquida": "Receita Líq.",
}


def build_farol_completo_payload(executive_df: pd.DataFrame) -> pd.DataFrame:
    df = executive_df.copy()
    if "portfolio_cluster" not in df.columns and "cluster" in df.columns:
        df["portfolio_cluster"] = df["cluster"]
    assert_required_columns(df, list(FAROL_COMPLETO_COLUMNS.keys()), "executive_df → Farol Completo")
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(FAROL_COMPLETO_COLUMNS.keys())].rename(columns=FAROL_COMPLETO_COLUMNS)
    out = out[FAROL_COMPLETO_HEADERS]
    assert_output_headers(out, FAROL_COMPLETO_HEADERS, "farol_completo_df")
    return out.reset_index(drop=True)


def build_priorizar_melhoria_payload(executive_df: pd.DataFrame) -> pd.DataFrame:
    df = executive_df.copy()
    if "portfolio_cluster" not in df.columns and "cluster" in df.columns:
        df["portfolio_cluster"] = df["cluster"]
    assert_required_columns(
        df,
        ["sinal_priorizacao"] + list(PRIORIZAR_MELHORIA_COLUMNS.keys()),
        "executive_df → Priorizar Melhoria",
    )
    df = df[df["sinal_priorizacao"].eq("Priorizar melhoria")].copy()
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(PRIORIZAR_MELHORIA_COLUMNS.keys())].rename(columns=PRIORIZAR_MELHORIA_COLUMNS)
    out.insert(0, "Rank", range(1, len(out) + 1))
    out = out[PRIORIZAR_MELHORIA_HEADERS]
    assert_output_headers(out, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
    return out.reset_index(drop=True)


def build_relatorio_pf_payload(scorecard_view_df: pd.DataFrame) -> pd.DataFrame:
    df = scorecard_view_df.copy()
    # Usa sinal_kill_keep quando disponível; fallback para sinal_priorizacao
    if "sinal_kill_keep" in df.columns:
        df["sinal_pf_final"] = df["sinal_kill_keep"]
    elif "sinal_priorizacao" in df.columns:
        df["sinal_pf_final"] = df["sinal_priorizacao"]
    else:
        df["sinal_pf_final"] = ""
    # Preenche colunas ausentes com NaN + aviso
    for col in RELATORIO_PF_COLUMNS.keys():
        if col not in df.columns:
            df[col] = float("nan")
            warnings.warn(f"Coluna ausente em scorecard_view_df, preenchida como NaN: {col}")
    df = df.sort_values(
        ["priority_score", "qt_items_returned", "receita_liquida"],
        ascending=[False, False, False],
    )
    out = df[list(RELATORIO_PF_COLUMNS.keys())].rename(columns=RELATORIO_PF_COLUMNS)
    out = out[RELATORIO_PF_HEADERS]
    assert_output_headers(out, RELATORIO_PF_HEADERS, "relatorio_pf_df")
    return out.reset_index(drop=True)


def build_benchmark_categoria_payload(benchmark_df: pd.DataFrame) -> pd.DataFrame:
    df = benchmark_df.copy()
    assert_required_columns(df, [
        "category", "produtos", "unidades_vendidas", "receita_liquida",
        "itens_retornados", "produtos_priorizar",
        "td_rate_medio_produto", "td_rate_mediano_produto", "td_rate_categoria_recalculado",
    ], "benchmark_categoria_raw_df")

    def _urgencia(row):
        n = row.get("produtos_priorizar", 0) or 0
        if n >= 3:
            return "🟠 Atenção"
        if n >= 1:
            return "🟡 Média"
        return "🟢 Baixa"

    out = pd.DataFrame({
        "Categoria":          df["category"].values,
        "Qtd Produtos":       df["produtos"].values,
        "Unid. Vendidas":     df["unidades_vendidas"].values,
        "Receita Líq.":       df["receita_liquida"].values,
        "Itens Retornados":   df["itens_retornados"].values,
        "Produtos Priorizar": df["produtos_priorizar"].values,
        "TD Rate Médio":      df["td_rate_medio_produto"].values,
        "TD Rate Mediano":    df["td_rate_mediano_produto"].values,
        "TD Rate Categoria":  df["td_rate_categoria_recalculado"].values,
        "Urgência":           df.apply(_urgencia, axis=1).values,
    })
    out = out.sort_values(
        ["Produtos Priorizar", "Itens Retornados", "Receita Líq."],
        ascending=[False, False, False],
    )
    out = out[BENCHMARK_CATEGORIA_HEADERS]
    assert_output_headers(out, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
    return out.reset_index(drop=True)


def classify_problem_tag(tag) -> str:
    if pd.isna(tag):
        return "🔍 Outros"
    tag = str(tag)
    if tag in LOGISTICA_TAGS:
        return "🚚 Logístico — fora do escopo PF"
    action = TAG_TO_ACTION.get(tag)
    if action == "Modelagem":
        return "✂️ Modelagem"
    if action == "Grade":
        return "📏 Grade/Tamanho"
    if action == "Tecido":
        return "🧵 Matéria-Prima"
    if action == "PDP/Comunicação":
        return "🖼️ PDP/Comunicação"
    if action:
        return f"🔍 {action}"
    return "🔍 Outros"


def build_top_problemas_payload(top_problemas_raw_df: pd.DataFrame) -> pd.DataFrame:
    df = top_problemas_raw_df.copy()
    assert_required_columns(df, [
        "problema_tag", "produtos", "produtos_priorizar", "pct_medio_no_produto",
    ], "top_problemas_raw_df")
    df = df.sort_values(
        ["produtos_priorizar", "produtos", "pct_medio_no_produto"],
        ascending=[False, False, False],
    ).reset_index(drop=True)
    out = pd.DataFrame({
        "Tag do Problema":      df["problema_tag"].values,
        "Produtos Afetados":    df["produtos"].values,
        "Produtos 'Priorizar'": df["produtos_priorizar"].values,
        "% Médio no Produto":   df["pct_medio_no_produto"].values,
        "Rank Severidade":      range(1, len(df) + 1),
        "Classificação":        df["problema_tag"].apply(classify_problem_tag).values,
    })
    out = out[TOP_PROBLEMAS_HEADERS]
    assert_output_headers(out, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
    return out


# ── Resumo executivo ──────────────────────────────────────────────────────────
BUCKETS = [
    "Priorizar melhoria",
    "Alerta em produto relevante",
    "Monitorar",
    "Não priorizar agora",
    "Sem evidência suficiente",
]


def _fmt_brl_compact(value: float) -> str:
    v = float(value or 0)
    if abs(v) >= 1_000_000:
        return f"R$ {v / 1_000_000:.1f}M"
    if abs(v) >= 1_000:
        return f"R$ {v / 1_000:.1f}k"
    return f"R$ {v:,.0f}"


def _fmt_int_br(value: float) -> str:
    return f"{int(value or 0):,}".replace(",", ".")


def build_resumo_executivo_payload(executive_df: pd.DataFrame) -> dict:
    df = executive_df.copy()
    assert_required_columns(
        df,
        ["product_name", "sinal_priorizacao", "receita_liquida", "qt_items_returned"],
        "executive_df → Resumo Executivo",
    )
    grouped = (
        df.groupby("sinal_priorizacao", dropna=False)
        .agg(
            produtos=("product_name", "nunique"),
            receita_liquida=("receita_liquida", "sum"),
            itens_retornados=("qt_items_returned", "sum"),
        )
        .reset_index()
    )
    gmap = grouped.set_index("sinal_priorizacao").to_dict(orient="index")
    bucket_cards = {}
    for bucket in BUCKETS:
        v = gmap.get(bucket, {"produtos": 0, "receita_liquida": 0, "itens_retornados": 0})
        receita = float(v.get("receita_liquida", 0) or 0)
        itens = float(v.get("itens_retornados", 0) or 0)
        bucket_cards[bucket] = {
            "produtos": int(v.get("produtos", 0) or 0),
            "receita_liquida": receita,
            "receita_liq_fmt": _fmt_brl_compact(receita),
            "itens_retornados": int(itens),
            "itens_retornados_fmt": _fmt_int_br(itens),
        }
    generated_at = RUN_DATE.strftime("%d/%m/%Y %H:%M")
    total_produtos = int(df["product_name"].nunique())
    return {
        "generated_at": generated_at,
        "total_produtos": total_produtos,
        "periodo_texto": "12 meses de dados (Troquecommerce)",
        "subtitle": (
            f"Análise de {total_produtos} produtos · "
            f"12 meses de dados (Troquecommerce) · "
            f"Gerado em {generated_at}"
        ),
        "bucket_cards": bucket_cards,
    }


print("✅ Funções de payload e resumo executivo definidas")


✅ Funções de payload e resumo executivo definidas


In [107]:
# ── Construção dos payloads finais ────────────────────────────────────────────
farol_completo_df = build_farol_completo_payload(executive_prepared_df)

priorizar_melhoria_df = build_priorizar_melhoria_payload(executive_prepared_df)

if scorecard_view_df is not None and not scorecard_view_df.empty:
    relatorio_pf_df = build_relatorio_pf_payload(scorecard_view_df)
else:
    warnings.warn("scorecard_view_df não disponível — relatorio_pf_df será None.")
    relatorio_pf_df = None

benchmark_categoria_df = build_benchmark_categoria_payload(benchmark_categoria_raw_df)

top_problemas_df = build_top_problemas_payload(top_problemas_raw_df)

resumo_payload = build_resumo_executivo_payload(executive_prepared_df)

# ── Resumo legado (mantido para export_priority_workbook no debug Excel) ──────
resumo_df = summary_df.copy()

print("✅ Payloads finais construídos")
print(f"  farol_completo_df      : {farol_completo_df.shape}")
print(f"  priorizar_melhoria_df  : {priorizar_melhoria_df.shape}")
print(f"  relatorio_pf_df        : {relatorio_pf_df.shape if relatorio_pf_df is not None else 'None'}")
print(f"  benchmark_categoria_df : {benchmark_categoria_df.shape}")
print(f"  top_problemas_df       : {top_problemas_df.shape}")
print(f"  resumo_payload buckets : {list(resumo_payload['bucket_cards'].keys())}")

# ── QA dos payloads ───────────────────────────────────────────────────────────
assert_output_headers(farol_completo_df, FAROL_COMPLETO_HEADERS, "farol_completo_df")
assert_output_headers(priorizar_melhoria_df, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
assert_output_headers(benchmark_categoria_df, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
assert_output_headers(top_problemas_df, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
if relatorio_pf_df is not None:
    assert_output_headers(relatorio_pf_df, RELATORIO_PF_HEADERS, "relatorio_pf_df")

if farol_completo_df.empty:
    raise ValueError("farol_completo_df vazio. Abortando — pipeline sem dados.")

if priorizar_melhoria_df.empty:
    warnings.warn("priorizar_melhoria_df vazio. A aba será limpa e ficará sem linhas de dados.")

if relatorio_pf_df is None or relatorio_pf_df.empty:
    warnings.warn("relatorio_pf_df vazio/None. Aba Relatório PF não será atualizada.")

print("✅ QA de payloads concluído")


✅ Payloads finais construídos
  farol_completo_df      : (117, 18)
  priorizar_melhoria_df  : (14, 17)
  relatorio_pf_df        : (50, 18)
  benchmark_categoria_df : (25, 10)
  top_problemas_df       : (22, 6)
  resumo_payload buckets : ['Priorizar melhoria', 'Alerta em produto relevante', 'Monitorar', 'Não priorizar agora', 'Sem evidência suficiente']
✅ QA de payloads concluído


/var/folders/m7/t8xk69wj7xl3lrprq0gflfqc0000gn/T/ipykernel_38000/117559295.py:122: UserWarning: Coluna ausente em scorecard_view_df, preenchida como NaN: delta_vs_categoria
  warnings.warn(f"Coluna ausente em scorecard_view_df, preenchida como NaN: {col}")


## 11. Atualização/exportação dos resultados

Atualiza a planilha Google Sheets **existente** e (opcionalmente) exporta Excel de debug.

**Entrada:** todos os DataFrames de payload do Bloco 10
**Saída:** planilha atualizada / arquivo Excel de debug
**Quando mexer:** ao alterar autenticação, ID da planilha ou tabs exportadas.

> ⚠️ **Não criar uma nova planilha** — sempre usar `open_by_key(SPREADSHEET_ID)`.
>
> **Variáveis de ambiente necessárias:**
> - `GOOGLE_SERVICE_ACCOUNT_JSON` (Deepnote/CI) — JSON do service account com acesso ao Drive/Sheets
> - `TD_PRIORIZACAO_SPREADSHEET_ID` (opcional) — substitui o ID hardcoded no Bloco 2
> - `BQ_PROJECT_ID` (opcional) — substitui o projeto BigQuery padrão
>
> **Fallback local (Mac):** se `GOOGLE_SERVICE_ACCOUNT_JSON` não estiver definido,
> usa `gcloud auth print-access-token` (requer `gcloud auth login --enable-gdrive-access`).


In [ ]:
# ── Constantes de formatação para Sheets API v4 ───────────────────────────────
# Cores como dicts RGB (0.0–1.0), formato exigido pela Sheets API.
# Referência: documento de formatação TD_Priorizacao_Melhorias.

def hex_to_rgb(h: str) -> dict:
    """Converte hex 6 dígitos sem '#' para dict Sheets API RGB (0.0–1.0)."""
    h = h.lstrip("#")
    return {
        "red":   int(h[0:2], 16) / 255,
        "green": int(h[2:4], 16) / 255,
        "blue":  int(h[4:6], 16) / 255,
    }

# ── Cores base ────────────────────────────────────────────────────────────────
_WHITE       = hex_to_rgb("FFFFFF")
_GRAY_LIGHT  = hex_to_rgb("F5F5F5")  # linhas ímpares (0-indexed)
_GRAY_BORDER = hex_to_rgb("E0E0E0")  # bordas thin
_DARK_TEXT   = hex_to_rgb("2D2D2D")  # texto padrão de valor

# ── Sinal Farol / Kill-Keep ───────────────────────────────────────────────────
_SINAL_FMT = {
    "Priorizar melhoria":          {"bg": hex_to_rgb("FF4444"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Alerta em produto relevante": {"bg": hex_to_rgb("FFC107"), "fg": hex_to_rgb("1A1A1A"), "bold": True},
    "Monitorar":                   {"bg": hex_to_rgb("FF8C00"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Não priorizar agora":         {"bg": hex_to_rgb("4CAF50"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
    "Sem evidência suficiente":    {"bg": hex_to_rgb("BDBDBD"), "fg": hex_to_rgb("FFFFFF"), "bold": True},
}

# ── Cluster (somente cor de fonte; fundo da linha permanece alternado) ────────
_CLUSTER_FG = {
    "HERO":       hex_to_rgb("1565C0"),  # azul escuro
    "CORE":       hex_to_rgb("2E7D32"),  # verde escuro
    "LONG TAIL":  hex_to_rgb("6A1B9A"),  # roxo
    "KILL":       hex_to_rgb("B71C1C"),  # vermelho escuro (+ italic)
    "LANCAMENTO": hex_to_rgb("F57F17"),  # laranja
}

# ── Tendência (fundo da célula inteira) ───────────────────────────────────────
_TENDENCIA_BG = {
    "Aumentou nos últimos 3 meses":  hex_to_rgb("FFEBEE"),
    "Apareceu nos últimos 3 meses":  hex_to_rgb("FFEBEE"),
    "Estável":                        hex_to_rgb("F5F5F5"),
    "Caiu nos últimos 3 meses":       hex_to_rgb("E8F5E9"),
    "Sem volume para tendência":      hex_to_rgb("F5F5F5"),
}

# ── Alertas de threshold (Benchmark Categoria e Top Problemas) ────────────────
_THRESHOLD_FMT = {
    "critical": {"fg": hex_to_rgb("C62828"), "bold": True},
    "warning":  {"fg": hex_to_rgb("E65100"), "bold": True},
    "normal":   {"fg": _DARK_TEXT,            "bold": False},
}

# ── Configuração de formatação por aba ────────────────────────────────────────
# Índices de coluna são 0-based (A=0, B=1, ...).
# "sinal_col"    : índice da coluna com valor de sinal (para colorir fundo/texto)
# "cluster_col"  : índice da coluna Cluster (para colorir fonte)
# "tendencia_col": índice da coluna Tendência (para colorir fundo)
# "produto_col"  : índice da coluna Produto/nome (bold + left-align)
# "rank_col"     : índice da coluna Rank, se existir (bold vermelho)
# "number_formats": {col_idx: format_string}  — formato numérico por coluna
# "wrap_cols"    : colunas com wrap_text=True
# "threshold_col": coluna com alerta de threshold + "threshold_fn" classificadora
# "n_cols"       : número total de colunas da aba

FORMATTING_CONFIG = {
    "farol_completo": {
        "produto_col":    0,    # Produto
        "cluster_col":    3,    # Cluster
        "sinal_col":      4,    # Sinal Farol
        "tendencia_col":  14,   # Tendência
        "rank_col":       None,
        "threshold_col":  None,
        "n_cols":         18,
        "number_formats": {
            5:  "0.000",      # Priority Score
            6:  "0.000",      # TD Score
            7:  "0.000",      # Comm. Score v2
            8:  "0.0%",       # TD Rate
            9:  "0.0%",       # TD Categ.
            10: "0.0%",       # Δ vs Cat.
            15: "#,##0",      # Itens Retorn.
            16: "R$ #,##0",   # Receita Líq.
            17: "0.0%",       # Share T&D
        },
        "wrap_cols": [],
    },
    "priorizar_melhoria": {
        "produto_col":    1,    # Produto (col A = Rank)
        "cluster_col":    4,    # Cluster
        "sinal_col":      None, # toda aba é "Priorizar melhoria" — sem coluna sinal
        "tendencia_col":  14,   # Tendência
        "rank_col":       0,    # Rank
        "threshold_col":  None,
        "n_cols":         17,
        "number_formats": {
            5:  "0.000",      # Priority Score
            6:  "0.000",      # TD Score
            7:  "0.000",      # Comm. Score v2
            8:  "0.0%",       # TD Rate
            9:  "0.0%",       # TD Categ.
            10: "0.0%",       # Δ vs Cat.
            15: "#,##0",      # Itens Retorn.
            16: "R$ #,##0",   # Receita Líq.
        },
        "wrap_cols": [],
    },
    "relatorio_pf": {
        "produto_col":    0,    # Produto
        "cluster_col":    3,    # Cluster
        "sinal_col":      4,    # Sinal Kill/Keep
        "tendencia_col":  12,   # Tendência
        "rank_col":       None,
        "threshold_col":  None,
        "n_cols":         18,
        "number_formats": {
            5:  "0.0",        # TD Score
            6:  "0.0",        # Score Tração
            7:  "0.0",        # Score Unit Econ.
            8:  "0.0",        # Score Satisf.
            9:  "0.0%",       # TD Categoria
            10: "0.0%",       # TD Produto
            11: "0.0%",       # Δ vs Cat.
            15: "0.000",      # Priority Score
            16: "#,##0",      # Itens Retorn.
            17: "R$ #,##0",   # Receita Líq.
        },
        "wrap_cols": [13, 14],  # Top 3 Motivos (13) e Diagnóstico (14)
    },
    "benchmark_categoria": {
        "produto_col":    0,    # Categoria (bold left)
        "cluster_col":    None,
        "sinal_col":      None,
        "tendencia_col":  None,
        "rank_col":       None,
        "threshold_col":  5,    # Produtos Priorizar
        "threshold_fn":   lambda v: (
            "critical" if (v or 0) >= 2 else
            "warning"  if (v or 0) == 1 else
            "normal"
        ),
        "n_cols": 10,
        "number_formats": {
            1: "#,##0",       # Qtd Produtos
            2: "#,##0",       # Unid. Vendidas
            3: "R$ #,##0",    # Receita Líq.
            4: "#,##0",       # Itens Retornados
            5: "#,##0",       # Produtos Priorizar
            6: "0.0%",        # TD Rate Médio
            7: "0.0%",        # TD Rate Mediano
            8: "0.0%",        # TD Rate Categoria
        },
        "wrap_cols": [],
    },
    "top_problemas": {
        "produto_col":    0,    # Tag do Problema
        "cluster_col":    None,
        "sinal_col":      None,
        "tendencia_col":  None,
        "rank_col":       None,
        "threshold_col":  2,    # Produtos 'Priorizar'
        "threshold_fn":   lambda v: (
            "critical" if (v or 0) >= 5 else
            "warning"  if (v or 0) >= 2 else
            "normal"
        ),
        "n_cols": 6,
        "number_formats": {
            1: "#,##0",       # Produtos Afetados
            2: "#,##0",       # Produtos 'Priorizar'
            3: "0.0%",        # % Médio no Produto
            4: "#,##0",       # Rank Severidade
        },
        "wrap_cols": [],
    },
}

print("✅ Constantes de formatação Sheets API definidas")
print("✅ FORMATTING_CONFIG definido")


In [ ]:
# ── Helpers internos de construção de requests ────────────────────────────────

def _range(sheet_id: int, r0: int, r1: int, c0: int, c1: int) -> dict:
    """Range object Sheets API. r0/c0 inclusive, r1/c1 exclusive (0-indexed)."""
    return {
        "sheetId":          sheet_id,
        "startRowIndex":    r0,
        "endRowIndex":      r1,
        "startColumnIndex": c0,
        "endColumnIndex":   c1,
    }

def _cell_fmt(sheet_id: int, row_0: int, col_0: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para UMA célula. row_0/col_0 são 0-indexed."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, row_0, row_0 + 1, col_0, col_0 + 1),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }

def _col_fmt(sheet_id: int, r0: int, r1: int, col_0: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para uma coluna inteira no range r0..r1 (exclusive)."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, r0, r1, col_0, col_0 + 1),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }

def _row_fmt(sheet_id: int, row_0: int, n_cols: int, fmt: dict, fields: str) -> dict:
    """repeatCell request para uma linha inteira (0..n_cols)."""
    return {
        "repeatCell": {
            "range": _range(sheet_id, row_0, row_0 + 1, 0, n_cols),
            "cell":  {"userEnteredFormat": fmt},
            "fields": fields,
        }
    }


def build_base_requests(sheet_id: int, start_row: int, n_rows: int, n_cols: int) -> list:
    """
    Retorna requests para:
      1. Fundo alternado (branco/cinza) por linha de dado.
      2. Bordas thin cinza em toda a range de dados.
      3. Alinhamento central (horizontal=CENTER, vertical=MIDDLE) em toda a range.

    start_row : linha Excel 1-indexed (ex: 3).
    n_rows    : número de linhas de dado.
    n_cols    : número de colunas da aba.
    """
    reqs   = []
    start_0 = start_row - 1  # 0-indexed

    # ── 1. Fundo alternado ─────────────────────────────────────────────────
    # i=0 (primeira linha de dado) → branco; i=1 → cinza; etc.
    for i in range(n_rows):
        bg = _WHITE if i % 2 == 0 else _GRAY_LIGHT
        reqs.append(_row_fmt(
            sheet_id,
            start_0 + i,
            n_cols,
            {"backgroundColor": bg},
            "userEnteredFormat.backgroundColor",
        ))

    # ── 2. Bordas thin em toda a range ────────────────────────────────────
    thin = {"style": "SOLID", "colorStyle": {"rgbColor": _GRAY_BORDER}, "width": 1}
    reqs.append({
        "updateBorders": {
            "range":           _range(sheet_id, start_0, start_0 + n_rows, 0, n_cols),
            "top":             thin,
            "bottom":          thin,
            "left":            thin,
            "right":           thin,
            "innerHorizontal": thin,
            "innerVertical":   thin,
        }
    })

    # ── 3. Alinhamento central padrão ──────────────────────────────────────
    reqs.append({
        "repeatCell": {
            "range": _range(sheet_id, start_0, start_0 + n_rows, 0, n_cols),
            "cell": {
                "userEnteredFormat": {
                    "horizontalAlignment": "CENTER",
                    "verticalAlignment":   "MIDDLE",
                }
            },
            "fields": (
                "userEnteredFormat.horizontalAlignment,"
                "userEnteredFormat.verticalAlignment"
            ),
        }
    })

    return reqs


def build_column_format_requests(
    sheet_id: int,
    start_row: int,
    n_rows: int,
    config: dict,
) -> list:
    """
    Retorna requests para:
      1. Formato numérico por coluna (number_formats do config).
      2. Coluna Produto: bold + left-align + padding.
      3. Coluna Rank (se existir): bold vermelho C62828 size 11.
      4. Colunas com wrap_text (wrapStrategy=WRAP, verticalAlignment=TOP).
         - Coluna 14 em relatorio_pf (Diagnóstico): também italic + cinza 424242.
    """
    reqs    = []
    start_0 = start_row - 1
    end_0   = start_0 + n_rows

    # ── 1. Formatos numéricos ─────────────────────────────────────────────
    for col_0, pattern in config.get("number_formats", {}).items():
        reqs.append(_col_fmt(
            sheet_id, start_0, end_0, col_0,
            {"numberFormat": {"type": "NUMBER", "pattern": pattern}},
            "userEnteredFormat.numberFormat",
        ))

    # ── 2. Coluna Produto: bold + left-align ──────────────────────────────
    pc = config["produto_col"]
    reqs.append(_col_fmt(
        sheet_id, start_0, end_0, pc,
        {
            "textFormat":          {"bold": True},
            "horizontalAlignment": "LEFT",
        },
        "userEnteredFormat.textFormat.bold,userEnteredFormat.horizontalAlignment",
    ))

    # ── 3. Coluna Rank: bold vermelho C62828 size 11 ──────────────────────
    rc = config.get("rank_col")
    if rc is not None:
        reqs.append(_col_fmt(
            sheet_id, start_0, end_0, rc,
            {
                "textFormat": {
                    "bold":     True,
                    "fontSize": 11,
                    "foregroundColorStyle": {"rgbColor": hex_to_rgb("C62828")},
                },
                "horizontalAlignment": "CENTER",
            },
            (
                "userEnteredFormat.textFormat.bold,"
                "userEnteredFormat.textFormat.fontSize,"
                "userEnteredFormat.textFormat.foregroundColorStyle,"
                "userEnteredFormat.horizontalAlignment"
            ),
        ))

    # ── 4. Colunas com wrap_text ───────────────────────────────────────────
    for col_0 in config.get("wrap_cols", []):
        is_diagnostico = (col_0 == 14)
        fmt = {
            "wrapStrategy":        "WRAP",
            "verticalAlignment":   "TOP",
            "horizontalAlignment": "LEFT",
        }
        fields = (
            "userEnteredFormat.wrapStrategy,"
            "userEnteredFormat.verticalAlignment,"
            "userEnteredFormat.horizontalAlignment"
        )
        if is_diagnostico:
            fmt["textFormat"] = {
                "italic": True,
                "foregroundColorStyle": {"rgbColor": hex_to_rgb("424242")},
            }
            fields += (
                ",userEnteredFormat.textFormat.italic"
                ",userEnteredFormat.textFormat.foregroundColorStyle"
            )
        reqs.append(_col_fmt(sheet_id, start_0, end_0, col_0, fmt, fields))

    return reqs


def build_conditional_cell_requests(
    sheet_id: int,
    start_row: int,
    df: "pd.DataFrame",
    config: dict,
) -> list:
    """
    Retorna requests por célula com base no valor do DataFrame:
      - sinal_col:     fundo + texto colorido (sinal farol / kill-keep)
      - cluster_col:   cor da fonte (+ italic se KILL)
      - tendencia_col: fundo da célula
      - threshold_col: cor da fonte por threshold_fn

    Usa acesso posicional (list(row)) para robustez com nomes de coluna
    contendo caracteres especiais (Δ, acentos, emojis).
    """
    if len(df.columns) != config["n_cols"]:
        raise ValueError(
            f"build_conditional_cell_requests: DataFrame tem {len(df.columns)} colunas, "
            f"FORMATTING_CONFIG espera {config['n_cols']}.\n"
            f"Colunas do df: {list(df.columns)}"
        )

    reqs          = []
    start_0       = start_row - 1
    sinal_col     = config.get("sinal_col")
    cluster_col   = config.get("cluster_col")
    tendencia_col = config.get("tendencia_col")
    threshold_col = config.get("threshold_col")
    threshold_fn  = config.get("threshold_fn")

    for i, row in enumerate(df.itertuples(index=False)):
        row_0  = start_0 + i
        row_v  = list(row)  # acesso posicional — seguro com qualquer nome de coluna

        # ── Sinal Farol / Kill-Keep ────────────────────────────────────────
        if sinal_col is not None:
            sinal  = str(row_v[sinal_col] or "")
            fmt_s  = _SINAL_FMT.get(sinal)
            if fmt_s:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, sinal_col,
                    {
                        "backgroundColor": fmt_s["bg"],
                        "textFormat": {
                            "bold":  fmt_s["bold"],
                            "foregroundColorStyle": {"rgbColor": fmt_s["fg"]},
                        },
                        "horizontalAlignment": "CENTER",
                    },
                    (
                        "userEnteredFormat.backgroundColor,"
                        "userEnteredFormat.textFormat.bold,"
                        "userEnteredFormat.textFormat.foregroundColorStyle,"
                        "userEnteredFormat.horizontalAlignment"
                    ),
                ))

        # ── Cluster: cor de fonte (+ italic se KILL) ───────────────────────
        if cluster_col is not None:
            cluster = str(row_v[cluster_col] or "").upper().strip()
            fg      = _CLUSTER_FG.get(cluster)
            if fg:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, cluster_col,
                    {
                        "textFormat": {
                            "bold":   True,
                            "italic": (cluster == "KILL"),
                            "foregroundColorStyle": {"rgbColor": fg},
                        }
                    },
                    (
                        "userEnteredFormat.textFormat.bold,"
                        "userEnteredFormat.textFormat.italic,"
                        "userEnteredFormat.textFormat.foregroundColorStyle"
                    ),
                ))

        # ── Tendência: fundo da célula ─────────────────────────────────────
        if tendencia_col is not None:
            tend = str(row_v[tendencia_col] or "")
            bg   = _TENDENCIA_BG.get(tend)
            if bg:
                reqs.append(_cell_fmt(
                    sheet_id, row_0, tendencia_col,
                    {"backgroundColor": bg},
                    "userEnteredFormat.backgroundColor",
                ))

        # ── Threshold (Benchmark Categoria, Top Problemas) ─────────────────
        if threshold_col is not None and threshold_fn is not None:
            raw_val = row_v[threshold_col]
            level   = threshold_fn(raw_val)
            fmt_t   = _THRESHOLD_FMT.get(level, _THRESHOLD_FMT["normal"])
            reqs.append(_cell_fmt(
                sheet_id, row_0, threshold_col,
                {
                    "textFormat": {
                        "bold": fmt_t["bold"],
                        "foregroundColorStyle": {"rgbColor": fmt_t["fg"]},
                    }
                },
                (
                    "userEnteredFormat.textFormat.bold,"
                    "userEnteredFormat.textFormat.foregroundColorStyle"
                ),
            ))

    return reqs


def apply_tab_formatting(
    spreadsheet,
    worksheet,
    df: "pd.DataFrame",
    tab_key: str,
    start_row: int = 3,
) -> None:
    """
    Aplica formatação completa às linhas de dados recém-escritas via Sheets API batchUpdate.

    Garantias:
      - Nunca toca nas linhas 1 (banner) ou 2 (headers).
      - Envia todos os requests em UM único batchUpdate.
      - Não aplica formatação fora do range de dados (start_row .. start_row+len(df)-1).

    Parâmetros:
      spreadsheet : objeto gspread.Spreadsheet
      worksheet   : objeto gspread.Worksheet (destino)
      df          : DataFrame escrito (sem header, sem index)
      tab_key     : chave em FORMATTING_CONFIG
      start_row   : linha Excel onde os dados começam (default 3, 1-indexed)
    """
    config = FORMATTING_CONFIG.get(tab_key)
    if config is None:
        print(f"  ⚠️  apply_tab_formatting: '{tab_key}' sem config — formatação pulada.")
        return

    n_rows = len(df)
    if n_rows == 0:
        return

    if start_row < 3:
        raise ValueError(
            f"apply_tab_formatting: start_row={start_row} inseguro. "
            f"Formatação nunca deve tocar nas linhas 1 (banner) ou 2 (headers)."
        )

    # sheetId inteiro exigido pela Sheets API
    sheet_id = getattr(worksheet, "id", None)
    if sheet_id is None:
        sheet_id = worksheet._properties["sheetId"]
    n_cols   = config["n_cols"]

    reqs  = []
    reqs += build_base_requests(sheet_id, start_row, n_rows, n_cols)
    reqs += build_column_format_requests(sheet_id, start_row, n_rows, config)
    reqs += build_conditional_cell_requests(sheet_id, start_row, df, config)

    if reqs:
        spreadsheet.batch_update({"requests": reqs})
        print(f"    ↳ formatação: {len(reqs)} requests → '{worksheet.title}'")

print("✅ Funções de formatação Sheets API definidas")


In [ ]:
# ── Helpers de autenticação e escrita segura no Google Sheets ─────────────────
import re
import subprocess
import gspread
import google.oauth2.credentials
import google.oauth2.service_account
from gspread_dataframe import set_with_dataframe


def get_gspread_client() -> gspread.Client:
    "Retorna cliente gspread autenticado. Prioridade: 1) GOOGLE_SERVICE_ACCOUNT_JSON (Deepnote/CI), 2) gcloud token (local)."
    sa_json = os.getenv("GOOGLE_SERVICE_ACCOUNT_JSON")
    if sa_json:
        info = json.loads(sa_json)
        creds = google.oauth2.service_account.Credentials.from_service_account_info(
            info,
            scopes=[
                "https://www.googleapis.com/auth/spreadsheets",
                "https://www.googleapis.com/auth/drive",
            ],
        )
        print("✅ GSheets auth: service account (GOOGLE_SERVICE_ACCOUNT_JSON)")
        return gspread.authorize(creds)

    # Fallback local: gcloud user credentials
    result = subprocess.run(
        ["gcloud", "auth", "print-access-token"],
        capture_output=True, text=True, check=True,
    )
    access_token = result.stdout.strip()
    creds = google.oauth2.credentials.Credentials(token=access_token)
    print("✅ GSheets auth: gcloud user credentials (local fallback)")
    return gspread.Client(auth=creds)


# ── Configuração das abas da planilha existente ───────────────────────────────
TAB_CONFIG = {
    "farol_completo": {
        "worksheet_name": "🗂️ Farol Completo",
        "data_range_prefix": "A3:R",
        "header_range": "A2:R2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": "A1",
        "title_template": "🗂️  FAROL COMPLETO — {n} Produtos com Evidência Suficiente",
        "expected_headers": FAROL_COMPLETO_HEADERS,
    },
    "priorizar_melhoria": {
        "worksheet_name": "🔴 Priorizar Melhoria",
        "data_range_prefix": "A3:Q",
        "header_range": "A2:Q2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": "A1",
        "title_template": "🔴  PRIORIZAR MELHORIA — Ranking de Urgência   ·   {n} produtos identificados",
        "expected_headers": PRIORIZAR_MELHORIA_HEADERS,
    },
    "relatorio_pf": {
        "worksheet_name": "📋 Relatório PF",
        "data_range_prefix": "A3:R",
        "header_range": "A2:R2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": RELATORIO_PF_HEADERS,
    },
    "benchmark_categoria": {
        "worksheet_name": "📂 Benchmark Categoria",
        "data_range_prefix": "A3:J",
        "header_range": "A2:J2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": BENCHMARK_CATEGORIA_HEADERS,
    },
    "top_problemas": {
        "worksheet_name": "🏷️ Top Problemas",
        "data_range_prefix": "A3:F",
        "header_range": "A2:F2",
        "start_row": 3,
        "start_col": 1,
        "title_cell": None,
        "title_template": None,
        "expected_headers": TOP_PROBLEMAS_HEADERS,
    },
}


# ── Helpers seguros — nunca criam abas, nunca limpam aba inteira ──────────────
def open_target_spreadsheet(gc: gspread.Client, spreadsheet_id: str = SPREADSHEET_ID):
    spreadsheet = gc.open_by_key(spreadsheet_id)
    actual_id = getattr(spreadsheet, "id", spreadsheet_id)
    if actual_id != spreadsheet_id:
        raise RuntimeError(
            f"Planilha aberta não corresponde ao SPREADSHEET_ID esperado. "
            f"Esperado: {spreadsheet_id}. Atual: {actual_id}"
        )
    return spreadsheet


def get_required_worksheet(spreadsheet: gspread.Spreadsheet, worksheet_name: str):
    try:
        return spreadsheet.worksheet(worksheet_name)
    except Exception as exc:
        available = [ws.title for ws in spreadsheet.worksheets()]
        raise RuntimeError(
            f"Aba esperada não encontrada: '{worksheet_name}'. "
            f"Abas disponíveis: {available}. "
            f"Abortando para evitar criar ou desformatar abas."
        ) from exc


def assert_headers_match(worksheet, expected_headers: list, header_range: str) -> None:
    values = worksheet.get(header_range)
    if not values:
        raise ValueError(
            f"Header vazio no range {worksheet.title}!{header_range}. Abortando escrita."
        )
    actual_headers = [str(h).strip() for h in values[0]]
    expected_clean = [str(h).strip() for h in expected_headers]
    if actual_headers != expected_clean:
        raise ValueError(
            f"Headers incompatíveis na aba '{worksheet.title}'.\n"
            f"Esperado: {expected_clean}\n"
            f"Atual   : {actual_headers}\n"
            f"Abortando para evitar sobrescrever dados em layout incompatível."
        )


def _range_with_last_row(data_range_prefix: str, last_row: int) -> str:
    if not data_range_prefix[-1].isalpha():
        return data_range_prefix
    return f"{data_range_prefix}{last_row}"


def clear_data_body_only(worksheet, data_range_prefix: str, min_rows_to_clear: int = 1000) -> None:
    "Limpa somente o corpo da tabela. Nunca inclui linhas 1–2. Nunca usa worksheet.clear()."
    last_row = max(worksheet.row_count, min_rows_to_clear)
    closed_range = _range_with_last_row(data_range_prefix, last_row)
    if not re.match(r"^[A-Z]+3:", closed_range):
        raise ValueError(
            f"Range de limpeza inseguro: '{closed_range}'. Deve começar na linha 3."
        )
    worksheet.batch_clear([closed_range])


def ensure_enough_rows(worksheet, required_rows: int) -> None:
    "Adiciona linhas se necessário. Não redimensiona para baixo."
    if worksheet.row_count < required_rows:
        worksheet.add_rows(required_rows - worksheet.row_count)


def write_dataframe_body(worksheet, df: pd.DataFrame, start_row: int = 3, start_col: int = 1) -> None:
    "Escreve DataFrame sem header e sem index, preservando a formatação existente."
    if start_row < 3:
        raise ValueError(
            f"start_row inseguro: {start_row}. Dados devem começar na linha 3 ou abaixo."
        )
    required_rows = start_row + max(len(df), 1) + 20
    ensure_enough_rows(worksheet, required_rows)
    set_with_dataframe(
        worksheet,
        df.astype(object).where(pd.notna(df), ""),
        row=start_row,
        col=start_col,
        include_index=False,
        include_column_header=False,
        resize=False,
    )


def update_title_if_needed(worksheet, title_cell, title) -> None:
    if title_cell and title:
        worksheet.update([[title]], title_cell)


def update_tab_from_dataframe(
    spreadsheet: gspread.Spreadsheet,
    tab_key: str,
    df: pd.DataFrame,
    title_n: "int | None" = None,
) -> None:
    config = TAB_CONFIG[tab_key]
    worksheet = get_required_worksheet(spreadsheet, config["worksheet_name"])
    assert_headers_match(worksheet, config["expected_headers"], config["header_range"])
    if list(df.columns) != config["expected_headers"]:
        raise ValueError(
            f"Payload de '{tab_key}' não bate com headers esperados.\n"
            f"Esperado: {config['expected_headers']}\n"
            f"Atual   : {list(df.columns)}"
        )
    clear_data_body_only(worksheet, config["data_range_prefix"], max(len(df) + 20, 1000))
    write_dataframe_body(worksheet, df, config["start_row"], config["start_col"])
    # ── Reaplica formatação a todas as linhas de dados após escrita ────────
    apply_tab_formatting(
        spreadsheet,
        worksheet,
        df,
        tab_key,
        start_row=config["start_row"],
    )
    if config["title_template"] and config["title_cell"]:
        n = title_n if title_n is not None else len(df)
        update_title_if_needed(worksheet, config["title_cell"], config["title_template"].format(n=n))
    print(f"  ✓ {config['worksheet_name']}: {len(df):,} linhas × {len(df.columns)} colunas")


def update_resumo_executivo(spreadsheet: gspread.Spreadsheet, resumo_payload: dict) -> None:
    "Atualiza apenas células específicas da aba Resumo Executivo. Não limpa a aba."
    worksheet = get_required_worksheet(spreadsheet, "📊 Resumo Executivo")
    if "subtitle" in resumo_payload:
        worksheet.update([[resumo_payload["subtitle"]]], "A2")
    bucket_cards = resumo_payload.get("bucket_cards", {})
    # ⚠️ Validar cell_map visualmente na planilha antes do primeiro uso em produção.
    cell_map = {
        "Priorizar melhoria":          {"produtos": "B6", "receita_liq_fmt": "B7", "itens_retornados_fmt": "B8"},
        "Alerta em produto relevante": {"produtos": "D6", "receita_liq_fmt": "D7", "itens_retornados_fmt": "D8"},
        "Monitorar":                   {"produtos": "F6", "receita_liq_fmt": "F7", "itens_retornados_fmt": "F8"},
        "Não priorizar agora":         {"produtos": "H6", "receita_liq_fmt": "H7", "itens_retornados_fmt": "H8"},
        "Sem evidência suficiente":    {"produtos": "J6", "receita_liq_fmt": "J7", "itens_retornados_fmt": "J8"},
    }
    updates = []
    for bucket, fields in cell_map.items():
        values = bucket_cards.get(bucket, {})
        for field, cell in fields.items():
            updates.append({"range": cell, "values": [[values.get(field, 0)]]})
    if updates:
        worksheet.batch_update(updates)
    print("  ✓ 📊 Resumo Executivo atualizado")


def update_all_tabs(
    spreadsheet: gspread.Spreadsheet,
    farol_df: pd.DataFrame,
    priorizar_df: pd.DataFrame,
    relatorio_pf_df: "pd.DataFrame | None",
    benchmark_df: pd.DataFrame,
    top_problemas_df: pd.DataFrame,
    resumo_payload: dict,
) -> None:
    """
    Atualiza a planilha existente preservando layout completo.

    Garantias:
    - NÃO cria planilha nova.
    - NÃO cria abas novas.
    - NÃO limpa abas inteiras (usa batch_clear somente no corpo, a partir de linha 3).
    - NÃO sobrescreve headers (linha 2) nem títulos mesclados (linha 1).
    - NÃO escreve index ou header no DataFrame.
    - NÃO altera a aba ℹ️ Legenda.
    - APLICA formatação completa (fundo alternado, bordas, cores condicionais)
      via Sheets API batchUpdate após cada write_dataframe_body.
    """
    update_tab_from_dataframe(spreadsheet, "priorizar_melhoria", priorizar_df, len(priorizar_df))
    if relatorio_pf_df is not None and not relatorio_pf_df.empty:
        update_tab_from_dataframe(spreadsheet, "relatorio_pf", relatorio_pf_df)
    else:
        print("  ⚠️  📋 Relatório PF: sem dados disponíveis — aba não atualizada.")
    update_tab_from_dataframe(spreadsheet, "benchmark_categoria", benchmark_df)
    update_tab_from_dataframe(spreadsheet, "top_problemas", top_problemas_df)
    update_tab_from_dataframe(spreadsheet, "farol_completo", farol_df, len(farol_df))
    # Resumo por último: reflete sucesso completo das abas tabulares.
    update_resumo_executivo(spreadsheet, resumo_payload)


print("✅ Helpers de escrita segura no GSheets definidos")


✅ Helpers de escrita segura no GSheets definidos


In [109]:
# ── Pre-flight: valida estrutura real da planilha antes de escrever ───────────
# Execute esta célula isolada ANTES de UPDATE_GOOGLE_SHEETS para confirmar:
#
#   Questão 2 — cell_map do Resumo Executivo:
#     Imprime o grid A1:J10 da aba para que você veja onde estão os cards
#     e possa corrigir o cell_map em update_resumo_executivo se necessário.
#
#   Questão 3 — headers reais vs esperados:
#     Para cada aba em TAB_CONFIG, lê a linha 2 e compara com EXPECTED_HEADERS.
#     Imprime ✅ MATCH ou ❌ DIVERGÊNCIA com diff detalhado.
#
# NÃO escreve nada. Seguro para rodar a qualquer momento.

_gc = get_gspread_client()
_sp = _gc.open_by_key(SPREADSHEET_ID)

print(f"Planilha: {_sp.title}")
print(f"URL     : https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
print(f"Abas    : {[ws.title for ws in _sp.worksheets()]}")
print()

# ── Questão 3: Validar headers linha 2 ───────────────────────────────────────
print("=" * 64)
print("QUESTÃO 3 — Headers reais vs esperados (linha 2 de cada aba)")
print("=" * 64)
_all_headers_ok = True
for _key, _cfg in TAB_CONFIG.items():
    try:
        _ws = _sp.worksheet(_cfg["worksheet_name"])
        _raw = _ws.get(_cfg["header_range"])
        _actual = [str(h).strip() for h in (_raw[0] if _raw else [])]
        _expected = [str(h).strip() for h in _cfg["expected_headers"]]
        if _actual == _expected:
            print(f"  ✅ {_cfg['worksheet_name']}")
        else:
            _all_headers_ok = False
            print(f"  ❌ {_cfg['worksheet_name']}")
            _max = max(len(_actual), len(_expected))
            for _i in range(_max):
                _a = _actual[_i] if _i < len(_actual) else "<ausente>"
                _e = _expected[_i] if _i < len(_expected) else "<ausente>"
                _mark = "  " if _a == _e else "⚠️"
                print(f"       col {_i+1:02d}  {_mark}  planilha={_a!r}  esperado={_e!r}")
    except Exception as _e:
        _all_headers_ok = False
        print(f"  ❌ {_cfg['worksheet_name']} — ERRO: {_e}")

if _all_headers_ok:
    print("\n✅ Todos os headers batem — escrita segura.")
else:
    print("\n⚠️  Divergências encontradas. Corrija EXPECTED_HEADERS ou a planilha antes de escrever.")
print()

# ── Questão 2: Grid do Resumo Executivo para calibrar cell_map ────────────────
print("=" * 64)
print("QUESTÃO 2 — Resumo Executivo: grid A1:J10 (calibração do cell_map)")
print("=" * 64)
try:
    _ws_re = _sp.worksheet("📊 Resumo Executivo")
    _grid = _ws_re.get("A1:J10")
    _cols = list("ABCDEFGHIJ")
    # Header da tabela
    print(f"  {'Linha':<6} " + "  ".join(f"{c:<22}" for c in _cols))
    print(f"  {'-'*6} " + "  ".join("-" * 22 for _ in _cols))
    for _r_idx, _row in enumerate(_grid, start=1):
        _cells = [str(_row[_c]).strip() if _c < len(_row) else "" for _c in range(10)]
        _non_empty = [c for c in _cells if c]
        if _non_empty:  # imprime apenas linhas com algum conteúdo
            print(f"  linha {_r_idx:<2}  " + "  ".join(f"{c:<22}" for c in _cells))
    print()
    print("  Use os valores acima para confirmar/corrigir o cell_map em update_resumo_executivo.")
    print("  Exemplo: se 'Priorizar melhoria' aparece na col B linha 5,")
    print("  os campos devem estar em B6 (produtos), B7 (receita), B8 (itens).")
except Exception as _e:
    print(f"  ❌ Erro ao ler Resumo Executivo: {_e}")


Python(6366) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


✅ GSheets auth: gcloud user credentials (local fallback)
Planilha: TD_Priorizacao_Melhorias
URL     : https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E/edit
Abas    : ['📊 Resumo Executivo', '🔴 Priorizar Melhoria', '📋 Relatório PF', '📂 Benchmark Categoria', '🏷️ Top Problemas', '🗂️ Farol Completo', 'ℹ️ Legenda', 'farol_executivo', 'resumo_farol', 'benchmark_categoria', 'resumo_tags', 'priorizar_melhoria', 'tweets_analiticos', 'relatorio_pf']

QUESTÃO 3 — Headers reais vs esperados (linha 2 de cada aba)
  ❌ 🗂️ Farol Completo
       col 01      planilha='Produto'  esperado='Produto'
       col 02      planilha='Categoria'  esperado='Categoria'
       col 03      planilha='Gênero'  esperado='Gênero'
       col 04      planilha='Cluster'  esperado='Cluster'
       col 05      planilha='Sinal Farol'  esperado='Sinal Farol'
       col 06      planilha='Priority Score'  esperado='Priority Score'
       col 07      planilha='TD Score'  esperado='TD Score'
      

In [113]:
# ── QA pré-escrita ────────────────────────────────────────────────────────────
assert_output_headers(farol_completo_df, FAROL_COMPLETO_HEADERS, "farol_completo_df")
assert_output_headers(priorizar_melhoria_df, PRIORIZAR_MELHORIA_HEADERS, "priorizar_melhoria_df")
assert_output_headers(benchmark_categoria_df, BENCHMARK_CATEGORIA_HEADERS, "benchmark_categoria_df")
assert_output_headers(top_problemas_df, TOP_PROBLEMAS_HEADERS, "top_problemas_df")
if relatorio_pf_df is not None:
    assert_output_headers(relatorio_pf_df, RELATORIO_PF_HEADERS, "relatorio_pf_df")

if farol_completo_df.empty:
    raise ValueError("farol_completo_df vazio. Abortando escrita.")

if priorizar_melhoria_df.empty:
    warnings.warn("priorizar_melhoria_df vazio. Aba Priorizar Melhoria será atualizada sem linhas de dados.")

if relatorio_pf_df is None or relatorio_pf_df.empty:
    warnings.warn("relatorio_pf_df vazio/None. Aba Relatório PF não será atualizada.")

# ── Execução: atualiza Google Sheets ou reporta skip ──────────────────────────
if UPDATE_GOOGLE_SHEETS:
    gc = get_gspread_client()
    spreadsheet = open_target_spreadsheet(gc, SPREADSHEET_ID)
    print(f"Planilha aberta: {spreadsheet.title}")
    print(f"URL: https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")

    update_all_tabs(
        spreadsheet=spreadsheet,
        farol_df=farol_completo_df,
        priorizar_df=priorizar_melhoria_df,
        relatorio_pf_df=relatorio_pf_df,
        benchmark_df=benchmark_categoria_df,
        top_problemas_df=top_problemas_df,
        resumo_payload=resumo_payload,
    )

    print("\n✅ Google Sheets atualizado com sucesso, sem recriar planilha ou limpar layout")
    print(f"   https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
else:
    print("⏭️  UPDATE_GOOGLE_SHEETS=False — planilha não atualizada.")

# ── Export debug Excel (opcional, não é output produtivo) ─────────────────────
if EXPORT_DEBUG_FILES:
    debug_path = OUTPUT_DIR / f"td_priorizacao_debug_{DATE_TAG}.xlsx"
    export_priority_workbook(
        executive_df,
        str(debug_path),
        analytical_df=analytical_df,
        scorecard_view_df=scorecard_view_df,
        tweets_df=tweets_df,
    )
    print(f"\n✅ Excel de debug gerado: {debug_path}")
else:
    print("⏭️  EXPORT_DEBUG_FILES=False — arquivo Excel não gerado.")


Python(6748) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


✅ GSheets auth: gcloud user credentials (local fallback)
Planilha aberta: TD_Priorizacao_Melhorias
URL: https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E/edit
  ✓ 🔴 Priorizar Melhoria: 14 linhas × 17 colunas
  ✓ 📋 Relatório PF: 50 linhas × 18 colunas
  ✓ 📂 Benchmark Categoria: 25 linhas × 10 colunas
  ✓ 🏷️ Top Problemas: 22 linhas × 6 colunas
  ✓ 🗂️ Farol Completo: 117 linhas × 18 colunas
  ✓ 📊 Resumo Executivo atualizado

✅ Google Sheets atualizado com sucesso, sem recriar planilha ou limpar layout
   https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E/edit
⏭️  EXPORT_DEBUG_FILES=False — arquivo Excel não gerado.


In [112]:
# ── Corrigir headers desatualizados no GSheets antes da escrita ──────────────
def _col_idx_to_letter(n: int) -> str:
    """Converte índice 1-based para letra de coluna Excel (A, B, ..., Z, AA, ...)."""
    result = ""
    while n > 0:
        n, r = divmod(n - 1, 26)
        result = chr(65 + r) + result
    return result

_gc_fix = get_gspread_client()
_sp_fix = open_target_spreadsheet(_gc_fix, SPREADSHEET_ID)

_fix_count = 0
for tab_key, config in TAB_CONFIG.items():
    if "expected_headers" not in config or "header_range" not in config:
        continue
    try:
        ws = _sp_fix.worksheet(config["worksheet_name"])
        values = ws.get(config["header_range"])
        if not values:
            continue
        actual  = [str(h).strip() for h in values[0]]
        expected = [str(h).strip() for h in config["expected_headers"]]
        if actual == expected:
            continue
        # Parse starting column from header_range (e.g. "A1:Q1" → col A, row 1)
        start_cell = config["header_range"].split(":")[0]          # "A1"
        start_col_letter = ''.join(filter(str.isalpha, start_cell))
        header_row = int(''.join(filter(str.isdigit, start_cell)))
        # Convert start column letter to 1-based index
        start_col_idx = sum(
            (ord(c) - 64) * (26 ** i)
            for i, c in enumerate(reversed(start_col_letter.upper()))
        )
        updates = []
        for i, (act, exp) in enumerate(zip(actual, expected)):
            if act != exp:
                col_letter = _col_idx_to_letter(start_col_idx + i)
                cell_addr  = f"{col_letter}{header_row}"
                updates.append({"range": cell_addr, "values": [[exp]]})
                print(f"  [{config['worksheet_name']}] {cell_addr}: '{act}' → '{exp}'")
        if updates:
            ws.batch_update(updates)
            _fix_count += len(updates)
    except Exception as e:
        print(f"  ⚠️  {tab_key}: {e}")

if _fix_count:
    print(f"\n✅ {_fix_count} header(s) corrigido(s) no GSheets.")
else:
    print("✅ Nenhum header desatualizado encontrado.")


Python(6634) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


✅ GSheets auth: gcloud user credentials (local fallback)
  [🗂️ Farol Completo] H2: 'Comm. Score' → 'Comm. Score v2'
  [🔴 Priorizar Melhoria] H2: 'Comm. Score' → 'Comm. Score v2'
  [📋 Relatório PF] M2: 'Tendência T&D (3m)' → 'Tendência'

✅ 3 header(s) corrigido(s) no GSheets.


## 12. Log final de execução

Resumo de todas as métricas do pipeline após execução completa.

**Entrada:** todos os DataFrames produzidos
**Saída:** log no stdout
**Quando mexer:** ao adicionar novo DataFrame ao pipeline.


In [114]:
print("=" * 70)
print("✅ PIPELINE TD — PRIORIZAÇÃO DE MELHORIAS PRODUTO FÍSICO")
print("=" * 70)
print(f"  Run date               : {RUN_DATE}")
print(f"  Date tag               : {DATE_TAG}")
print()
print(f"  executive_df           : {executive_df.shape[0]:,} produtos")
print(f"  executive_prepared_df  : {executive_prepared_df.shape[0]:,} produtos")
print(f"  farol_completo_df      : {farol_completo_df.shape}")
print(f"  priorizar_melhoria_df  : {priorizar_melhoria_df.shape}")
print(f"  relatorio_pf_df        : {relatorio_pf_df.shape if relatorio_pf_df is not None else 'None'}")
print(f"  benchmark_categoria_df : {benchmark_categoria_df.shape}")
print(f"  top_problemas_df       : {top_problemas_df.shape}")
print()
print(f"  LOAD_ANALITICA         : {LOAD_ANALITICA}")
print(f"  EXPORT_DEBUG_FILES     : {EXPORT_DEBUG_FILES}")
print(f"  UPDATE_GOOGLE_SHEETS   : {UPDATE_GOOGLE_SHEETS}")
if UPDATE_GOOGLE_SHEETS:
    print(f"  Planilha               : https://docs.google.com/spreadsheets/d/{SPREADSHEET_ID}/edit")
print("=" * 70)


✅ PIPELINE TD — PRIORIZAÇÃO DE MELHORIAS PRODUTO FÍSICO
  Run date               : 2026-06-23 18:07:42.380520-03:00
  Date tag               : 20260623

  executive_df           : 117 produtos
  executive_prepared_df  : 117 produtos
  farol_completo_df      : (117, 18)
  priorizar_melhoria_df  : (14, 17)
  relatorio_pf_df        : (50, 18)
  benchmark_categoria_df : (25, 10)
  top_problemas_df       : (22, 6)

  LOAD_ANALITICA         : True
  EXPORT_DEBUG_FILES     : False
  UPDATE_GOOGLE_SHEETS   : True
  Planilha               : https://docs.google.com/spreadsheets/d/1ikFhRdMUe1uK4bVn8lh8_7ZwESf9dmSJ5c3RV-16J_E/edit


## 13. Apêndice de debug

Seção opcional para análises exploratórias, checks pontuais e inspeção de dados.

**Controlado por `RUN_DEBUG_APPENDIX = True` no Bloco 2.**
Não faz parte do fluxo produtivo.

> Inclui as células de debug do notebook original:
> - Print loop com sinal/top-tags/tweet por produto
> - Query de tags distintas de `return_reason_tags` (sobrescreve `df_tags`)


In [34]:
if RUN_DEBUG_APPENDIX:
    # ── D1: Print loop de produtos (diagnóstico textual) ──────────────────────
    # Equivale à célula c722c1f7 do notebook original.
    print("── D1: Diagnóstico textual por produto ──────────────────────────────")
    for _, row in executive_prepared_df.iterrows():
        sinal = row.get("sinal_priorizacao", "?")
        nome = row.get("product_name", "?")
        tags = " | ".join(filter(None, [
            str(row.get("top_1_problema") or ""),
            str(row.get("top_2_problema") or ""),
            str(row.get("top_3_problema") or ""),
        ]))
        tweet_txt = build_tweet_heuristic(row.to_dict())
        print(f"\n[{sinal}] {nome}")
        print(f"  Tags: {tags}")
        print(f"  Tweet: {tweet_txt}")
    print("\n── Fim do print loop ──────────────────────────────────────────────")


In [35]:
if RUN_DEBUG_APPENDIX:
    # ── D2: Todas as tags distintas no sistema ────────────────────────────────
    # Equivale à célula f8f5db02 do notebook original.
    # Nota: esta query sobrescreve df_tags com dados de debug — use com cuidado.
    print("── D2: Tags distintas em return_reason_tags ─────────────────────────")
    _debug_tags_query = '''
        SELECT DISTINCT tag
        FROM `insider-data-lake.sop_silver.return_reason_tags`,
        UNNEST(tags) AS tag
        WHERE tag IS NOT NULL
        ORDER BY tag
    '''
    df_tags_debug = client.query(_debug_tags_query).to_dataframe()
    print(f"  {len(df_tags_debug)} tags distintas encontradas")
    display(df_tags_debug)
